# Output-wise exact-combo selection notebook — FIXED v2

This version uses the final 5th INPUT/OUTPUT layout while still evaluating the 1st/2nd/3rd/4th method families independently.

Key changes:
- Uses `RANGE_INPUT = I:FU`, `RANGE_OUTPUT = FV:HL`, and the selected output columns requested by the user.
- Defines the newly added feature block as `I:Y`.
- Defines the legacy-compatible old feature block as `Z:FU` because it has the same column count as the old `K:FF` block.
- Compares feature scopes independently: `legacy_core_only`, `added_only`, and `all_features`.
- Selects the best exact candidate for each Output directly: stage × feature_scope × Y-strategy × method × model × transform × top-k.
- Keeps final selection Output-wise, so added features are automatically excluded for Outputs where they reduce CV performance.


# Feature-block logic in this version

The current input range is `I:FU`. The added variables are `I:Y`. The legacy variables corresponding to the old `K:FF` input block are therefore treated as `Z:FU`, which gives the same 152-column width as `K:FF`.

The notebook now evaluates these scopes separately:

1. `legacy_core_only`: only the old-compatible variables (`Z:FU`). This preserves the behavior of the 1st/2nd/3rd method files as closely as possible while using the 5th file's outputs.
2. `added_only`: only the newly added variables (`I:Y`). This is diagnostic and can win only if the new variables alone generalize best.
3. `all_features`: all current variables (`I:FU`).

Because selection is performed per Output, the final model can use `legacy_core_only` for one Output and `all_features` for another Output. This directly addresses the case where added variables reduce explanatory power for specific Outputs.


In [1]:
# ============================================================
# Cell A1. Imports + Global Config
# ============================================================

# ----------------------------
# Hardware-aware execution profile
# ----------------------------
# Target system: AMD Ryzen 9 7900X = 12 physical cores / 24 logical threads,
# AMD Radeon iGPU, NVIDIA RTX4070 dGPU.
# Most sklearn models in this notebook are CPU-bound. The AMD iGPU is intentionally
# ignored for training. RTX4070 is used only if optional GPU libraries are available.

import os

HARDWARE_PROFILE = "AMD_RYZEN_9_7900X_PLUS_RTX4070"
CPU_PHYSICAL_CORES = 12
CPU_LOGICAL_THREADS = 24
NVIDIA_GPU_NAME = "RTX4070"
AMD_IGPU_NAME = "AMD Radeon (TM) Graphics"

# Options: "balanced_fast" or "full_quality".
# - balanced_fast: faster default for iterative research.
# - full_quality : heavier CV budget, closer to v3 but still with caching/CPU controls.
TRAINING_SPEED_PROFILE = "balanced_fast"

# Avoid nested over-threading: many sklearn estimators call BLAS/OpenMP internally.
# For small/medium tabular data, one BLAS thread per Python worker is usually faster
# and more stable on Windows/Jupyter.
LIMIT_INTERNAL_NUM_THREADS = True
INTERNAL_NUM_THREADS = 1
if LIMIT_INTERNAL_NUM_THREADS:
    for _var in [
        "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS"
    ]:
        os.environ[_var] = str(INTERNAL_NUM_THREADS)

# Controlled CPU workers for estimators that support n_jobs.
# Keep a few logical threads free for OS/Jupyter and Excel export.
CPU_TREE_N_JOBS = max(1, min(18, CPU_LOGICAL_THREADS - 4))
CPU_SEARCH_N_JOBS = 1       # keep search serial to avoid nested parallel explosions
CPU_OUTPUT_N_JOBS = 1       # notebook-safe; increase only when memory is abundant
CPU_METHOD_N_JOBS = 1       # method-level parallelization disabled by default

try:
    from threadpoolctl import threadpool_limits, threadpool_info
    THREADPOOLCTL_AVAILABLE = True
except Exception:
    THREADPOOLCTL_AVAILABLE = False
    threadpool_limits = None
    threadpool_info = lambda: []

# Optional GPU availability probe. This notebook does not force GPU use because
# the main sklearn pipeline is CPU-bound. GPU candidates can be added later through
# XGBoost/CuML if installed.
try:
    import cupy as cp
    try:
        _gpu_count = cp.cuda.runtime.getDeviceCount()
        CUDA_GPU_AVAILABLE = _gpu_count > 0
    except Exception:
        CUDA_GPU_AVAILABLE = False
except Exception:
    CUDA_GPU_AVAILABLE = False

import os
import re
import json
import math
import warnings
from copy import deepcopy
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import joblib
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.utils import column_index_from_string, get_column_letter

from scipy.stats import spearmanr, loguniform, uniform, randint

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression

from sklearn.model_selection import GroupKFold, GroupShuffleSplit, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, ElasticNetCV, BayesianRidge, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

# ----------------------------
# Raw import config
# ----------------------------
IMPORT_PATH = r"C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx"

RUN_TAG = datetime.now().strftime("%y%m%d_%H%M%S")
EXPORT_ROOT_NAME = f"Result_sequential_stage_outputwise_{RUN_TAG}"
EXPORT_ROOT_DIR = os.path.join(os.path.dirname(IMPORT_PATH), EXPORT_ROOT_NAME)

RAW_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "01_RawData")
MODEL_EXPORT_DIR = os.path.join(EXPORT_ROOT_DIR, "02_ModelComparison")
FINAL_BACKUP_DIR = os.path.join(EXPORT_ROOT_DIR, "03_FinalModelBackup")
FINAL_DATA_DIR = os.path.join(EXPORT_ROOT_DIR, "04_FinalSelectedDatasets")
PER_OUTPUT_DIR = os.path.join(MODEL_EXPORT_DIR, "per_output")

HEADER_MAIN_ROW = 2
HEADER_SUB_ROWS = [3, 4, 5]

DATA_ROW_START = 7
DATA_ROW_END = 204

RANGE_INPUT  = ("I",  DATA_ROW_START, "FU", DATA_ROW_END)
RANGE_OUTPUT = ("FV", DATA_ROW_START, "HL", DATA_ROW_END)

OUTPUT_COLUMNS = ["FW", "FX", "FZ", "GA", "GC", "GG", "GJ", "GZ", "HA", "HB", "HC", "HD", "HE", "HG", "HI", "HK"]

EXPORT_FORMAT_XLSX = True
EXPORT_FORMAT_CSV  = True

# ----------------------------
# Dataset config
# ----------------------------
ROUND_X_DECIMALS = 8
MIN_GROUPS_FOR_MODEL = 12
TEST_OUTPUTS = None

# ----------------------------
# CV config
# ----------------------------
RANDOM_STATE = 42
FINAL_OUTER_SPLITS = 5
if TRAINING_SPEED_PROFILE == "full_quality":
    FINAL_OUTER_SPLITS = 5
    OUTER_REPEATS = 12
    INNER_GSS_SPLITS = 24
    BLEND_EVAL_FOLDS = 5
    FEATUREAWARE_INNER_SPLITS = 5
else:
    # balanced_fast: roughly 35–50% faster while preserving stage-wise comparison logic.
    FINAL_OUTER_SPLITS = 5
    OUTER_REPEATS = 8
    INNER_GSS_SPLITS = 12
    BLEND_EVAL_FOLDS = 4
    FEATUREAWARE_INNER_SPLITS = 4

OUTER_TEST_SIZE = 0.22
INNER_GSS_TEST_SIZE = 0.20
SEARCH_N_JOBS = CPU_SEARCH_N_JOBS

# ----------------------------
# Feature/model config
# ----------------------------
ROUGH_PREFILTER_TOPK = 20
ROUGH_PREFILTER_TOPK_BY_FAMILY = {
    "mechanical_energy": 16,
    "thermal": 20,
    "vibrational": 22,
}

FINAL_TOPK_CANDIDATES = [2, 3, 4, 6]
TOPK_BY_FAMILY = {
    "mechanical_energy": [2, 3, 4],
    "thermal": [2, 3, 4, 6],
    "vibrational": [2, 3, 4, 6],
}

MAX_FINAL_FEATURES = 6
CORR_PRUNE_THRESHOLD = 0.85

FEATURE_SCORE_WEIGHTS = {
    "spearman": 0.30,
    "pearson": 0.15,
    "mutual_info": 0.15,
    "elastic_net": 0.40,
}

MODEL_AWARE_SCORING_MODELS = ["Bayesian_Ridge", "Ridge", "PLS_Regression"]

FINAL_MODEL_NAMES = [
    "Bayesian_Ridge",
    "Ridge",
    "ElasticNet_CV",
    "Huber_Regressor",
    "PLS_Regression",
    "PCR_Ridge",
    "Kernel_Ridge_RBF",
    "SVR_RBF",
]

MODEL_COMPLEXITY_RANK = {
    "Bayesian_Ridge": 1,
    "Ridge": 2,
    "ElasticNet_CV": 3,
    "Huber_Regressor": 4,
    "PLS_Regression": 5,
    "PCR_Ridge": 6,
    "Kernel_Ridge_RBF": 7,
    "SVR_RBF": 8,
    "WeightedBlend_2": 9,
    "WeightedBlend_3": 10,
}
MODEL_COMPLEXITY_RANK.update({
    "featureaware_HT_missing_ensemble": 11,
    "featureaware_partial_missing_ensemble": 11,
    "HT_missing_ensemble": 11,
    "PartialFeature_Missing_Ensemble": 11,
})

TARGET_TRANSFORM_CANDIDATES = ["raw", "yeo_johnson"]

INNER_SCORING_BY_FAMILY = {
    "mechanical_energy": "r2",
    "thermal": "r2",
    "vibrational": "r2",
}

SUMMARY_STD_PENALTY = 0.10
SUMMARY_SUPPORT_BONUS = 0.12

MIN_SUPPORT_SHARE_BY_FAMILY = {
    "mechanical_energy": 0.15,
    "thermal": 0.15,
    "vibrational": 0.20,
}

SELECTION_TOLERANCE_FRAC = 0.03
MIN_TRAIN_R2_GATE = 0.01
CONSENSUS_MIN_FOLD_SHARE = 0.40
ROBUST_TRIM_Z = 1.5

# ----------------------------
# Conservative feature engineering config
# ----------------------------
ENGINEERED_FEATURES_ENABLED = True

ENGINEERED_MAX_BASE_BY_FAMILY = {
    "mechanical_energy": 4,
    "thermal": 5,
    "vibrational": 5,
}

ENGINEERED_INCLUDE_SIGNED_LOG = True
ENGINEERED_INCLUDE_SQUARE = True

ENGINEERED_INCLUDE_PRODUCT_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": True,
}

ENGINEERED_INCLUDE_RATIO_BY_FAMILY = {
    "mechanical_energy": True,
    "thermal": True,
    "vibrational": False,
}

ENGINEERED_MAX_TOTAL_ADDED = 18

# ----------------------------
# Repeated-Y handling
# ----------------------------
APPLY_REPEATED_Y_ONLY_IF_NEEDED = True

# ----------------------------
# Selection objective
# ----------------------------
TOPK_PENALTY = 0.010
COMPLEXITY_PENALTY = 0.005
MIN_VALID_EVAL_SAMPLES = 3

# ----------------------------
# Legacy + user-defined partial-feature exhaustive comparison config
# ----------------------------
# Build GROUP_ID using the legacy-compatible feature block by default.
# This prevents newly added / partially missing descriptors from splitting the same
# experimental condition into artificial groups.
GROUP_ID_FEATURE_SCOPE = "legacy_core_only"

# Compare old-compatible variables, new variables alone, and all variables.
# The Output-wise exact selector will automatically exclude new variables when they reduce CV R2.
COMPARE_FEATURE_SCOPES = ["legacy_core_only", "added_only", "all_features"]

COMPARE_ALL_Y_STRATEGIES = True
Y_STRATEGY_CANDIDATES = [
    "auto",
    "singleton_raw",
    "hard_closest_oof",
    "robust_trimmed_median",
]

OUTPUTWISE_R2_WEIGHT = 1.00
OUTPUTWISE_STD_PENALTY = 0.12
OUTPUTWISE_SUPPORT_BONUS = 0.10
OUTPUTWISE_TOPK_PENALTY = 0.01
OUTPUTWISE_SCOPE_BONUS = {
    "legacy_core_only": 0.000,
    "added_only": 0.000,
    "all_features": 0.000,
    "common_only": 0.000,
}

# ----------------------------
# Robust output-wise selection config
# ----------------------------
# Selection is performed in two stages:
#   1) method-level aggregation: feature_scope × Y-strategy × method_name
#   2) representative exact combo: model_name × target_transform × topk
# This prevents strong methods from being fragmented into many one-fold exact combos.
SELECTION_AGGREGATION_MODE = "method_level_then_representative_combo"

# Enforce D1-selected exact model/transform/top-k during final refit whenever the method supports it.
EXACT_REFIT_ENFORCE_SELECTED_COMBO = True

# support_share = n_cv_runs for a method or combo / total CV runs.
# Stable methods are preferred; low-support methods are penalized and filtered if stable alternatives exist.
MIN_SELECTION_SUPPORT_SHARE = 0.30
LOW_SUPPORT_EXTRA_PENALTY = 0.25

# Exact-combo diagnostic scoring parameters.
# Method-level selector uses OUTPUTWISE_STD_PENALTY / OUTPUTWISE_SUPPORT_BONUS.
COMBO_STD_PENALTY = 0.08
COMBO_SUPPORT_BONUS = 0.06

# ----------------------------
# Blend / robust search config
# ----------------------------
BLEND_TOP_CANDIDATES = [2, 3]
BLEND_MIN_BASE_SCORE = 0.0
# BLEND_EVAL_FOLDS is set by TRAINING_SPEED_PROFILE above.
TRAIN_WINSOR_CLIP = 0.02

# ----------------------------
# User-defined partial-feature ensemble config
# ----------------------------
# Partial INPUT descriptors can be defined here by Excel column letters.
# Default: I:Y because these descriptors exist only for part of the samples.
#
# Examples:
#   PARTIAL_FEATURE_EXCEL_START = "I"; PARTIAL_FEATURE_EXCEL_END = "Y"
#   PARTIAL_FEATURE_EXCEL_START = "H"; PARTIAL_FEATURE_EXCEL_END = "T"
#   PARTIAL_FEATURE_EXCEL_START = None; PARTIAL_FEATURE_EXCEL_END = None  # disable partial branch
# Newly added variable block in the 5th file.
ADDED_FEATURE_EXCEL_START = "I"
ADDED_FEATURE_EXCEL_END   = "Y"

# Legacy-compatible feature block corresponding to the old 1st–3rd RANGE_INPUT K:FF.
# With current RANGE_INPUT I:FU and added variables I:Y, the old-compatible block is Z:FU.
# K:FF and Z:FU both contain 152 columns.
LEGACY_FEATURE_EXCEL_START = "Z"
LEGACY_FEATURE_EXCEL_END   = "FU"

# Backward-compatible names used by existing feature-aware ensemble functions.
PARTIAL_FEATURE_EXCEL_START = ADDED_FEATURE_EXCEL_START
PARTIAL_FEATURE_EXCEL_END   = ADDED_FEATURE_EXCEL_END

def _excel_col_count_or_zero(start_col, end_col):
    if start_col is None or end_col is None:
        return 0
    return max(0, column_index_from_string(str(end_col)) - column_index_from_string(str(start_col)) + 1)

PARTIAL_FEATURE_N_COLS = _excel_col_count_or_zero(PARTIAL_FEATURE_EXCEL_START, PARTIAL_FEATURE_EXCEL_END)

# Backward-compatible aliases used by downstream cells/functions.
STRUCTURAL_FACTOR_EXCEL_START = PARTIAL_FEATURE_EXCEL_START
STRUCTURAL_FACTOR_EXCEL_END   = PARTIAL_FEATURE_EXCEL_END
STRUCTURAL_FACTOR_N_COLS      = PARTIAL_FEATURE_N_COLS

STRUCTURAL_MIN_ROW_COVERAGE = 0.15
STRUCTURAL_MIN_GROUPS_FOR_BRANCH = 8
STRUCTURAL_MIN_PRESENT_PER_ROW = 1

STRUCTURAL_TOPK_CANDIDATES = [1, 2, 3, 4, 6]
COMMON_TOPK_CANDIDATES_FOR_ENSEMBLE = [2, 3, 4, 6]

FEATUREAWARE_BASE_MODELS = [
    "Bayesian_Ridge",
    "Ridge",
    "Huber_Regressor",
]

FEATUREAWARE_TARGET_TRANSFORMS = [
    "raw",
    "yeo_johnson",
]

FEATUREAWARE_USE_MISSING_INDICATOR = True
FEATUREAWARE_INNER_REPEATS = 1
# FEATUREAWARE_INNER_SPLITS is set by TRAINING_SPEED_PROFILE above.
FEATUREAWARE_WEIGHT_FLOOR = 1e-4

# ----------------------------
# Create export folders
# ----------------------------
os.makedirs(RAW_EXPORT_DIR, exist_ok=True)
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)
os.makedirs(FINAL_BACKUP_DIR, exist_ok=True)
os.makedirs(FINAL_DATA_DIR, exist_ok=True)
os.makedirs(PER_OUTPUT_DIR, exist_ok=True)


HARDWARE_CONFIG = {
    "hardware_profile": HARDWARE_PROFILE,
    "cpu_physical_cores": CPU_PHYSICAL_CORES,
    "cpu_logical_threads": CPU_LOGICAL_THREADS,
    "cpu_tree_n_jobs": CPU_TREE_N_JOBS,
    "cpu_search_n_jobs": CPU_SEARCH_N_JOBS,
    "cpu_output_n_jobs": CPU_OUTPUT_N_JOBS,
    "cpu_method_n_jobs": CPU_METHOD_N_JOBS,
    "limit_internal_num_threads": LIMIT_INTERNAL_NUM_THREADS,
    "internal_num_threads": INTERNAL_NUM_THREADS,
    "training_speed_profile": TRAINING_SPEED_PROFILE,
    "cuda_gpu_available": CUDA_GPU_AVAILABLE,
    "nvidia_gpu_name": NVIDIA_GPU_NAME,
    "amd_igpu_name": AMD_IGPU_NAME,
    "outer_repeats": OUTER_REPEATS,
    "inner_gss_splits": INNER_GSS_SPLITS,
    "blend_eval_folds": BLEND_EVAL_FOLDS,
    "featureaware_inner_splits": FEATUREAWARE_INNER_SPLITS,
}
with open(os.path.join(MODEL_EXPORT_DIR, "hardware_runtime_config.json"), "w", encoding="utf-8") as f:
    json.dump(HARDWARE_CONFIG, f, ensure_ascii=False, indent=2)

print("HARDWARE_PROFILE:", HARDWARE_PROFILE)
print("TRAINING_SPEED_PROFILE:", TRAINING_SPEED_PROFILE)
print("CPU_PHYSICAL/LOGICAL:", CPU_PHYSICAL_CORES, "/", CPU_LOGICAL_THREADS)
print("CPU_TREE_N_JOBS:", CPU_TREE_N_JOBS)
print("CPU_SEARCH_N_JOBS:", CPU_SEARCH_N_JOBS)
print("CUDA_GPU_AVAILABLE:", CUDA_GPU_AVAILABLE)
print("IMPORT_PATH      :", IMPORT_PATH)
print("EXPORT_ROOT_DIR  :", EXPORT_ROOT_DIR)
print("FINAL_BACKUP_DIR :", FINAL_BACKUP_DIR)
print("FINAL_DATA_DIR   :", FINAL_DATA_DIR)
print("FINAL_MODEL_NAMES:", FINAL_MODEL_NAMES)
print("OUTER_REPEATS    :", OUTER_REPEATS)
print("TOPK_BY_FAMILY   :", TOPK_BY_FAMILY)
print("GROUP_ID_FEATURE_SCOPE:", GROUP_ID_FEATURE_SCOPE)
SEQUENTIAL_STAGE_NOTE = "Stage A/B/C/D are evaluated independently before final Output-wise selection."
print("SEQUENTIAL_STAGE_NOTE:", SEQUENTIAL_STAGE_NOTE)
print("COMPARE_FEATURE_SCOPES:", COMPARE_FEATURE_SCOPES)
print("COMPARE_ALL_Y_STRATEGIES:", COMPARE_ALL_Y_STRATEGIES)
print("SELECTION_AGGREGATION_MODE:", SELECTION_AGGREGATION_MODE)
print("MIN_SELECTION_SUPPORT_SHARE:", MIN_SELECTION_SUPPORT_SHARE)
print("PARTIAL_FEATURE_EXCEL_RANGE:", f"{PARTIAL_FEATURE_EXCEL_START}:{PARTIAL_FEATURE_EXCEL_END}" if PARTIAL_FEATURE_N_COLS else "DISABLED")
print("PARTIAL_FEATURE_N_COLS:", PARTIAL_FEATURE_N_COLS)
print("STRUCTURAL_FACTOR_N_COLS:", STRUCTURAL_FACTOR_N_COLS)

HARDWARE_PROFILE: AMD_RYZEN_9_7900X_PLUS_RTX4070
TRAINING_SPEED_PROFILE: balanced_fast
CPU_PHYSICAL/LOGICAL: 12 / 24
CPU_TREE_N_JOBS: 18
CPU_SEARCH_N_JOBS: 1
CUDA_GPU_AVAILABLE: False
IMPORT_PATH      : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Total data_260503.xlsx
EXPORT_ROOT_DIR  : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_sequential_stage_outputwise_260508_104115
FINAL_BACKUP_DIR : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_sequential_stage_outputwise_260508_104115\03_FinalModelBackup
FINAL_DATA_DIR   : C:\Users\김민겸\Desktop\Metal metamaterial\Training\Result_sequential_stage_outputwise_260508_104115\04_FinalSelectedDatasets
FINAL_MODEL_NAMES: ['Bayesian_Ridge', 'Ridge', 'ElasticNet_CV', 'Huber_Regressor', 'PLS_Regression', 'PCR_Ridge', 'Kernel_Ridge_RBF', 'SVR_RBF']
OUTER_REPEATS    : 8
TOPK_BY_FAMILY   : {'mechanical_energy': [2, 3, 4], 'thermal': [2, 3, 4, 6], 'vibrational': [2, 3, 4, 6]}
GROUP_ID_FEATURE_SCOPE: legacy_core_only
SEQUENTIAL_STAGE

In [2]:
# ============================================================
# Cell A2. Helper Functions - Excel / Header / Export
# ============================================================

def _ffill_horiz(values):
    out = []
    last = None
    for v in values:
        if v is None or str(v).strip() == "":
            out.append(last)
        else:
            last = str(v).strip()
            out.append(last)
    return out

def _sanitize_header(text):
    text = "" if text is None else str(text)
    text = text.replace("\n", " ").replace("\r", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def _make_unique(names):
    seen = Counter()
    out = []
    for n in names:
        base = n if n else "Unnamed"
        seen[base] += 1
        out.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return out

def build_headers_from_rows(ws, start_col_letter, end_col_letter, main_row, sub_rows):
    start_col = column_index_from_string(start_col_letter)
    end_col = column_index_from_string(end_col_letter)
    col_indices = list(range(start_col, end_col + 1))

    all_rows = [main_row] + list(sub_rows)
    row_values = []
    for r in all_rows:
        vals = [ws.cell(row=r, column=c).value for c in col_indices]
        row_values.append(_ffill_horiz(vals))

    headers = []
    for i, col_idx in enumerate(col_indices):
        parts = []
        for row_vals in row_values:
            v = row_vals[i]
            if v is not None and str(v).strip() != "":
                v = _sanitize_header(v)
                if len(parts) == 0 or parts[-1] != v:
                    parts.append(v)
        if len(parts) == 0:
            parts = [f"COL_{get_column_letter(col_idx)}"]
        headers.append(" | ".join(parts))

    return _make_unique(headers), [get_column_letter(c) for c in col_indices]

def extract_range_df(ws, range_spec, header_main_row, header_sub_rows):
    start_col, start_row, end_col, end_row = range_spec
    headers, letters = build_headers_from_rows(
        ws=ws,
        start_col_letter=start_col,
        end_col_letter=end_col,
        main_row=header_main_row,
        sub_rows=header_sub_rows
    )
    start_idx = column_index_from_string(start_col)
    end_idx = column_index_from_string(end_col)

    data = []
    for r in range(start_row, end_row + 1):
        row_vals = [ws.cell(row=r, column=c).value for c in range(start_idx, end_idx + 1)]
        data.append(row_vals)

    df = pd.DataFrame(data, columns=headers)
    df.attrs["excel_letters"] = letters
    return df

def export_df(df, path_no_ext, index=False):
    if EXPORT_FORMAT_CSV:
        df.to_csv(path_no_ext + ".csv", index=index, encoding="utf-8-sig")
    if EXPORT_FORMAT_XLSX:
        df.to_excel(path_no_ext + ".xlsx", index=index)

def _to_jsonable(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, np.ndarray):
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.Series):
        if obj.index.is_unique:
            return {str(k): _to_jsonable(v) for k, v in obj.to_dict().items()}
        return [_to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, pd.DataFrame):
        return [{str(k): _to_jsonable(v) for k, v in row.items()} for row in obj.to_dict(orient="records")]
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_jsonable(x) for x in obj]
    if hasattr(obj, "item"):
        try:
            return _to_jsonable(obj.item())
        except Exception:
            pass
    return str(obj)

def export_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(_to_jsonable(obj), f, ensure_ascii=False, indent=2)

def safe_numeric_df(df):
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def sanitize_filename(text, max_len=120):
    text = re.sub(r'[\\/:*?"<>|]+', "_", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_len]

In [3]:
# ============================================================
# Cell B1. Workbook Load + Raw Range Extraction
# ============================================================

wb = load_workbook(IMPORT_PATH, data_only=True)
sheet_name = wb.sheetnames[0]
ws = wb[sheet_name]

input_raw_df = extract_range_df(ws, RANGE_INPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)
output_raw_all_df = extract_range_df(ws, RANGE_OUTPUT, HEADER_MAIN_ROW, HEADER_SUB_ROWS)

output_letter_to_name = dict(zip(output_raw_all_df.attrs["excel_letters"], output_raw_all_df.columns))
selected_output_names = [output_letter_to_name[c] for c in OUTPUT_COLUMNS if c in output_letter_to_name]
output_raw_df = output_raw_all_df[selected_output_names].copy()

export_df(input_raw_df, os.path.join(RAW_EXPORT_DIR, "input_raw"))
export_df(output_raw_df, os.path.join(RAW_EXPORT_DIR, "output_raw_selected"))

print("Loaded workbook :", sheet_name)
print("Input shape     :", input_raw_df.shape)
print("Output shape    :", output_raw_df.shape)
print("Selected outputs:", len(selected_output_names))
display(pd.DataFrame({"output_excel_col": OUTPUT_COLUMNS, "output_name": selected_output_names}))

Loaded workbook : 총정리
Input shape     : (198, 169)
Output shape    : (198, 16)
Selected outputs: 16


,output_excel_col,output_name
0,FW,Modulus
1,FX,Com. Strength
2,FZ,APS
3,GA,AS
4,GC,Yield strength
5,GG,Densif. strength
6,GJ,Total energy
7,GZ,Thermal characteristics | Thermal conductivity...
8,HA,Thermal characteristics | h | W/m.K
9,HB,Thermal characteristics | Heating rate | °C/s


In [4]:
# ============================================================
# Cell B2. Numeric conversion + Same-X Group ID + DATA_BY_OUTPUT
# ============================================================

X_all_raw_numeric = safe_numeric_df(input_raw_df)
Y_all = safe_numeric_df(output_raw_df)

# Keep the original Excel-letter-to-feature mapping before dropping all-empty columns.
INPUT_FEATURE_EXCEL_MAP_DF = pd.DataFrame({
    "input_position_1based": np.arange(1, len(input_raw_df.columns) + 1),
    "excel_column": input_raw_df.attrs.get("excel_letters", [None] * len(input_raw_df.columns)),
    "feature_name": list(input_raw_df.columns),
})

X_all = X_all_raw_numeric.loc[:, X_all_raw_numeric.notna().any(axis=0)].copy()
Y_all = Y_all.loc[:, Y_all.notna().any(axis=0)].copy()


def get_feature_cols_from_excel_range(input_df, input_range, start_col, end_col, existing_cols=None):
    """Return feature names corresponding to an Excel column range within RANGE_INPUT.

    This is position-based and robust to custom/merged headers. It also works when
    the partial range does not start at the first input column.
    """
    if start_col is None or end_col is None:
        return []

    input_start_idx = column_index_from_string(str(input_range[0]))
    input_end_idx = column_index_from_string(str(input_range[2]))
    partial_start_idx = column_index_from_string(str(start_col))
    partial_end_idx = column_index_from_string(str(end_col))

    lo = max(input_start_idx, min(partial_start_idx, partial_end_idx))
    hi = min(input_end_idx, max(partial_start_idx, partial_end_idx))
    if hi < lo:
        return []

    existing = set(existing_cols) if existing_cols is not None else set(input_df.columns)
    selected = []
    for excel_idx in range(lo, hi + 1):
        pos0 = excel_idx - input_start_idx
        if 0 <= pos0 < len(input_df.columns):
            feature_name = input_df.columns[pos0]
            if feature_name in existing:
                selected.append(feature_name)
    return selected


def get_feature_excel_letters_from_range(input_df, input_range, start_col, end_col, existing_cols=None):
    if start_col is None or end_col is None:
        return []

    input_start_idx = column_index_from_string(str(input_range[0]))
    input_end_idx = column_index_from_string(str(input_range[2]))
    partial_start_idx = column_index_from_string(str(start_col))
    partial_end_idx = column_index_from_string(str(end_col))

    lo = max(input_start_idx, min(partial_start_idx, partial_end_idx))
    hi = min(input_end_idx, max(partial_start_idx, partial_end_idx))
    if hi < lo:
        return []

    existing = set(existing_cols) if existing_cols is not None else set(input_df.columns)
    letters = []
    for excel_idx in range(lo, hi + 1):
        pos0 = excel_idx - input_start_idx
        if 0 <= pos0 < len(input_df.columns):
            feature_name = input_df.columns[pos0]
            if feature_name in existing:
                letters.append(get_column_letter(excel_idx))
    return letters


# User-defined added descriptors and legacy-compatible descriptors are selected by Excel-letter ranges in Cell A1.
# Added block: I:Y. Legacy-compatible block: Z:FU.
ADDED_FEATURE_COLS = get_feature_cols_from_excel_range(
    input_raw_df, RANGE_INPUT, ADDED_FEATURE_EXCEL_START, ADDED_FEATURE_EXCEL_END, existing_cols=X_all.columns
)
ADDED_FEATURE_EXCEL_LETTERS = get_feature_excel_letters_from_range(
    input_raw_df, RANGE_INPUT, ADDED_FEATURE_EXCEL_START, ADDED_FEATURE_EXCEL_END, existing_cols=X_all.columns
)
LEGACY_FEATURE_COLS = get_feature_cols_from_excel_range(
    input_raw_df, RANGE_INPUT, LEGACY_FEATURE_EXCEL_START, LEGACY_FEATURE_EXCEL_END, existing_cols=X_all.columns
)
LEGACY_FEATURE_EXCEL_LETTERS = get_feature_excel_letters_from_range(
    input_raw_df, RANGE_INPUT, LEGACY_FEATURE_EXCEL_START, LEGACY_FEATURE_EXCEL_END, existing_cols=X_all.columns
)

PARTIAL_FEATURE_COLS = ADDED_FEATURE_COLS
PARTIAL_FEATURE_EXCEL_LETTERS = ADDED_FEATURE_EXCEL_LETTERS
STRUCTURAL_FACTOR_COLS = ADDED_FEATURE_COLS
STRUCTURAL_FACTOR_EXCEL_LETTERS = ADDED_FEATURE_EXCEL_LETTERS
COMMON_FEATURE_COLS = LEGACY_FEATURE_COLS

FEATURE_SCOPE_COLS = {
    "legacy_core_only": list(LEGACY_FEATURE_COLS),
    "added_only": list(ADDED_FEATURE_COLS),
    "all_features": list(X_all.columns),
    "common_only": list(LEGACY_FEATURE_COLS),
}
FEATURE_SCOPE_LETTERS = {
    "legacy_core_only": list(LEGACY_FEATURE_EXCEL_LETTERS),
    "added_only": list(ADDED_FEATURE_EXCEL_LETTERS),
    "all_features": list(input_raw_df.attrs.get("excel_letters", [])),
    "common_only": list(LEGACY_FEATURE_EXCEL_LETTERS),
}

FEATURE_SCOPE_SUMMARY_DF = pd.DataFrame([
    {
        "feature_scope": scope,
        "n_features": len(cols),
        "excel_start": FEATURE_SCOPE_LETTERS.get(scope, [""])[0] if FEATURE_SCOPE_LETTERS.get(scope) else "",
        "excel_end": FEATURE_SCOPE_LETTERS.get(scope, [""])[-1] if FEATURE_SCOPE_LETTERS.get(scope) else "",
        "excel_columns": ", ".join(FEATURE_SCOPE_LETTERS.get(scope, [])),
    }
    for scope, cols in FEATURE_SCOPE_COLS.items()
])
export_df(FEATURE_SCOPE_SUMMARY_DF, os.path.join(RAW_EXPORT_DIR, "feature_scope_summary"))

PARTIAL_COVERAGE_DF = pd.DataFrame({
    "excel_column": PARTIAL_FEATURE_EXCEL_LETTERS,
    "feature_name": PARTIAL_FEATURE_COLS,
    "partial_feature_range_setting": [f"{PARTIAL_FEATURE_EXCEL_START}:{PARTIAL_FEATURE_EXCEL_END}" for _ in PARTIAL_FEATURE_COLS],
    "non_missing_rows": [int(X_all[c].notna().sum()) for c in PARTIAL_FEATURE_COLS],
    "non_missing_ratio": [float(X_all[c].notna().mean()) for c in PARTIAL_FEATURE_COLS],
})
export_df(PARTIAL_COVERAGE_DF, os.path.join(MODEL_EXPORT_DIR, "partial_feature_coverage"))
# Backward-compatible export name from the previous H:T version.
export_df(PARTIAL_COVERAGE_DF, os.path.join(MODEL_EXPORT_DIR, "structural_factor_partial_coverage"))
export_df(INPUT_FEATURE_EXCEL_MAP_DF, os.path.join(RAW_EXPORT_DIR, "input_feature_excel_column_map"))

print("Input Excel range:", f"{RANGE_INPUT[0]}:{RANGE_INPUT[2]}")
print("Added feature range setting:", f"{ADDED_FEATURE_EXCEL_START}:{ADDED_FEATURE_EXCEL_END}")
print("Legacy-compatible feature range setting:", f"{LEGACY_FEATURE_EXCEL_START}:{LEGACY_FEATURE_EXCEL_END}")
print("Detected added feature columns:", len(ADDED_FEATURE_COLS), ADDED_FEATURE_COLS)
print("Detected added Excel columns:", ADDED_FEATURE_EXCEL_LETTERS)
print("Detected legacy feature columns:", len(LEGACY_FEATURE_COLS))
display(FEATURE_SCOPE_SUMMARY_DF[["feature_scope", "n_features", "excel_start", "excel_end"]])


def build_group_ids_from_x(X_df, decimals=8):
    X_round = X_df.round(decimals)
    key_series = X_round.astype(str).agg("||".join, axis=1)
    codes, _ = pd.factorize(key_series, sort=False)
    return pd.Series(codes, name="GROUP_ID"), key_series.rename("X_KEY")

# Build GROUP_ID from the configured feature scope.
# Default is legacy_core_only so that newly added descriptors do not split identical old experimental conditions.
GROUP_ID_SOURCE_FEATURE_COLS = FEATURE_SCOPE_COLS.get(GROUP_ID_FEATURE_SCOPE, [])
if len(GROUP_ID_SOURCE_FEATURE_COLS) < 1:
    GROUP_ID_SOURCE_FEATURE_COLS = LEGACY_FEATURE_COLS if len(LEGACY_FEATURE_COLS) >= 1 else X_all.columns.tolist()

GROUP_ID_SOURCE_DF = X_all[GROUP_ID_SOURCE_FEATURE_COLS].copy()
GROUP_ID, X_KEY = build_group_ids_from_x(GROUP_ID_SOURCE_DF, decimals=ROUND_X_DECIMALS)

GROUP_ID_SCOPE_DF = pd.DataFrame({
    "group_id_feature_scope": [GROUP_ID_FEATURE_SCOPE],
    "n_group_id_source_features": [len(GROUP_ID_SOURCE_FEATURE_COLS)],
    "n_total_features": [X_all.shape[1]],
    "n_added_features": [len(ADDED_FEATURE_COLS)],
    "n_legacy_features": [len(LEGACY_FEATURE_COLS)],
    "n_added_features_excluded_from_grouping": [len([c for c in ADDED_FEATURE_COLS if c not in GROUP_ID_SOURCE_FEATURE_COLS])],
    "n_groups": [int(GROUP_ID.nunique())],
})
export_df(GROUP_ID_SCOPE_DF, os.path.join(RAW_EXPORT_DIR, "group_id_feature_scope"))
print("GROUP_ID built from scope:", GROUP_ID_FEATURE_SCOPE)
print("GROUP_ID source feature count:", len(GROUP_ID_SOURCE_FEATURE_COLS))
print("Detected groups:", int(GROUP_ID.nunique()))

MASTER_DF = pd.concat([X_all, Y_all, GROUP_ID, X_KEY], axis=1)
group_size_df = GROUP_ID.value_counts().sort_index().rename("group_size").reset_index()
group_size_df.columns = ["GROUP_ID", "group_size"]

export_df(MASTER_DF, os.path.join(RAW_EXPORT_DIR, "master_numeric_with_group"))
export_df(group_size_df, os.path.join(RAW_EXPORT_DIR, "group_size_summary"))

DATA_BY_OUTPUT = {}
target_output_names = Y_all.columns.tolist()
if TEST_OUTPUTS is not None:
    target_output_names = [c for c in target_output_names if c in TEST_OUTPUTS]

summary_rows = []
for output_name in target_output_names:
    tmp = pd.concat([X_all, Y_all[[output_name]], GROUP_ID, X_KEY], axis=1)
    tmp = tmp.dropna(subset=[output_name]).reset_index(drop=True)
    if tmp.empty:
        continue

    DATA_BY_OUTPUT[output_name] = {
        "df": tmp.copy(),
        "feature_cols": X_all.columns.tolist(),
        "target_col": output_name,
        "group_col": "GROUP_ID",
        "xkey_col": "X_KEY",
    }

    summary_rows.append({
        "output_name": output_name,
        "n_rows": int(len(tmp)),
        "n_groups": int(tmp["GROUP_ID"].nunique()),
        "n_features": int(len(X_all.columns)),
    })

DATASET_SUMMARY_DF = pd.DataFrame(summary_rows).sort_values(["n_groups", "n_rows"], ascending=[False, False])
export_df(DATASET_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "dataset_summary_by_output"))

display(DATASET_SUMMARY_DF)
print("Prepared outputs:", len(DATA_BY_OUTPUT))

Input Excel range: I:FU
Added feature range setting: I:Y
Legacy-compatible feature range setting: Z:FU
Detected added feature columns: 17 ['node | 5x5x5', 'strut | 5x5x5', 'Criteria | 5x5x5', 'Global: Strut No. at Nodes-AVG | Criteria | 5x5x5', 'Global: Strut No. at Nodes-STDEV | Criteria | 5x5x5', 'Global: l/d AVG | Criteria | 5x5x5', 'Global: l/d STDEV | Criteria | 5x5x5', 'Global: Length - AVG | Criteria | 5x5x5', 'Global: Length - STDEV | Criteria | 5x5x5', 'Global: Angle (weighted with length)-AVG | Criteria | 5x5x5', 'Global: Angle (weighted with length)-STDEV | Criteria | 5x5x5', 'Node: l/d AVG | Criteria | 5x5x5', 'Node: l/d- STDEV | Criteria | 5x5x5', 'Node: Length - AVG | Criteria | 5x5x5', 'Node: Length - STDEV | Criteria | 5x5x5', 'Node: Angle (weighted with length)-AVG | Criteria | 5x5x5', 'Node: Angle (weighted with length)-STDEV | Criteria | 5x5x5']
Detected added Excel columns: ['I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
Detecte

,feature_scope,n_features,excel_start,excel_end
0,legacy_core_only,152,Z,FU
1,added_only,17,I,Y
2,all_features,169,I,FU
3,common_only,152,Z,FU


GROUP_ID built from scope: legacy_core_only
GROUP_ID source feature count: 152
Detected groups: 70


,output_name,n_rows,n_groups,n_features
12,Vibrational response | FRF (g/N) | 300-8000 Hz...,193,65,169
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,193,65,169
14,Vibrational response | FRF (g/N) | 3000-6500 H...,193,65,169
15,Vibrational response | FRF (g/N) | 6500-8000 H...,193,65,169
7,Thermal characteristics | Thermal conductivity...,65,65,169
9,Thermal characteristics | Heating rate | °C/s,65,65,169
10,Thermal characteristics | Cooling rate | °C/s,65,65,169
11,Thermal characteristics | Heating Temp | °C/s,65,65,169
0,Modulus,56,56,169
1,Com. Strength,56,56,169


Prepared outputs: 16


In [5]:
# ============================================================
# Cell C1. Repeated-group target variance diagnostics
# ============================================================

diag_rows = []

for output_name, bundle in DATA_BY_OUTPUT.items():
    df = bundle["df"].copy()
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]

    # numeric safety
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col, group_col]).copy()

    if df.empty:
        continue

    grp = (
        df.groupby(group_col)[target_col]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index(drop=True)
    )

    grp["std"] = grp["std"].fillna(0.0)
    grp["range"] = (grp["max"] - grp["min"]).fillna(0.0)
    grp["cv_like"] = grp["std"] / (grp["mean"].abs() + 1e-12)
    grp["cv_like"] = grp["cv_like"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    diag_rows.append({
        "output_name": output_name,
        "n_groups": int(df[group_col].nunique()),
        "n_rows": int(len(df)),
        "median_group_size": float(df.groupby(group_col).size().median()),
        "median_group_std": float(grp["std"].median()),
        "mean_group_std": float(grp["std"].mean()),
        "median_group_range": float(grp["range"].median()),
        "mean_group_cv_like": float(grp["cv_like"].mean()),
    })

GROUP_VARIANCE_DIAG_DF = pd.DataFrame(diag_rows)

if not GROUP_VARIANCE_DIAG_DF.empty:
    GROUP_VARIANCE_DIAG_DF = GROUP_VARIANCE_DIAG_DF.sort_values(
        "mean_group_std", ascending=False
    ).reset_index(drop=True)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "repeated_group_target_variance_diag")
)

def classify_output_family(output_name):
    name = str(output_name)
    if "Thermal characteristics" in name:
        return "thermal"
    if "Vibrational response" in name:
        return "vibrational"
    return "mechanical_energy"

GROUP_VARIANCE_DIAG_DF["output_family"] = (
    GROUP_VARIANCE_DIAG_DF["output_name"].map(classify_output_family)
)

display(GROUP_VARIANCE_DIAG_DF)
export_df(
    GROUP_VARIANCE_DIAG_DF,
    os.path.join(MODEL_EXPORT_DIR, "target_variance_diagnostics")
)

,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like
0,Vibrational response | FRF (g/N) | 6500-8000 H...,65,193,3.0,0.078882,0.163502,0.147067,0.175819
1,Vibrational response | FRF (g/N) | 3000-6500 H...,65,193,3.0,0.042393,0.061436,0.080041,0.144881
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,65,193,3.0,0.036103,0.048223,0.071635,0.122471
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,65,193,3.0,0.007093,0.012829,0.013342,0.154173
4,Modulus,56,56,1.0,0.000000,0.000000,0.000000,0.000000
5,Com. Strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000
6,APS,56,56,1.0,0.000000,0.000000,0.000000,0.000000
7,AS,56,56,1.0,0.000000,0.000000,0.000000,0.000000
8,Yield strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000
9,Densif. strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000


,output_name,n_groups,n_rows,median_group_size,median_group_std,mean_group_std,median_group_range,mean_group_cv_like,output_family
0,Vibrational response | FRF (g/N) | 6500-8000 H...,65,193,3.0,0.078882,0.163502,0.147067,0.175819,vibrational
1,Vibrational response | FRF (g/N) | 3000-6500 H...,65,193,3.0,0.042393,0.061436,0.080041,0.144881,vibrational
2,Vibrational response | FRF (g/N) | 300-8000 Hz...,65,193,3.0,0.036103,0.048223,0.071635,0.122471,vibrational
3,Vibrational response | FRF (g/N) | 300-3000 Hz...,65,193,3.0,0.007093,0.012829,0.013342,0.154173,vibrational
4,Modulus,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy
5,Com. Strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy
6,APS,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy
7,AS,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy
8,Yield strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy
9,Densif. strength,56,56,1.0,0.000000,0.000000,0.000000,0.000000,mechanical_energy


In [6]:

# ============================================================
# Cell C2. Core helper functions required by method-comparison
# ============================================================

def rankdata_average_ties(x):
    order = np.argsort(x)
    ranks = np.empty(len(x), dtype=float)
    i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]:
            j += 1
        rank = 0.5 * (i + j) + 1
        ranks[order[i:j + 1]] = rank
        i = j + 1
    return ranks

def minmax_series(s):
    s = pd.Series(s).astype(float)
    if len(s) == 0 or s.nunique(dropna=False) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    mn, mx = np.nanmin(s.values), np.nanmax(s.values)
    return (s - mn) / (mx - mn + 1e-12)

def pearson_corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return 0.0
    xs, ys = x[mask], y[mask]
    if np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        return 0.0
    return float(np.corrcoef(xs, ys)[0, 1])

def spearman_corr_safe(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return 0.0
    xs, ys = x[mask], y[mask]
    if np.std(xs) < 1e-12 or np.std(ys) < 1e-12:
        return 0.0
    return pearson_corr_safe(rankdata_average_ties(xs), rankdata_average_ties(ys))

def repeated_group_splits(groups, n_splits=5, n_repeats=8, random_state=42, return_split_id=False, test_size=0.22):
    groups = np.asarray(groups)
    idx = np.arange(len(groups))
    for rep in range(n_repeats):
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state + rep)
        tr, te = next(gss.split(idx, groups=groups))
        if return_split_id:
            yield rep, rep, 0, tr, te
        else:
            yield rep, (tr, te)

def group_repeat_stats(groups):
    ser = pd.Series(groups)
    vc = ser.value_counts()
    return {
        "n_groups": int(vc.shape[0]),
        "max_group_size": int(vc.max()) if len(vc) else 0,
        "has_repeats": bool((vc > 1).any()),
        "repeat_fraction": float((vc > 1).mean()) if len(vc) else 0.0,
    }

def get_best_y_strategy_for_output(output_name, groups=None):
    fam = classify_output_family(output_name)
    if APPLY_REPEATED_Y_ONLY_IF_NEEDED and groups is not None:
        rep = group_repeat_stats(groups)
        if not rep["has_repeats"]:
            return "singleton_raw"
    if fam in ("thermal", "vibrational"):
        return "robust_trimmed_median"
    return "hard_closest_oof"

def robust_group_target(values, z=1.5):
    v = pd.Series(np.asarray(values, dtype=float)).dropna()
    if len(v) == 0:
        return np.nan
    if len(v) <= 2:
        return float(v.median())
    med = float(v.median())
    mad = float(np.median(np.abs(v - med)))
    if mad < 1e-12:
        return med
    zscore = 0.6745 * (v - med) / mad
    keep = v[np.abs(zscore) <= z]
    if len(keep) == 0:
        keep = v
    return float(keep.median())

def aggregate_by_group_firstX(X_df, y, groups, y_method="median"):
    rows, ys, gs = [], [], []
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups)
    for g in pd.unique(groups):
        idx = np.where(groups == g)[0]
        block_X = X_df.iloc[idx]
        block_y = y[idx]
        rows.append(block_X.iloc[0].copy())
        ys.append(float(np.nanmean(block_y) if y_method == "mean" else np.nanmedian(block_y)))
        gs.append(g)
    Xg = pd.DataFrame(rows).reset_index(drop=True)
    yg = pd.Series(ys, name="target").reset_index(drop=True)
    gg = pd.Series(gs, name="GROUP_ID").reset_index(drop=True)
    return Xg, yg, gg

def make_target_transformer(method):
    if method == "raw":
        return None
    if method == "yeo_johnson":
        return PowerTransformer(method="yeo-johnson", standardize=True)
    return None

def make_pipeline_and_space(model_name, n_features, random_state=42):
    max_comp = max(1, min(int(n_features), 8))
    if model_name == "Bayesian_Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", BayesianRidge())])
        return pipe, {}, "grid", 1
    if model_name == "Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", Ridge(random_state=random_state))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e3)}, "random", 20
    if model_name == "ElasticNet_CV":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNet(max_iter=30000, random_state=random_state))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e1), "model__l1_ratio": uniform(0.05, 0.90)}, "random", 24
    if model_name == "Huber_Regressor":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", HuberRegressor(max_iter=3000))])
        return pipe, {"model__alpha": loguniform(1e-6, 1e-1), "model__epsilon": uniform(1.15, 0.85)}, "random", 18
    if model_name == "PLS_Regression":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", PLSRegression())])
        return pipe, {"model__n_components": list(range(1, max_comp + 1))}, "grid", 1
    if model_name == "PCR_Ridge":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("pca", PCA()), ("model", Ridge(random_state=random_state))])
        return pipe, {"pca__n_components": list(range(1, max_comp + 1)), "model__alpha": loguniform(1e-4, 1e3)}, "random", 24
    if model_name == "Kernel_Ridge_RBF":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", KernelRidge(kernel="rbf"))])
        return pipe, {"model__alpha": loguniform(1e-4, 1e2), "model__gamma": loguniform(1e-4, 1e1)}, "random", 24
    if model_name == "SVR_RBF":
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", SVR(kernel="rbf"))])
        return pipe, {"model__C": loguniform(1e-2, 1e2), "model__gamma": loguniform(1e-4, 1e1), "model__epsilon": loguniform(1e-4, 1e0)}, "random", 28
    raise ValueError(model_name)

def fit_search_model(X_train, y_train, groups_train, model_name, target_transform="raw", scoring="r2", random_state=42):
    pipe, params, search_type, n_iter = make_pipeline_and_space(model_name, X_train.shape[1], random_state=random_state)
    transformer = make_target_transformer(target_transform)
    estimator = pipe if transformer is None else TransformedTargetRegressor(regressor=pipe, transformer=transformer, check_inverse=False)
    wrapped_params = params if transformer is None else {f"regressor__{k}": v for k, v in params.items()}
    n_unique_groups = len(pd.unique(groups_train))
    cv = GroupShuffleSplit(n_splits=INNER_GSS_SPLITS, test_size=INNER_GSS_TEST_SIZE, random_state=random_state) if n_unique_groups >= 4 else GroupKFold(n_splits=max(2, n_unique_groups))
    if search_type == "grid":
        search = GridSearchCV(estimator=estimator, param_grid=wrapped_params, scoring=scoring, cv=cv, n_jobs=SEARCH_N_JOBS, refit=True, error_score=np.nan)
    else:
        search = RandomizedSearchCV(estimator=estimator, param_distributions=wrapped_params, n_iter=n_iter, scoring=scoring, cv=cv, n_jobs=SEARCH_N_JOBS, refit=True, random_state=random_state, error_score=np.nan)
    search.fit(X_train, y_train, groups=groups_train)
    cvres = pd.DataFrame(search.cv_results_)
    score_col = "mean_test_score"
    std_col = "std_test_score"
    yhat = np.asarray(search.best_estimator_.predict(X_train)).reshape(-1)
    train_r2 = r2_score(y_train, yhat) if len(y_train) >= 2 else np.nan
    return {
        "best_estimator": search.best_estimator_,
        "best_params": search.best_params_,
        "best_score": float(cvres.loc[search.best_index_, score_col]),
        "best_score_std": float(cvres.loc[search.best_index_, std_col]) if std_col in cvres.columns else 0.0,
        "train_r2": float(train_r2),
    }

def prefilter_features_groupwise(X_df, y, groups, top_k=20, corr_threshold=0.85, random_state=42):
    X = X_df.copy()
    y = np.asarray(y, dtype=float)
    grp_df = X.copy()
    grp_df["target"] = y
    grp_df["GROUP_ID"] = groups
    grp_med = grp_df.groupby("GROUP_ID").median(numeric_only=True).reset_index(drop=True)
    y_group = grp_med.pop("target").to_numpy(dtype=float)
    X_group = grp_med.copy()
    structural_cols_local = set(globals().get("STRUCTURAL_FACTOR_COLS", []))
    keep_cols = []
    for c in X_group.columns:
        n_valid_c = int(X_group[c].notna().sum())
        if c in structural_cols_local:
            min_valid_c = max(3, int(math.ceil(STRUCTURAL_MIN_ROW_COVERAGE * len(X_group))))
        else:
            min_valid_c = max(6, int(0.6 * len(X_group)))
        if n_valid_c >= min_valid_c:
            keep_cols.append(c)
    X_group = X_group[keep_cols].copy()
    if X_group.shape[1] == 0:
        return [], pd.DataFrame(columns=["feature_name", "ensemble_score"])
    X_imp = X_group.fillna(X_group.median())
    try:
        mi = mutual_info_regression(X_imp, y_group, random_state=random_state)
    except Exception:
        mi = np.zeros(X_imp.shape[1])
    try:
        en = Pipeline([("sc", StandardScaler()), ("m", ElasticNetCV(l1_ratio=[0.2,0.5,0.8], cv=5, random_state=random_state, n_jobs=CPU_TREE_N_JOBS, max_iter=10000))])
        en.fit(X_imp, y_group)
        coefs = np.abs(en.named_steps["m"].coef_)
    except Exception:
        coefs = np.zeros(X_imp.shape[1])
    score_df = pd.DataFrame({
        "feature_name": X_imp.columns,
        "spearman": [abs(spearman_corr_safe(X_imp[c].to_numpy(dtype=float), y_group)) for c in X_imp.columns],
        "pearson": [abs(pearson_corr_safe(X_imp[c].to_numpy(dtype=float), y_group)) for c in X_imp.columns],
        "mutual_info": list(mi),
        "elastic_net": list(coefs),
    })
    score_df["ensemble_score"] = (
        0.35 * minmax_series(score_df["spearman"]) +
        0.15 * minmax_series(score_df["pearson"]) +
        0.15 * minmax_series(score_df["mutual_info"]) +
        0.35 * minmax_series(score_df["elastic_net"])
    )
    score_df = score_df.sort_values("ensemble_score", ascending=False).reset_index(drop=True)
    ranked = score_df["feature_name"].tolist()
    kept = []
    for feat in ranked:
        if len(kept) >= top_k:
            break
        ok = True
        for prev in kept:
            corr = abs(pearson_corr_safe(X_imp[feat].to_numpy(dtype=float), X_imp[prev].to_numpy(dtype=float)))
            if corr >= corr_threshold:
                ok = False
                break
        if ok:
            kept.append(feat)
    return kept, score_df

def get_topk_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    return TOPK_BY_FAMILY.get(fam, FINAL_TOPK_CANDIDATES)

def get_inner_scoring_for_output(output_name):
    fam = classify_output_family(output_name)
    return INNER_SCORING_BY_FAMILY.get(fam, "r2")

def get_candidate_models_for_output(output_name):
    fam = classify_output_family(output_name)
    if fam == "vibrational":
        return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor", "Kernel_Ridge_RBF"]
    if fam == "thermal":
        return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor", "ElasticNet_CV", "Kernel_Ridge_RBF"]
    return ["PCR_Ridge", "PLS_Regression", "Ridge", "Bayesian_Ridge", "Huber_Regressor"]

def get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols, scoring_models=None, random_state=42):
    if scoring_models is None:
        scoring_models = MODEL_AWARE_SCORING_MODELS
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw).reset_index(drop=True)
    pred_lists = defaultdict(list)
    splits = list(repeated_group_splits(groups_raw.to_numpy(), n_splits=min(5, len(pd.unique(groups_raw))), n_repeats=8, random_state=random_state))
    for rep, (tr_idx, va_idx) in splits:
        X_tr_raw = X_raw.iloc[tr_idx].reset_index(drop=True)
        y_tr_raw = y_raw.iloc[tr_idx].reset_index(drop=True)
        g_tr_raw = groups_raw.iloc[tr_idx].reset_index(drop=True)
        X_va_raw = X_raw.iloc[va_idx].reset_index(drop=True)
        X_tr_g, y_tr_g, g_tr_g = aggregate_by_group_firstX(X_tr_raw[rough_cols], y_tr_raw.to_numpy(), g_tr_raw.to_numpy(), y_method="median")
        X_va = X_va_raw[rough_cols].copy()
        for model_name in scoring_models:
            try:
                fit_info = fit_search_model(X_tr_g, y_tr_g.to_numpy(), g_tr_g.to_numpy(), model_name=model_name, target_transform="raw", random_state=random_state + rep)
                pred_va = np.asarray(fit_info["best_estimator"].predict(X_va)).reshape(-1)
                for idx_global, pred in zip(va_idx, pred_va):
                    pred_lists[int(idx_global)].append(float(pred))
            except Exception:
                continue
    fallback = float(np.nanmedian(y_raw))
    return pd.Series([float(np.mean(pred_lists.get(i, [fallback]))) for i in range(len(y_raw))], name="model_aware_oof_pred")

def select_best_y_within_group(X_raw, y_raw, groups_raw, xkey_raw, rough_cols, strategy="hard_closest_oof", random_state=42):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").astype(str).reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="XKEY").astype(str).reset_index(drop=True)
    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)
    if strategy == "singleton_raw":
        tmp["selected_as_final_y"] = 1
        tmp["selected_strategy"] = strategy
        tmp["group_median_y"] = tmp.groupby("GROUP_ID")["target"].transform("median")
        tmp["abs_to_group_median"] = (tmp["target"] - tmp["group_median_y"]).abs()
        tmp["model_aware_oof_pred"] = np.nan
        tmp["abs_resid_to_oof"] = np.nan
        selected_df = tmp.groupby("GROUP_ID", sort=False).head(1).reset_index(drop=True)
        return selected_df, tmp
    if strategy == "hard_closest_oof":
        oof_pred = get_model_aware_oof_predictions(X_raw, y_raw, groups_raw, rough_cols=rough_cols, random_state=random_state)
        tmp = pd.concat([tmp, oof_pred], axis=1)
        tmp["abs_resid_to_oof"] = (tmp["target"] - tmp["model_aware_oof_pred"]).abs()
        tmp["group_median_y"] = tmp.groupby("GROUP_ID")["target"].transform("median")
        tmp["abs_to_group_median"] = (tmp["target"] - tmp["group_median_y"]).abs()
        tmp = tmp.sort_values(["GROUP_ID", "abs_resid_to_oof", "abs_to_group_median"]).copy()
        chosen = tmp.groupby("GROUP_ID", sort=False).head(1).copy()
        chosen["selected_as_final_y"] = 1
        chosen["selected_strategy"] = strategy
        tmp["selected_as_final_y"] = 0
        tmp.loc[chosen.index, "selected_as_final_y"] = 1
        tmp["selected_strategy"] = strategy
        return chosen.reset_index(drop=True), tmp.reset_index(drop=True)
    # robust representative
    selected_rows = []
    candidate_rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        rep_y = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        chosen = block.iloc[[0]].copy()
        chosen["target"] = rep_y
        chosen["selected_as_final_y"] = 1
        chosen["selected_strategy"] = strategy
        chosen["group_median_y"] = float(pd.Series(block["target"]).median())
        chosen["abs_to_group_median"] = abs(rep_y - chosen["group_median_y"].iloc[0])
        chosen["model_aware_oof_pred"] = np.nan
        chosen["abs_resid_to_oof"] = np.nan
        selected_rows.append(chosen)
        b = block.copy()
        b["group_representative_y"] = rep_y
        b["selected_as_final_y"] = 0
        b["selected_strategy"] = strategy
        b["group_median_y"] = float(pd.Series(block["target"]).median())
        b["abs_to_group_median"] = (b["target"] - b["group_median_y"]).abs()
        b["model_aware_oof_pred"] = np.nan
        b["abs_resid_to_oof"] = np.nan
        candidate_rows.append(b)
    return pd.concat(selected_rows, axis=0).reset_index(drop=True), pd.concat(candidate_rows, axis=0).reset_index(drop=True)

def aggregate_test_groups_by_strategy(X_raw, y_raw, groups_raw, xkey_raw, strategy="hard_closest_oof"):
    X_raw = X_raw.reset_index(drop=True)
    y_raw = pd.Series(np.asarray(y_raw, dtype=float), name="target").reset_index(drop=True)
    groups_raw = pd.Series(groups_raw, name="GROUP_ID").astype(str).reset_index(drop=True)
    xkey_raw = pd.Series(xkey_raw, name="XKEY").astype(str).reset_index(drop=True)
    tmp = pd.concat([X_raw.copy(), y_raw, groups_raw, xkey_raw], axis=1)
    rows = []
    for g, block in tmp.groupby("GROUP_ID", sort=False):
        row = block.iloc[[0]].copy()
        if strategy == "robust_trimmed_median":
            row["target"] = robust_group_target(block["target"].to_numpy(), z=ROBUST_TRIM_Z)
        elif strategy == "singleton_raw":
            row["target"] = float(block["target"].iloc[0])
        else:
            row["target"] = float(pd.Series(block["target"]).median())
        rows.append(row)
    return pd.concat(rows, axis=0).reset_index(drop=True)

def build_feature_frequency(X_df, y, groups, candidate_cols, n_repeats=24, random_state=42):
    candidate_cols = list(candidate_cols)
    freq = pd.Series(0.0, index=candidate_cols, dtype=float)
    if not candidate_cols:
        return pd.DataFrame({"feature_name": [], "selection_frequency": []})
    splits = list(repeated_group_splits(groups, n_splits=min(5, len(pd.unique(groups))), n_repeats=n_repeats, random_state=random_state))
    if len(splits) == 0:
        return pd.DataFrame({"feature_name": candidate_cols, "selection_frequency": np.zeros(len(candidate_cols))})
    for rep, (tr_idx, va_idx) in splits:
        X_tr = X_df.iloc[tr_idx][candidate_cols].copy()
        y_tr = np.asarray(y)[tr_idx]
        if len(np.unique(y_tr)) < 2:
            continue
        try:
            pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", ElasticNetCV(l1_ratio=[0.1,0.3,0.5,0.7,0.9], alphas=np.logspace(-4,1,40), cv=min(5, len(y_tr)), random_state=random_state+rep, max_iter=30000))])
            pipe.fit(X_tr, y_tr)
            coef = np.abs(pipe.named_steps["model"].coef_)
            for f, c in zip(candidate_cols, coef):
                if abs(c) > 1e-12:
                    freq[f] += 1.0
        except Exception:
            continue
    if freq.max() > 0:
        freq = freq / freq.max()
    out = freq.sort_values(ascending=False).rename("selection_frequency").reset_index()
    out.columns = ["feature_name", "selection_frequency"]
    return out

def corr_prune_from_ranked(X_df, ranked_features, keep_k=20, threshold=0.90):
    ranked_features = [f for f in ranked_features if f in X_df.columns]
    if len(ranked_features) <= 1:
        return ranked_features[:keep_k]
    corr = X_df[ranked_features].corr(method="spearman").abs().fillna(0.0)
    selected = []
    for feat in ranked_features:
        if all(corr.loc[feat, chosen] < threshold for chosen in selected):
            selected.append(feat)
        if len(selected) >= keep_k:
            break
    return selected

def choose_best_model_and_topk(X_train, y_train, g_train, ranked_features, output_name="", target_transform_candidates=None, model_names=None, random_state=42):
    if target_transform_candidates is None:
        target_transform_candidates = TARGET_TRANSFORM_CANDIDATES
    if model_names is None:
        model_names = FINAL_MODEL_NAMES
    rows = []
    fam = classify_output_family(output_name)
    for topk in get_topk_candidates_for_output(output_name):
        feat_list = ranked_features[:min(topk, len(ranked_features))]
        if len(feat_list) < 2:
            continue
        X_sel = X_train[feat_list].copy()
        for tt in target_transform_candidates:
            for model_name in model_names:
                try:
                    fit_info = fit_search_model(X_sel, y_train, g_train, model_name=model_name, target_transform=tt, scoring=get_inner_scoring_for_output(output_name), random_state=random_state)
                    fam_topk_pen = TOPK_PENALTY * (1.25 if fam == "mechanical_energy" else 1.0)
                    stability_objective = (
                        float(fit_info["best_score"]) -
                        SUMMARY_STD_PENALTY * float(fit_info.get("best_score_std", 0.0)) -
                        fam_topk_pen * float(topk) -
                        COMPLEXITY_PENALTY * float(MODEL_COMPLEXITY_RANK.get(model_name, 10))
                    )
                    rows.append({
                        "model_name": model_name,
                        "target_transform": tt,
                        "topk": int(topk),
                        "selected_features": feat_list,
                        "best_estimator": fit_info["best_estimator"],
                        "best_params": fit_info["best_params"],
                        "search_score": float(fit_info["best_score"]),
                        "search_score_std": float(fit_info.get("best_score_std", 0.0)),
                        "stability_objective": float(stability_objective),
                        "train_r2": float(fit_info.get("train_r2", np.nan)),
                        "complexity_rank": int(MODEL_COMPLEXITY_RANK.get(model_name, 10)),
                    })
                except Exception:
                    continue
    df = pd.DataFrame(rows)
    if df.empty:
        return None, df
    best = df.sort_values(["stability_objective", "search_score", "topk", "complexity_rank"], ascending=[False, False, True, True]).iloc[0].to_dict()
    return best, df


In [7]:

# ============================================================
# Cell C2+. New helper methods: SPCA / BlockPCA / Stability / Bagged / MultiTask
# ============================================================

from sklearn.linear_model import MultiTaskElasticNetCV, LassoCV
from sklearn.base import BaseEstimator, RegressorMixin

# ----------------------------
# small utilities
# ----------------------------
def safe_power_name(name):
    return "yeo-johnson" if str(name).lower().startswith("yeo") else None

def make_y_transformer(name):
    if str(name).lower().startswith("yeo"):
        return PowerTransformer(method="yeo-johnson", standardize=True)
    return None

def build_regressor(model_name, random_state=42, n_components=None):
    if model_name == "Ridge":
        return Ridge(alpha=1.0, random_state=random_state)
    if model_name == "Huber":
        return HuberRegressor(epsilon=1.35, alpha=0.0001)
    if model_name == "PLS":
        return PLSRegression(n_components=int(max(1, n_components or 2)), scale=False)
    raise ValueError(model_name)

def fit_predict_single_model(X_train, y_train, X_test, model_name="Ridge", y_transform="raw", random_state=42, n_components=None):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    reg = build_regressor(model_name, random_state=random_state, n_components=n_components)
    pipe = Pipeline(steps + [("model", reg)])
    ytfm = make_y_transformer(y_transform)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred

def score_prediction(y_true, y_pred):
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < MIN_VALID_EVAL_SAMPLES:
        return np.nan, np.nan, np.nan
    yt = np.asarray(y_true)[mask]
    yp = np.asarray(y_pred)[mask]
    return (
        float(r2_score(yt, yp)),
        float(math.sqrt(mean_squared_error(yt, yp))),
        float(mean_absolute_error(yt, yp))
    )

def inner_group_cv_score(X, y, groups, model_name="Ridge", y_transform="raw", n_components=None,
                         n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        # drop invalid rows
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_single_model(Xtr, ytr, Xte, model_name=model_name, y_transform=y_transform,
                                                 random_state=random_state + cv_run_id, n_components=n_components)
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df) == 1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }

def get_prefilter_topk_for_output_local(output_name):
    fam = classify_output_family(output_name)
    return int(ROUGH_PREFILTER_TOPK_BY_FAMILY.get(fam, ROUGH_PREFILTER_TOPK))

# ----------------------------
# feature set builders
# ----------------------------
def build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, cv_run_id):
    fam = classify_output_family(output_name)
    prefilter_topk = get_prefilter_topk_for_output_local(output_name)

    final_prefilter_cols, final_score_df = prefilter_features_groupwise(
        X_train_sel, y_train_sel, g_train_sel,
        top_k=prefilter_topk,
        corr_threshold=CORR_PRUNE_THRESHOLD,
        random_state=RANDOM_STATE + 1000 + cv_run_id
    )
    if len(final_prefilter_cols) < 2:
        return [], pd.DataFrame()

    freq_df = build_feature_frequency(
        X_train_sel, y_train_sel, g_train_sel,
        candidate_cols=final_prefilter_cols,
        n_repeats=24,
        random_state=RANDOM_STATE + 2000 + cv_run_id
    )
    rank_df = final_score_df[["feature_name", "ensemble_score"]].merge(freq_df, on="feature_name", how="left").fillna(0.0)
    rank_df["rank_score"] = 0.55 * minmax_series(rank_df["selection_frequency"]) + 0.45 * minmax_series(rank_df["ensemble_score"])
    rank_df = rank_df.sort_values("rank_score", ascending=False).reset_index(drop=True)
    ranked = rank_df["feature_name"].tolist()
    ranked = corr_prune_from_ranked(
        X_train_sel[ranked], ranked,
        keep_k=max(12, MAX_FINAL_FEATURES),
        threshold=CORR_PRUNE_THRESHOLD
    )
    return ranked, rank_df

def supervised_screen_features(X_train, y_train, max_keep=16):
    cols = list(X_train.columns)
    rows = []
    for c in cols:
        x = pd.to_numeric(X_train[c], errors="coerce")
        mask = np.isfinite(x) & np.isfinite(y_train)
        if mask.sum() < 5:
            continue
        try:
            pr = np.corrcoef(x[mask], y_train[mask])[0,1]
        except Exception:
            pr = np.nan
        try:
            sp = spearmanr(x[mask], y_train[mask]).correlation
        except Exception:
            sp = np.nan
        rows.append((c, abs(0.55*(0 if pd.isna(pr) else pr) + 0.45*(0 if pd.isna(sp) else sp))))
    if not rows:
        return []
    sdf = pd.DataFrame(rows, columns=["feature","score"]).sort_values("score", ascending=False)
    return sdf["feature"].head(max_keep).tolist()

def fit_spca_transform(X_train, y_train, screen_keep=16, n_components=2):
    screen_feats = supervised_screen_features(X_train, y_train, max_keep=screen_keep)
    if len(screen_feats) < 2:
        return None
    prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    Xp = prep.fit_transform(X_train[screen_feats])
    pca = PCA(n_components=min(n_components, Xp.shape[1], max(1, Xp.shape[0]-1)), random_state=RANDOM_STATE)
    Z = pca.fit_transform(Xp)
    return {"prep": prep, "pca": pca, "features": screen_feats, "Z_train": Z}

def transform_spca(obj, X):
    Xp = obj["prep"].transform(X[obj["features"]])
    return pd.DataFrame(obj["pca"].transform(Xp), index=X.index, columns=[f"SPC{i+1}" for i in range(obj["pca"].n_components_)])

def build_corr_blocks(X_df, threshold=0.65):
    corr = X_df.corr().abs().fillna(0.0)
    cols = list(corr.columns)
    unassigned = set(cols)
    blocks = []
    while unassigned:
        c = next(iter(unassigned))
        grp = [k for k in cols if (corr.loc[c, k] >= threshold)]
        grp = [g for g in grp if g in unassigned]
        if len(grp) == 0:
            grp = [c]
        blocks.append(sorted(grp))
        for g in grp:
            if g in unassigned:
                unassigned.remove(g)
    return blocks

def fit_block_pca_transform(X_train, ranked_features, corr_threshold=0.65, max_blocks=8):
    use_feats = ranked_features[:min(len(ranked_features), 20)]
    if len(use_feats) < 2:
        return None
    X_use = X_train[use_feats].copy()
    prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    Xp = pd.DataFrame(prep.fit_transform(X_use), columns=use_feats, index=X_use.index)
    blocks = build_corr_blocks(Xp, threshold=corr_threshold)[:max_blocks]
    models = []
    Z_list = []
    names = []
    for bi, block in enumerate(blocks):
        pca = PCA(n_components=1, random_state=RANDOM_STATE)
        z = pca.fit_transform(Xp[block])
        models.append((block, pca))
        Z_list.append(z.reshape(-1,1))
        names.append(f"BC{bi+1}")
    Z = np.hstack(Z_list) if Z_list else np.empty((len(X_train),0))
    return {"prep": prep, "models": models, "feature_names": names, "Z_train": Z}

def transform_block_pca(obj, X):
    cols = obj["prep"].feature_names_in_
    Xp = pd.DataFrame(obj["prep"].transform(X[list(cols)]), columns=list(cols), index=X.index)
    arrs = []
    for block, pca in obj["models"]:
        arrs.append(pca.transform(Xp[block]).reshape(-1,1))
    Z = np.hstack(arrs) if arrs else np.empty((len(X),0))
    return pd.DataFrame(Z, index=X.index, columns=obj["feature_names"])

def stability_select_features(X_train, y_train, groups, ranked_features, top_keep=6, n_boot=60, subsample=0.80, random_state=42):
    feats = ranked_features[:min(len(ranked_features), 18)]
    if len(feats) < 2:
        return []
    rng = np.random.RandomState(random_state)
    uniq = np.array(pd.unique(groups))
    counts = Counter()
    for b in range(n_boot):
        n_take = max(4, int(len(uniq) * subsample))
        sel_groups = rng.choice(uniq, size=n_take, replace=False)
        mask = pd.Series(groups).isin(sel_groups).to_numpy()
        Xb = X_train.loc[mask, feats].copy()
        yb = np.asarray(y_train)[mask]
        valid = np.isfinite(yb)
        if valid.sum() < 8:
            continue
        Xb = Xb.loc[valid]
        yb = yb[valid]
        try:
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("lasso", LassoCV(cv=4, random_state=random_state + b, n_alphas=40, max_iter=10000))
            ])
            pipe.fit(Xb, yb)
            coef = np.asarray(pipe.named_steps["lasso"].coef_).reshape(-1)
            for f, c in zip(feats, coef):
                if abs(c) > 1e-10:
                    counts[f] += 1
        except Exception:
            continue
    if not counts:
        return feats[:min(top_keep, len(feats))]
    freq = pd.DataFrame({"feature": feats, "count":[counts.get(f,0) for f in feats]})
    freq["prob"] = freq["count"] / max(1, n_boot)
    freq = freq.sort_values(["prob"], ascending=False)
    out = freq.loc[freq["prob"] >= 0.35, "feature"].tolist()
    if len(out) < 2:
        out = freq["feature"].head(min(top_keep, len(freq))).tolist()
    return out[:min(top_keep, len(out))]

class BaggedSubspaceRidge(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators=40, feature_frac=0.7, alpha=1.0, random_state=42):
        self.n_estimators = n_estimators
        self.feature_frac = feature_frac
        self.alpha = alpha
        self.random_state = random_state

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        self.columns_ = list(X.columns)
        rng = np.random.RandomState(self.random_state)
        self.models_ = []
        min_k = max(2, int(len(self.columns_) * self.feature_frac))
        for i in range(self.n_estimators):
            feat_idx = rng.choice(len(self.columns_), size=min_k, replace=False)
            feats = [self.columns_[j] for j in feat_idx]
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=self.alpha, random_state=self.random_state + i))
            ])
            pipe.fit(X[feats], y)
            self.models_.append((feats, pipe))
        return self

    def predict(self, X):
        X = pd.DataFrame(X).copy()
        preds = []
        for feats, pipe in self.models_:
            preds.append(np.asarray(pipe.predict(X[feats])).reshape(-1))
        return np.mean(np.vstack(preds), axis=0)

def fit_predict_bagged_ridge(X_train, y_train, X_test, y_transform="raw", random_state=42):
    ytfm = make_y_transformer(y_transform)
    model = BaggedSubspaceRidge(n_estimators=40, feature_frac=0.7, alpha=1.0, random_state=random_state)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=model, transformer=ytfm)
    else:
        est = model
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred


# ----------------------------
# family multi-task screen
# ----------------------------
def build_family_multitask_feature_map():
    fam_map = {}
    candidate_outputs = list(DATA_BY_OUTPUT.keys())
    families = {"thermal": [], "vibrational": []}

    for out in candidate_outputs:
        fam = classify_output_family(out)
        if fam in families:
            families[fam].append(out)

    for fam, outs in families.items():
        if len(outs) < 2:
            continue

        tmp_frames = []

        for out in outs:
            bundle = DATA_BY_OUTPUT[out]
            d = bundle["df"].copy()

            xkey_col = bundle.get("xkey_col", "XKEY")
            group_col = bundle.get("group_col", "GROUP_ID")
            target_col = bundle["target_col"]

            missing_cols = [c for c in [xkey_col, group_col, target_col] if c not in d.columns]
            if missing_cols:
                print(f"[build_family_multitask_feature_map] skip {out} | missing columns: {missing_cols}")
                continue

            tmp = d[[xkey_col, group_col, target_col]].copy()
            tmp.columns = ["XKEY", "GROUP_ID", out]

            tmp["XKEY"] = tmp["XKEY"].astype(str)
            tmp["GROUP_ID"] = tmp["GROUP_ID"].astype(str)
            tmp[out] = pd.to_numeric(tmp[out], errors="coerce")

            tmp_frames.append(tmp)

        if len(tmp_frames) < 2:
            continue

        merged = tmp_frames[0]
        for t in tmp_frames[1:]:
            merged = merged.merge(t, on=["XKEY", "GROUP_ID"], how="inner")

        if not merged.empty:
            fam_map[fam] = merged.reset_index(drop=True)

    return fam_map

FAMILY_MT_DATA = build_family_multitask_feature_map()

def multitask_screen_features_for_family(output_name, X_train_sel, selected_train_df, top_keep=10, random_state=42):
    fam = classify_output_family(output_name)
    if fam not in FAMILY_MT_DATA:
        return []

    fam_df = FAMILY_MT_DATA[fam].copy()
    if fam_df.empty:
        return []

    if "GROUP_ID" not in selected_train_df.columns:
        return []

    tr_groups = set(selected_train_df["GROUP_ID"].astype(str).unique())
    fam_df["GROUP_ID"] = fam_df["GROUP_ID"].astype(str)
    fam_df = fam_df[fam_df["GROUP_ID"].isin(tr_groups)].copy()
    if fam_df.empty:
        return []

    out_cols = [c for c in fam_df.columns if c not in ("XKEY", "GROUP_ID")]
    if len(out_cols) < 2:
        return []

    if "XKEY" not in selected_train_df.columns:
        bundle = DATA_BY_OUTPUT[output_name]
        xkey_col = bundle.get("xkey_col", "XKEY")
        if xkey_col in selected_train_df.columns:
            base = selected_train_df.copy().rename(columns={xkey_col: "XKEY"})
        else:
            return []
    else:
        base = selected_train_df.copy()

    feature_cols = list(X_train_sel.columns)
    keep_cols = ["XKEY"] + [c for c in feature_cols if c in base.columns]
    base = base[keep_cols].drop_duplicates(subset=["XKEY"]).copy()
    base["XKEY"] = base["XKEY"].astype(str)

    merged = fam_df.merge(base, on="XKEY", how="inner")
    if merged.shape[0] < 12:
        return []

    Y = merged[out_cols].apply(pd.to_numeric, errors="coerce")
    valid_y = Y.notna().all(axis=1)
    merged = merged.loc[valid_y].reset_index(drop=True)
    if merged.shape[0] < 12:
        return []

    Y = merged[out_cols].to_numpy(dtype=float)
    X = merged[feature_cols].copy()

    valid_x = ~X.isna().all(axis=1)
    X = X.loc[valid_x].reset_index(drop=True)
    Y = Y[valid_x.to_numpy()]

    if X.shape[0] < 12:
        return []

    try:
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("mt", MultiTaskElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                alphas=None,
                n_alphas=40,
                cv=min(5, max(2, X.shape[0] // 4)),
                random_state=random_state,
                max_iter=15000
            ))
        ])
        pipe.fit(X, Y)

        coef = np.asarray(pipe.named_steps["mt"].coef_)
        score = np.sqrt((coef ** 2).sum(axis=0))

        feat_df = pd.DataFrame({
            "feature": list(X.columns),
            "score": score
        }).sort_values("score", ascending=False)

        return feat_df["feature"].head(min(top_keep, len(feat_df))).tolist()

    except Exception as e:
        print(f"[multitask_screen_features_for_family] {output_name} failed: {type(e).__name__}: {e}")
        return []

# ----------------------------
# Method candidates
# ----------------------------
def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = [
        "baseline_stability",
        "stability_lasso_ridge",
        "spca_ridge",
        "spca_huber",
        "block_pca_ridge",
        "bagged_subspace_ridge",
        "minimal_class_average",
    ]
    if fam in ("thermal", "vibrational"):
        methods += ["multitask_screen_ridge", "multitask_screen_pls"]
    if fam == "mechanical_energy":
        methods += ["spca_pls"]
    return methods

def choose_within_tolerance(rows_df, tol_frac=0.03):
    if rows_df.empty:
        return None
    best = rows_df["score_obj"].max()
    keep = rows_df[rows_df["score_obj"] >= best - abs(best) * tol_frac].copy()
    keep = keep.sort_values(["score_obj", "n_features", "complexity_rank"], ascending=[False, True, True]).reset_index(drop=True)
    return keep.iloc[0].to_dict()

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    rows = []
    fam = classify_output_family(output_name)

    if method_name == "baseline_stability":
        best_choice, model_search_df = choose_best_model_and_topk(
            X_train=X_train_sel[ranked_features],
            y_train=y_train_sel,
            g_train=g_train_sel,
            ranked_features=ranked_features,
            output_name=output_name,
            target_transform_candidates=TARGET_TRANSFORM_CANDIDATES,
            model_names=get_candidate_models_for_output(output_name),
            random_state=random_state
        )
        if best_choice is None:
            return None
        feats = list(best_choice["selected_features"])
        Xtr = X_train_sel[feats].copy()
        Xte = X_test_group[feats].copy()
        pred = np.asarray(best_choice["best_estimator"].predict(Xte)).reshape(-1)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {
            "method_name": method_name,
            "model_name": best_choice["model_name"],
            "target_transform": best_choice["target_transform"],
            "topk": int(best_choice["topk"]),
            "selected_features": feats,
            "n_features": len(feats),
            "complexity_rank": int(MODEL_COMPLEXITY_RANK.get(best_choice["model_name"], 10)),
            "search_score": float(best_choice["search_score"]),
            "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae,
            "fitted_estimator": best_choice["best_estimator"],
        }

    if method_name == "stability_lasso_ridge":
        feats = stability_select_features(X_train_sel, y_train_sel, g_train_sel, ranked_features, top_keep=6, random_state=random_state)
        if len(feats) < 2:
            return None
        best = None
        for tt in TARGET_TRANSFORM_CANDIDATES:
            inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                         n_splits=5, n_repeats=1, random_state=random_state)
            score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
            row = {"tt": tt, "inner": inner, "score_obj": score_obj}
            if (best is None) or (score_obj > best["score_obj"]):
                best = row
        est, pred = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_test_group[feats], model_name="Ridge",
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name": method_name, "model_name":"Ridge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features": feats, "n_features":len(feats), "complexity_rank":2, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":est}

    if method_name in ("spca_ridge","spca_huber","spca_pls"):
        base_model = {"spca_ridge":"Ridge", "spca_huber":"Huber", "spca_pls":"PLS"}[method_name]
        cands = []
        for screen_keep in [8, 12, 16]:
            for nc in [1, 2, 3]:
                obj = fit_spca_transform(X_train_sel[ranked_features], y_train_sel, screen_keep=screen_keep, n_components=nc)
                if obj is None:
                    continue
                Ztr = transform_spca(obj, X_train_sel[ranked_features])
                Zte = transform_spca(obj, X_test_group[ranked_features])
                for tt in TARGET_TRANSFORM_CANDIDATES:
                    inner = inner_group_cv_score(Ztr, y_train_sel, g_train_sel, model_name=base_model, y_transform=tt,
                                                 n_components=min(nc, Ztr.shape[1]), n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.005*nc
                    cands.append({"obj":obj, "Ztr":Ztr, "Zte":Zte, "tt":tt, "nc":nc, "inner":inner, "score_obj":score_obj})
        if not cands:
            return None
        cdf = pd.DataFrame([{"score_obj":c["score_obj"], "nc":c["nc"], "std":c["inner"]["std_r2"]} for c in cands])
        best = cands[int(cdf["score_obj"].idxmax())]
        est, pred = fit_predict_single_model(best["Ztr"], y_train_sel, best["Zte"], model_name=base_model,
                                             y_transform=best["tt"], random_state=random_state, n_components=min(best["nc"], best["Ztr"].shape[1]))
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = list(best["obj"]["features"])
        return {"method_name":method_name, "model_name":base_model, "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":5 if base_model=="PLS" else 2,
                "search_score":best["inner"]["mean_r2"], "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae,
                "fitted_estimator": {"spca_obj": best["obj"], "reg_model": est}}

    if method_name == "block_pca_ridge":
        best = None
        for th in [0.55, 0.65, 0.75]:
            obj = fit_block_pca_transform(X_train_sel, ranked_features, corr_threshold=th, max_blocks=8)
            if obj is None or obj["Z_train"].shape[1] < 2:
                continue
            Ztr = transform_block_pca(obj, X_train_sel)
            Zte = transform_block_pca(obj, X_test_group)
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(Ztr, y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                             n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.005*Ztr.shape[1]
                cand = {"obj":obj, "Ztr":Ztr, "Zte":Zte, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        est, pred = fit_predict_single_model(best["Ztr"], y_train_sel, best["Zte"], model_name="Ridge",
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = sum([blk for blk, _ in best["obj"]["models"]], [])
        return {"method_name":method_name, "model_name":"Ridge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":3, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":{"block_obj":best["obj"], "reg_model":est}}

    if method_name == "bagged_subspace_ridge":
        cands = []
        feats = ranked_features[:min(len(ranked_features), 8)]
        if len(feats) < 2:
            return None
        for tt in TARGET_TRANSFORM_CANDIDATES:
            # simple inner scoring with repeated fits
            inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name="Ridge", y_transform=tt,
                                         n_splits=5, n_repeats=1, random_state=random_state)
            score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
            cands.append({"tt":tt, "inner":inner, "score_obj":score_obj})
        best = max(cands, key=lambda z: z["score_obj"])
        est, pred = fit_predict_bagged_ridge(X_train_sel[feats], y_train_sel, X_test_group[feats],
                                             y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name":method_name, "model_name":"BaggedSubspaceRidge", "target_transform":best["tt"], "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":4, "search_score":best["inner"]["mean_r2"],
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":est}

    if method_name in ("multitask_screen_ridge","multitask_screen_pls"):
        base_model = "Ridge" if method_name.endswith("ridge") else "PLS"
        mt_feats = multitask_screen_features_for_family(output_name, X_train_sel[ranked_features], selected_train_df, top_keep=8, random_state=random_state)
        if len(mt_feats) < 2:
            return None
        best = None
        for topk in [2,3,4,6]:
            feats = mt_feats[:min(topk, len(mt_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel, model_name=base_model, y_transform=tt,
                                             n_components=min(3, len(feats)), n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        est, pred = fit_predict_single_model(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
                                             model_name=base_model, y_transform=best["tt"],
                                             random_state=random_state, n_components=min(3, len(best["feats"])))
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        return {"method_name":method_name, "model_name":base_model, "target_transform":best["tt"], "topk":len(best["feats"]),
                "selected_features":best["feats"], "n_features":len(best["feats"]), "complexity_rank":4 if base_model=="PLS" else 2,
                "search_score":best["inner"]["mean_r2"], "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae,
                "fitted_estimator":est}

    if method_name == "minimal_class_average":
        # build a small class of close-performing models and average them
        best_choice, search_df = choose_best_model_and_topk(
            X_train=X_train_sel[ranked_features],
            y_train=y_train_sel,
            g_train=g_train_sel,
            ranked_features=ranked_features,
            output_name=output_name,
            target_transform_candidates=TARGET_TRANSFORM_CANDIDATES,
            model_names=get_candidate_models_for_output(output_name),
            random_state=random_state
        )
        if best_choice is None or search_df.empty:
            return None
        sdf = search_df.copy().sort_values(["stability_objective", "search_score_std", "topk"], ascending=[False, True, True]).reset_index(drop=True)
        best_obj = float(sdf["stability_objective"].max())
        sdf = sdf[sdf["stability_objective"] >= best_obj - abs(best_obj)*0.03].head(3).copy()
        if sdf.empty:
            return None
        preds = []
        ests = []
        for _, row in sdf.iterrows():
            feats = list(row["selected_features"])
            try:
                est, pred = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_test_group[feats],
                                                     model_name=row["model_name"].replace("_Regression","").replace("PCR_Ridge","Ridge") if row["model_name"] not in ("PLS_Regression","PCR_Ridge") else ("PLS" if row["model_name"]=="PLS_Regression" else "Ridge"),
                                                     y_transform=row["target_transform"], random_state=random_state,
                                                     n_components=min(3, len(feats)))
            except Exception:
                # fallback to existing estimator if compatible
                try:
                    pred = np.asarray(row["best_estimator"].predict(X_test_group[feats])).reshape(-1)
                    est = row["best_estimator"]
                except Exception:
                    continue
            preds.append(pred)
            ests.append({"feats":feats, "name":row["model_name"], "est":est, "w":max(1e-6, float(row["stability_objective"]))})
        if len(preds) < 2:
            return None
        W = np.array([e["w"] for e in ests], dtype=float)
        W = W / W.sum()
        pred = np.average(np.vstack(preds), axis=0, weights=W)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        feats = sorted(set(sum([e["feats"] for e in ests], [])))
        return {"method_name":method_name, "model_name":"MinimalClassAverage", "target_transform":"mixed", "topk":len(feats),
                "selected_features":feats, "n_features":len(feats), "complexity_rank":6, "search_score":float(np.mean(W)),
                "outer_r2":r2, "outer_rmse":rmse, "outer_mae":mae, "fitted_estimator":{"members":ests, "weights":W}}

    return None



# ============================================================
# Cell C2++. Additional untried methods: ARD / OMP / Quantile / Residual-ExtraTrees
# ============================================================

from sklearn.linear_model import ARDRegression, OrthogonalMatchingPursuitCV, QuantileRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.base import BaseEstimator, RegressorMixin, clone

_BASE_GET_METHOD_CANDIDATES_FOR_OUTPUT = get_method_candidates_for_output
_BASE_EVALUATE_METHOD_TRAIN_TEST = evaluate_method_train_test

class OMPThenRidge(BaseEstimator, RegressorMixin):
    def __init__(self, max_nonzero_coefs=None, alpha=1.0):
        self.max_nonzero_coefs = max_nonzero_coefs
        self.alpha = alpha

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.selector_ = OrthogonalMatchingPursuitCV(max_iter=self.max_nonzero_coefs)
        self.selector_.fit(X, y)
        coef = np.asarray(self.selector_.coef_).reshape(-1)
        self.support_idx_ = np.where(np.abs(coef) > 1e-12)[0]
        if len(self.support_idx_) == 0:
            self.support_idx_ = np.array([int(np.argmax(np.abs(coef)))]) if coef.size else np.array([], dtype=int)
        self.reg_ = Ridge(alpha=self.alpha)
        if len(self.support_idx_) > 0:
            self.reg_.fit(X[:, self.support_idx_], y)
        else:
            self.reg_.fit(X, y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if hasattr(self, "support_idx_") and len(self.support_idx_) > 0:
            return self.reg_.predict(X[:, self.support_idx_])
        return self.reg_.predict(X)


class ResidualExtraTreesRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_model="ridge", n_components=2, random_state=42):
        self.base_model = base_model
        self.n_components = n_components
        self.random_state = random_state

    def _make_base(self):
        if str(self.base_model).lower().startswith("pls"):
            return PLSRegression(n_components=int(max(1, self.n_components)), scale=False)
        return Ridge(alpha=1.0, random_state=self.random_state)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.base_ = self._make_base()
        self.base_.fit(X, y)
        base_pred = np.asarray(self.base_.predict(X)).reshape(-1)
        resid = y - base_pred
        self.resid_model_ = ExtraTreesRegressor(
            n_estimators=250,
            max_depth=3,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=self.random_state
        )
        self.resid_model_.fit(X, resid)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        base_pred = np.asarray(self.base_.predict(X)).reshape(-1)
        resid_pred = np.asarray(self.resid_model_.predict(X)).reshape(-1)
        return base_pred + resid_pred


def fit_predict_custom_estimator(X_train, y_train, X_test, estimator_obj, y_transform="raw", random_state=42):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", estimator_obj)]
    pipe = Pipeline(steps)
    ytfm = make_y_transformer(y_transform)
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred


def inner_group_cv_score_custom(X, y, groups, estimator_factory, y_transform="raw",
                                n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_custom_estimator(
                Xtr, ytr, Xte,
                estimator_obj=estimator_factory(random_state=random_state + cv_run_id),
                y_transform=y_transform,
                random_state=random_state + cv_run_id
            )
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df) == 1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }


def _custom_method_feature_grid(output_name):
    fam = classify_output_family(output_name)
    if fam == "mechanical_energy":
        return [2, 3, 4, 6]
    return [2, 3, 4, 6, 8]


def get_method_candidates_for_output(output_name):
    methods = _BASE_GET_METHOD_CANDIDATES_FOR_OUTPUT(output_name)
    fam = classify_output_family(output_name)
    extra = [
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
    ]
    if fam in ("thermal", "vibrational"):
        extra += [
            "direct_quantile_median",
            "residual_ridge_extratrees",
            "residual_pls_extratrees",
        ]
    else:
        extra += [
            "residual_ridge_extratrees",
        ]
    # deduplicate while preserving order
    seen = set()
    out = []
    for m in methods + extra:
        if m not in seen:
            out.append(m)
            seen.add(m)
    return out


def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    # handle new methods first
    custom_methods = {
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
        "direct_quantile_median",
        "residual_ridge_extratrees",
        "residual_pls_extratrees",
    }
    if method_name not in custom_methods:
        return _BASE_EVALUATE_METHOD_TRAIN_TEST(
            method_name=method_name,
            output_name=output_name,
            X_train_sel=X_train_sel,
            y_train_sel=y_train_sel,
            g_train_sel=g_train_sel,
            selected_train_df=selected_train_df,
            X_test_group=X_test_group,
            y_test_group=y_test_group,
            ranked_features=ranked_features,
            random_state=random_state,
        )

    fam = classify_output_family(output_name)
    feature_grid = _custom_method_feature_grid(output_name)

    def factory_for(method_name, n_feats):
        if method_name == "direct_ard":
            return lambda random_state=42: ARDRegression()
        if method_name == "direct_elasticnet_cv":
            return lambda random_state=42: ElasticNetCV(
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                cv=4,
                random_state=random_state,
                n_alphas=60,
                n_jobs=CPU_TREE_N_JOBS, max_iter=15000
            )
        if method_name == "direct_omp_ridge":
            return lambda random_state=42: OMPThenRidge(max_nonzero_coefs=min(max(2, n_feats), 6), alpha=1.0)
        if method_name == "direct_quantile_median":
            return lambda random_state=42: QuantileRegressor(
                quantile=0.5,
                alpha=0.01,
                solver="highs"
            )
        if method_name == "residual_ridge_extratrees":
            return lambda random_state=42: ResidualExtraTreesRegressor(base_model="ridge", n_components=min(3, n_feats), random_state=random_state)
        if method_name == "residual_pls_extratrees":
            return lambda random_state=42: ResidualExtraTreesRegressor(base_model="pls", n_components=min(3, n_feats), random_state=random_state)
        raise ValueError(method_name)

    best = None
    target_transforms = TARGET_TRANSFORM_CANDIDATES
    # quantile regression often behaves best without transform; still compare raw/yeo
    for topk in feature_grid:
        feats = ranked_features[:min(topk, len(ranked_features))]
        if len(feats) < 2:
            continue
        Xtr = X_train_sel[feats].copy()
        Xte = X_test_group[feats].copy()

        for tt in target_transforms:
            inner = inner_group_cv_score_custom(
                Xtr, y_train_sel, g_train_sel,
                estimator_factory=factory_for(method_name, len(feats)),
                y_transform=tt,
                n_splits=5,
                n_repeats=1,
                random_state=random_state
            )
            if pd.isna(inner["mean_r2"]):
                continue
            complexity_rank = {
                "direct_ard": 2,
                "direct_elasticnet_cv": 3,
                "direct_omp_ridge": 3,
                "direct_quantile_median": 3,
                "residual_ridge_extratrees": 5,
                "residual_pls_extratrees": 6,
            }[method_name]
            score_obj = (
                inner["mean_r2"]
                - 0.12 * (0 if pd.isna(inner["std_r2"]) else inner["std_r2"])
                - 0.008 * len(feats)
                - 0.01 * complexity_rank
            )
            cand = {
                "feats": feats,
                "tt": tt,
                "inner": inner,
                "score_obj": score_obj,
                "complexity_rank": complexity_rank,
            }
            if (best is None) or (score_obj > best["score_obj"]):
                best = cand

    if best is None:
        return None

    est, pred = fit_predict_custom_estimator(
        X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
        estimator_obj=factory_for(method_name, len(best["feats"]))(random_state=random_state),
        y_transform=best["tt"],
        random_state=random_state
    )
    r2, rmse, mae = score_prediction(y_test_group, pred)
    if not np.isfinite(r2):
        return None

    model_name_map = {
        "direct_ard": "ARD_Regression",
        "direct_elasticnet_cv": "ElasticNet_CV_Direct",
        "direct_omp_ridge": "OMP_then_Ridge",
        "direct_quantile_median": "Quantile_Median",
        "residual_ridge_extratrees": "Residual_Ridge_ExtraTrees",
        "residual_pls_extratrees": "Residual_PLS_ExtraTrees",
    }

    return {
        "method_name": method_name,
        "model_name": model_name_map[method_name],
        "target_transform": best["tt"],
        "topk": len(best["feats"]),
        "selected_features": list(best["feats"]),
        "n_features": len(best["feats"]),
        "complexity_rank": best["complexity_rank"],
        "search_score": float(best["inner"]["mean_r2"]),
        "outer_r2": float(r2),
        "outer_rmse": float(rmse),
        "outer_mae": float(mae),
        "fitted_estimator": est,
    }


# ============================================================
# Cell C2++ Override: safer multitask + extra methods
# ============================================================
from sklearn.linear_model import ARDRegression, ElasticNetCV, OrthogonalMatchingPursuitCV, TheilSenRegressor, RANSACRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RationalQuadratic, WhiteKernel, ConstantKernel
from sklearn.kernel_ridge import KernelRidge

# --- safer family multitask map (override old version) ---
def build_family_multitask_feature_map():
    fam_map = {}
    candidate_outputs = list(DATA_BY_OUTPUT.keys())
    families = {"thermal": [], "vibrational": []}
    for out in candidate_outputs:
        fam = classify_output_family(out)
        if fam in families:
            families[fam].append(out)

    for fam, outs in families.items():
        if len(outs) < 2:
            continue
        tmp_frames = []
        for out in outs:
            bundle = DATA_BY_OUTPUT[out]
            d = bundle["df"].copy()
            xkey_col = bundle.get("xkey_col", "XKEY")
            group_col = bundle.get("group_col", "GROUP_ID")
            target_col = bundle["target_col"]
            missing_cols = [c for c in [xkey_col, group_col, target_col] if c not in d.columns]
            if missing_cols:
                print(f"[build_family_multitask_feature_map] skip {out} | missing columns: {missing_cols}")
                continue
            tmp = d[[xkey_col, group_col, target_col]].copy()
            tmp.columns = ["XKEY", "GROUP_ID", out]
            tmp["XKEY"] = tmp["XKEY"].astype(str)
            tmp["GROUP_ID"] = tmp["GROUP_ID"].astype(str)
            tmp[out] = pd.to_numeric(tmp[out], errors="coerce")
            tmp_frames.append(tmp)
        if len(tmp_frames) < 2:
            continue
        merged = tmp_frames[0]
        for t in tmp_frames[1:]:
            merged = merged.merge(t, on=["XKEY", "GROUP_ID"], how="inner")
        if not merged.empty:
            fam_map[fam] = merged.reset_index(drop=True)
    return fam_map

FAMILY_MT_DATA = build_family_multitask_feature_map()

def multitask_screen_features_for_family(output_name, X_train_sel, selected_train_df, top_keep=10, random_state=42):
    fam = classify_output_family(output_name)
    if fam not in FAMILY_MT_DATA:
        return []
    fam_df = FAMILY_MT_DATA[fam].copy()
    if fam_df.empty or "GROUP_ID" not in selected_train_df.columns:
        return []
    tr_groups = set(selected_train_df["GROUP_ID"].astype(str).unique())
    fam_df["GROUP_ID"] = fam_df["GROUP_ID"].astype(str)
    fam_df = fam_df[fam_df["GROUP_ID"].isin(tr_groups)].copy()
    if fam_df.empty:
        return []
    out_cols = [c for c in fam_df.columns if c not in ("XKEY","GROUP_ID")]
    if len(out_cols) < 2:
        return []
    base = selected_train_df.copy()
    if "XKEY" not in base.columns:
        bundle = DATA_BY_OUTPUT[output_name]
        xkey_col = bundle.get("xkey_col", "XKEY")
        if xkey_col in base.columns:
            base = base.rename(columns={xkey_col: "XKEY"})
        else:
            return []
    feature_cols = list(X_train_sel.columns)
    keep_cols = ["XKEY"] + [c for c in feature_cols if c in base.columns]
    base = base[keep_cols].drop_duplicates(subset=["XKEY"]).copy()
    base["XKEY"] = base["XKEY"].astype(str)
    merged = fam_df.merge(base, on="XKEY", how="inner")
    if merged.shape[0] < 12:
        return []
    Y = merged[out_cols].apply(pd.to_numeric, errors="coerce")
    valid_y = Y.notna().all(axis=1)
    merged = merged.loc[valid_y].reset_index(drop=True)
    if merged.shape[0] < 12:
        return []
    Y = merged[out_cols].to_numpy(dtype=float)
    X = merged[feature_cols].copy()
    valid_x = ~X.isna().all(axis=1)
    X = X.loc[valid_x].reset_index(drop=True)
    Y = Y[valid_x.to_numpy()]
    if X.shape[0] < 12:
        return []
    try:
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("mt", MultiTaskElasticNetCV(
                l1_ratio=[0.1,0.3,0.5,0.7,0.9],
                alphas=None,
                n_alphas=40,
                cv=min(5, max(2, X.shape[0]//4)),
                random_state=random_state,
                max_iter=15000
            ))
        ])
        pipe.fit(X, Y)
        coef = np.asarray(pipe.named_steps["mt"].coef_)
        score = np.sqrt((coef**2).sum(axis=0))
        feat_df = pd.DataFrame({"feature": list(X.columns), "score": score}).sort_values("score", ascending=False)
        return feat_df["feature"].head(min(top_keep, len(feat_df))).tolist()
    except Exception as e:
        print(f"[multitask_screen_features_for_family] {output_name} failed: {type(e).__name__}: {e}")
        return []


def fit_predict_special_method(X_train, y_train, X_test, method_name, y_transform="raw", random_state=42):
    # y_transform intentionally ignored for quantile/theilsen/ransac robust cases except where practical
    if method_name == "direct_ard":
        reg = ARDRegression()
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        est = TransformedTargetRegressor(regressor=pipe, transformer=make_y_transformer(y_transform)) if make_y_transformer(y_transform) is not None else pipe
        est.fit(X_train, y_train)
        return est, np.asarray(est.predict(X_test)).reshape(-1)
    if method_name == "direct_elasticnet_cv":
        reg = ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9,.95,.99], cv=5, random_state=random_state, n_jobs=CPU_TREE_N_JOBS, max_iter=20000)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        est = TransformedTargetRegressor(regressor=pipe, transformer=make_y_transformer(y_transform)) if make_y_transformer(y_transform) is not None else pipe
        est.fit(X_train, y_train)
        return est, np.asarray(est.predict(X_test)).reshape(-1)
    if method_name == "direct_omp_ridge":
        # OMP select then ridge refit
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        omp = OrthogonalMatchingPursuitCV(cv=5)
        omp.fit(Xtr, y_train)
        coef = np.asarray(omp.coef_).reshape(-1)
        idx = np.flatnonzero(np.abs(coef) > 1e-12)
        if len(idx) == 0:
            idx = np.arange(min(2, Xtr.shape[1]))
        ridge = Ridge(alpha=1.0, random_state=random_state)
        ridge.fit(Xtr[:, idx], y_train)
        pred = ridge.predict(Xte[:, idx])
        return {"prep":prep, "omp":omp, "ridge":ridge, "idx":idx, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_theilsen":
        reg = TheilSenRegressor(random_state=random_state, max_subpopulation=1e4)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    if method_name == "direct_ransac_ridge":
        base = Ridge(alpha=1.0, random_state=random_state)
        try:
            reg = RANSACRegressor(estimator=base, random_state=random_state)
        except TypeError:
            reg = RANSACRegressor(base_estimator=base, random_state=random_state)
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    if method_name == "direct_gpr_matern":
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        kernel = ConstantKernel(1.0, (1e-3,1e3)) * Matern(length_scale=np.ones(Xtr.shape[1]), nu=1.5) + WhiteKernel(noise_level=1e-3)
        gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=random_state, n_restarts_optimizer=1)
        gpr.fit(Xtr, y_train)
        pred = gpr.predict(Xte)
        return {"prep":prep, "gpr":gpr, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_gpr_rq":
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train)
        Xte = prep.transform(X_test)
        kernel = ConstantKernel(1.0, (1e-3,1e3)) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1e-3)
        gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=random_state, n_restarts_optimizer=1)
        gpr.fit(Xtr, y_train)
        pred = gpr.predict(Xte)
        return {"prep":prep, "gpr":gpr, "columns":list(X_train.columns)}, np.asarray(pred).reshape(-1)
    if method_name == "direct_quantile_median":
        reg = QuantileRegressor(quantile=0.5, alpha=0.001, solver="highs")
        pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("model", reg)])
        pipe.fit(X_train, y_train)
        return pipe, np.asarray(pipe.predict(X_test)).reshape(-1)
    raise ValueError(method_name)


def _model_complexity_override(name):
    mapping = {
        "ARD_Regression": 2,
        "ElasticNetCV": 2,
        "OMP_Ridge": 3,
        "TheilSen": 3,
        "RANSAC_Ridge": 3,
        "GaussianProcess_Matern": 6,
        "GaussianProcess_RQ": 6,
        "QuantileMedian": 2,
        "Residual_Huber_KRR": 5,
        "Residual_ARD_KRR": 5,
    }
    return mapping.get(name, 10)

# override method candidate function with extra methods
_old_get_method_candidates_for_output = get_method_candidates_for_output

def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = _old_get_method_candidates_for_output(output_name)
    methods += [
        "direct_theilsen",
        "direct_ransac_ridge",
        "direct_gpr_matern",
        "direct_gpr_rq",
        "residual_huber_krr",
        "residual_ard_krr",
    ]
    if fam != "vibrational":
        methods += ["direct_quantile_median"]
    return list(dict.fromkeys(methods))

_old_evaluate_method_train_test = evaluate_method_train_test

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    # keep old behavior first
    old_methods = {
        "baseline_stability","stability_lasso_ridge","spca_ridge","spca_huber","block_pca_ridge",
        "bagged_subspace_ridge","minimal_class_average","multitask_screen_ridge","multitask_screen_pls",
        "spca_pls","direct_ard","direct_elasticnet_cv","direct_omp_ridge","direct_quantile_median",
        "residual_ridge_extratrees","residual_pls_extratrees"
    }
    if method_name in old_methods:
        return _old_evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                               selected_train_df, X_test_group, y_test_group, ranked_features,
                                               random_state=random_state)

    # new direct robust/GP methods
    if method_name in ("direct_theilsen","direct_ransac_ridge","direct_gpr_matern","direct_gpr_rq"):
        best = None
        for topk in [2,3,4,6]:
            feats = ranked_features[:min(topk, len(ranked_features))]
            if len(feats) < 2:
                continue
            # robust methods mostly without transform except GPR optionally
            tt_list = ["raw"] if method_name in ("direct_theilsen","direct_ransac_ridge") else TARGET_TRANSFORM_CANDIDATES
            for tt in tt_list:
                try:
                    # lightweight proxy inner score: use ridge-like scorer or direct fit in split loop once
                    inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel,
                                                 model_name="Ridge", y_transform=tt,
                                                 n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                    cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                    if (best is None) or (score_obj > best["score_obj"]):
                        best = cand
                except Exception:
                    continue
        if best is None:
            return None
        est, pred = fit_predict_special_method(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]],
                                               method_name=method_name, y_transform=best["tt"], random_state=random_state)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        model_name = {
            "direct_theilsen":"TheilSen",
            "direct_ransac_ridge":"RANSAC_Ridge",
            "direct_gpr_matern":"GaussianProcess_Matern",
            "direct_gpr_rq":"GaussianProcess_RQ",
        }[method_name]
        return {"method_name": method_name, "model_name": model_name, "target_transform": best["tt"], "topk": len(best["feats"]),
                "selected_features": best["feats"], "n_features": len(best["feats"]),
                "complexity_rank": _model_complexity_override(model_name), "search_score": best["inner"]["mean_r2"],
                "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae, "fitted_estimator": est}

    if method_name in ("residual_huber_krr", "residual_ard_krr"):
        base_model = "Huber" if method_name == "residual_huber_krr" else "Ridge"
        # ARD approximated via direct special fit for base prediction
        best = None
        for topk in [2,3,4,6]:
            feats = ranked_features[:min(topk, len(ranked_features))]
            if len(feats) < 2:
                continue
            for tt in TARGET_TRANSFORM_CANDIDATES:
                inner = inner_group_cv_score(X_train_sel[feats], y_train_sel, g_train_sel,
                                             model_name=("Huber" if method_name == "residual_huber_krr" else "Ridge"),
                                             y_transform=tt, n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = inner["mean_r2"] - 0.10*(0 if pd.isna(inner["std_r2"]) else inner["std_r2"]) - 0.01*len(feats)
                cand = {"feats":feats, "tt":tt, "inner":inner, "score_obj":score_obj}
                if (best is None) or (score_obj > best["score_obj"]):
                    best = cand
        if best is None:
            return None
        feats = best["feats"]
        # fit base
        if method_name == "residual_huber_krr":
            base_est, base_pred_tr = fit_predict_single_model(X_train_sel[feats], y_train_sel, X_train_sel[feats], model_name="Huber", y_transform=best["tt"], random_state=random_state)
            base_pred_te = np.asarray(base_est.predict(X_test_group[feats])).reshape(-1)
        else:
            base_est, base_pred_tr = fit_predict_special_method(X_train_sel[feats], y_train_sel, X_train_sel[feats], method_name="direct_ard", y_transform=best["tt"], random_state=random_state)
            # predict test for ARD special object
            try:
                base_pred_te = np.asarray(base_est.predict(X_test_group[feats])).reshape(-1)
            except Exception:
                # TransformedTargetRegressor/pipeline path handled above; dict shouldn't happen for direct_ard
                return None
        resid = y_train_sel - np.asarray(base_pred_tr).reshape(-1)
        prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
        Xtr = prep.fit_transform(X_train_sel[feats])
        Xte = prep.transform(X_test_group[feats])
        krr = KernelRidge(alpha=1.0, kernel='rbf', gamma=None)
        krr.fit(Xtr, resid)
        pred = base_pred_te + krr.predict(Xte)
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2):
            return None
        model_name = "Residual_Huber_KRR" if method_name == "residual_huber_krr" else "Residual_ARD_KRR"
        fitted = {"base_estimator": base_est, "krr": krr, "prep": prep, "feats": feats, "base_kind": method_name, "transform": best["tt"]}
        return {"method_name": method_name, "model_name": model_name, "target_transform": best["tt"], "topk": len(feats),
                "selected_features": feats, "n_features": len(feats), "complexity_rank": _model_complexity_override(model_name),
                "search_score": best["inner"]["mean_r2"], "outer_r2": r2, "outer_rmse": rmse, "outer_mae": mae,
                "fitted_estimator": fitted}

    return None


In [8]:

# ============================================================
# Extra untried methods: shallow ensembles / SVR / KNN
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

def fit_predict_direct_estimator(X_train, y_train, X_test, estimator, y_transform="raw"):
    steps = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
    pipe = Pipeline(steps + [("model", estimator)])
    ytfm = make_y_transformer(y_transform) if "make_y_transformer" in globals() else None
    if ytfm is not None:
        est = TransformedTargetRegressor(regressor=pipe, transformer=ytfm)
    else:
        est = pipe
    est.fit(X_train, y_train)
    pred = np.asarray(est.predict(X_test)).reshape(-1)
    return est, pred

def get_method_candidates_for_output(output_name):
    fam = classify_output_family(output_name)
    methods = [
        "baseline_stability",
        "stability_lasso_ridge",
        "spca_ridge",
        "spca_huber",
        "block_pca_ridge",
        "bagged_subspace_ridge",
        "minimal_class_average",
        "direct_ard",
        "direct_elasticnet_cv",
        "direct_omp_ridge",
        "direct_quantile_median",
        "direct_theilsen",
        "direct_ransac_ridge",
        "direct_gpr_matern",
        "direct_gpr_rq",
        "residual_huber_krr",
        "residual_ard_krr",
        "residual_ridge_extratrees",
        "residual_pls_extratrees",
        # newly added
        "direct_extra_trees_shallow",
        "direct_random_forest_shallow",
        "direct_hist_gb",
        "direct_svr_rbf",
        "direct_svr_linear",
        "direct_knn_distance",
    ]
    if fam in ("thermal", "vibrational"):
        methods += ["multitask_screen_ridge", "multitask_screen_pls"]
    if fam == "mechanical_energy":
        methods += ["spca_pls"]
    return methods

_prev_eval_method_train_test = evaluate_method_train_test

def _pick_best_direct_candidate(cands):
    if not cands:
        return None
    cdf = pd.DataFrame([{
        "score_obj": c["score_obj"],
        "n_features": c["n_features"],
        "complexity_rank": c["complexity_rank"]
    } for c in cands])
    idx = choose_within_tolerance(cdf.reset_index())
    if idx is None:
        return cands[int(cdf["score_obj"].idxmax())]
    # choose_within_tolerance returns row dict containing original index if reset_index used
    chosen_index = int(idx["index"]) if "index" in idx else int(cdf["score_obj"].idxmax())
    return cands[chosen_index]

def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    fam = classify_output_family(output_name)

    # ---- NEW METHODS FIRST ----
    if method_name == "direct_extra_trees_shallow":
        base_feats = ranked_features[:min(len(ranked_features), 10 if fam!="mechanical_energy" else 8)]
        if len(base_feats) < 2:
            return None
        cands = []
        for topk in [2, 3, 4, 6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = ExtraTreesRegressor(
                    n_estimators=180,
                    max_depth=3,
                    min_samples_leaf=3,
                    min_samples_split=4,
                    random_state=random_state,
                    n_jobs=CPU_TREE_N_JOBS,
                )
                try:
                    inner = inner_group_cv_score_direct(
                        X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                        n_splits=5, n_repeats=1, random_state=random_state
                    )
                except NameError:
                    # local fallback if helper absent
                    inner = {"mean_r2": np.nan, "std_r2": np.nan}
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.15*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]):
            return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"ExtraTrees_Shallow","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_random_forest_shallow":
        base_feats = ranked_features[:min(len(ranked_features), 10 if fam!="mechanical_energy" else 8)]
        if len(base_feats) < 2:
            return None
        cands = []
        for topk in [2, 3, 4, 6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = RandomForestRegressor(
                    n_estimators=220,
                    max_depth=4,
                    min_samples_leaf=3,
                    min_samples_split=4,
                    random_state=random_state,
                    n_jobs=CPU_TREE_N_JOBS,
                )
                inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                    n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.15*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]):
            return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"RandomForest_Shallow","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_hist_gb":
        base_feats = ranked_features[:min(len(ranked_features), 10)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                est_model = HistGradientBoostingRegressor(
                    max_depth=3,
                    learning_rate=0.04,
                    max_iter=180,
                    min_samples_leaf=4,
                    l2_regularization=0.2,
                    random_state=random_state,
                )
                inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                    n_splits=5, n_repeats=1, random_state=random_state)
                score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.16*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.012*len(feats)
                cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":7})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"HistGradientBoosting","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":7,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_svr_rbf":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for C in [0.5, 1.0, 3.0]:
                    for eps in [0.02, 0.05, 0.1]:
                        est_model = SVR(kernel="rbf", C=C, epsilon=eps, gamma="scale")
                        inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                            n_splits=5, n_repeats=1, random_state=random_state)
                        score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.14*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                        cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"SVR_RBF_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_svr_linear":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for C in [0.5, 1.0, 3.0]:
                    for eps in [0.02, 0.05, 0.1]:
                        est_model = SVR(kernel="linear", C=C, epsilon=eps)
                        inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                            n_splits=5, n_repeats=1, random_state=random_state)
                        score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.12*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                        cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":5})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"SVR_Linear_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":5,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    if method_name == "direct_knn_distance":
        base_feats = ranked_features[:min(len(ranked_features), 8)]
        if len(base_feats) < 2: return None
        cands = []
        for topk in [2,3,4,6]:
            feats = base_feats[:min(topk, len(base_feats))]
            for tt in TARGET_TRANSFORM_CANDIDATES:
                for k in [3,5,7]:
                    est_model = KNeighborsRegressor(n_neighbors=k, weights="distance", p=2)
                    inner = inner_group_cv_score_direct(X_train_sel[feats], y_train_sel, g_train_sel, est_model, y_transform=tt,
                                                        n_splits=5, n_repeats=1, random_state=random_state)
                    score_obj = (inner.get("mean_r2", np.nan) if pd.notna(inner.get("mean_r2", np.nan)) else -1e9) - 0.16*(0 if pd.isna(inner.get("std_r2", np.nan)) else inner["std_r2"]) - 0.01*len(feats)
                    cands.append({"feats":feats,"tt":tt,"est_model":est_model,"inner":inner,"score_obj":score_obj,"n_features":len(feats),"complexity_rank":6})
        best = _pick_best_direct_candidate(cands)
        if best is None or not np.isfinite(best["score_obj"]): return None
        est, pred = fit_predict_direct_estimator(X_train_sel[best["feats"]], y_train_sel, X_test_group[best["feats"]], best["est_model"], y_transform=best["tt"])
        r2, rmse, mae = score_prediction(y_test_group, pred)
        if not np.isfinite(r2): return None
        return {"method_name":method_name,"model_name":"KNN_Distance_Direct","target_transform":best["tt"],"topk":len(best["feats"]),
                "selected_features":best["feats"],"n_features":len(best["feats"]),"complexity_rank":6,
                "search_score":float(best["inner"].get("mean_r2", np.nan)),"outer_r2":r2,"outer_rmse":rmse,"outer_mae":mae,"fitted_estimator":est}

    # fallback to previous implementation
    return _prev_eval_method_train_test(
        method_name=method_name,
        output_name=output_name,
        X_train_sel=X_train_sel,
        y_train_sel=y_train_sel,
        g_train_sel=g_train_sel,
        selected_train_df=selected_train_df,
        X_test_group=X_test_group,
        y_test_group=y_test_group,
        ranked_features=ranked_features,
        random_state=random_state
    )

def inner_group_cv_score_direct(X, y, groups, estimator, y_transform="raw",
                                n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE
    )
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        Xtr = X.iloc[tr_idx].copy()
        Xte = X.iloc[te_idx].copy()
        ytr = np.asarray(y)[tr_idx]
        yte = np.asarray(y)[te_idx]
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        Xtr = Xtr.loc[train_mask].reset_index(drop=True)
        Xte = Xte.loc[test_mask].reset_index(drop=True)
        ytr = ytr[train_mask]
        yte = yte[test_mask]
        try:
            est, pred = fit_predict_direct_estimator(Xtr, ytr, Xte, estimator, y_transform=y_transform)
            r2, rmse, mae = score_prediction(yte, pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df = pd.DataFrame(rows)
    return {
        "mean_r2": float(df["r2"].mean()),
        "std_r2": float(0.0 if len(df)==1 else df["r2"].std(ddof=1)),
        "mean_rmse": float(df["rmse"].mean()),
        "mean_mae": float(df["mae"].mean()),
    }


In [9]:

# ============================================================
# Cell C3. Integrated non-duplicate candidate registry
# - adds 1st-method-style weighted blending
# - adds conservative engineered-feature candidates
# - deduplicates all methods before output-wise selection
# ============================================================
from sklearn.base import BaseEstimator, RegressorMixin


def unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def safe_signed_log1p_series(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce")
    return np.sign(s) * np.log1p(np.abs(s))


def build_engineered_feature_space(X_df, base_ranked_cols, output_name, max_added=ENGINEERED_MAX_TOTAL_ADDED):
    """Conservative feature engineering from the 1st notebook.
    Generated features are used only inside wrapped estimators, so final export/prediction remains stable.
    """
    X_df = X_df.copy()
    fam = classify_output_family(output_name)
    if (not ENGINEERED_FEATURES_ENABLED) or len(base_ranked_cols) == 0:
        return X_df, []

    base_cols = [c for c in base_ranked_cols if c in X_df.columns]
    base_cols = base_cols[:ENGINEERED_MAX_BASE_BY_FAMILY.get(fam, 4)]
    added_cols = []

    if ENGINEERED_INCLUDE_SIGNED_LOG:
        for c in base_cols:
            new_c = f"{c}__slog"
            if new_c not in X_df.columns:
                X_df[new_c] = safe_signed_log1p_series(X_df[c])
                added_cols.append(new_c)
            if len(added_cols) >= max_added:
                return X_df, added_cols

    if ENGINEERED_INCLUDE_SQUARE:
        for c in base_cols:
            new_c = f"{c}__sq"
            if new_c not in X_df.columns:
                x = pd.to_numeric(X_df[c], errors="coerce")
                X_df[new_c] = x * x
                added_cols.append(new_c)
            if len(added_cols) >= max_added:
                return X_df, added_cols

    if ENGINEERED_INCLUDE_PRODUCT_BY_FAMILY.get(fam, False):
        for i in range(len(base_cols)):
            for j in range(i + 1, len(base_cols)):
                a, b = base_cols[i], base_cols[j]
                new_c = f"{a}__mul__{b}"
                if new_c not in X_df.columns:
                    xa = pd.to_numeric(X_df[a], errors="coerce")
                    xb = pd.to_numeric(X_df[b], errors="coerce")
                    X_df[new_c] = xa * xb
                    added_cols.append(new_c)
                if len(added_cols) >= max_added:
                    return X_df, added_cols

    if ENGINEERED_INCLUDE_RATIO_BY_FAMILY.get(fam, False):
        for i in range(len(base_cols)):
            for j in range(len(base_cols)):
                if i == j:
                    continue
                a, b = base_cols[i], base_cols[j]
                new_c = f"{a}__div__{b}"
                if new_c not in X_df.columns:
                    xa = pd.to_numeric(X_df[a], errors="coerce")
                    xb = pd.to_numeric(X_df[b], errors="coerce")
                    denom = xb.where(xb.abs() > 1e-12, np.nan)
                    X_df[new_c] = xa / denom
                    added_cols.append(new_c)
                if len(added_cols) >= max_added:
                    return X_df, added_cols

    return X_df, added_cols


class WeightedBlendRegressor(BaseEstimator, RegressorMixin):
    """1st-method-style weighted ensemble wrapper."""
    def __init__(self, estimators=None, weights=None, model_names=None):
        self.estimators = estimators
        self.weights = weights
        self.model_names = model_names

    def fit(self, X, y):
        estimators = [] if self.estimators is None else list(self.estimators)
        if len(estimators) == 0:
            raise ValueError("WeightedBlendRegressor requires at least one estimator.")
        raw_w = np.ones(len(estimators), dtype=float) if self.weights is None else np.asarray(self.weights, dtype=float)
        raw_w = np.where(np.isfinite(raw_w), raw_w, 1.0)
        raw_w = np.maximum(raw_w, 1e-12)
        self.weights_ = raw_w / raw_w.sum()
        self.model_names_ = self.model_names if self.model_names is not None else [f"base_{i+1}" for i in range(len(estimators))]
        self.fitted_estimators_ = []
        for est in estimators:
            e = clone(est)
            e.fit(X, y)
            self.fitted_estimators_.append(e)
        return self

    def predict(self, X):
        preds = [np.asarray(est.predict(X)).reshape(-1) for est in self.fitted_estimators_]
        return np.average(np.vstack(preds), axis=0, weights=self.weights_)


class ConservativeEngineeredRegressor(BaseEstimator, RegressorMixin):
    """Feature-engineered estimator that accepts original columns at predict time.
    This avoids final-refit/export breakage caused by engineered column names not existing in the raw selected dataframe.
    """
    def __init__(self, model_name="Ridge", y_transform="raw", output_name="", random_state=42, max_added=None):
        self.model_name = model_name
        self.y_transform = y_transform
        self.output_name = output_name
        self.random_state = random_state
        self.max_added = max_added

    def _augment(self, X):
        X = pd.DataFrame(X).copy()
        X_aug, added = build_engineered_feature_space(
            X, list(X.columns), self.output_name,
            max_added=ENGINEERED_MAX_TOTAL_ADDED if self.max_added is None else self.max_added
        )
        return X_aug, added

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = list(X.columns)
        X_aug, added = self._augment(X)
        self.added_columns_ = added
        self.train_columns_ = list(X_aug.columns)
        # keep this intentionally conservative; engineered terms can overfit small grouped datasets.
        model_alias = self.model_name
        if model_alias == "Engineered_Ridge":
            model_alias = "Ridge"
        elif model_alias == "Engineered_Bayesian_Ridge":
            model_alias = "Bayesian_Ridge"
        elif model_alias == "Engineered_Huber":
            model_alias = "Huber_Regressor"
        pipe, params, _, _ = make_pipeline_and_space(model_alias, X_aug.shape[1], random_state=self.random_state)
        transformer = make_target_transformer(self.y_transform)
        self.estimator_ = pipe if transformer is None else TransformedTargetRegressor(regressor=pipe, transformer=transformer, check_inverse=False)
        self.estimator_.fit(X_aug[self.train_columns_], y)
        return self

    def predict(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.input_columns_:
            if c not in X.columns:
                X[c] = np.nan
        X = X[self.input_columns_]
        X_aug, _ = self._augment(X)
        for c in self.train_columns_:
            if c not in X_aug.columns:
                X_aug[c] = np.nan
        return np.asarray(self.estimator_.predict(X_aug[self.train_columns_])).reshape(-1)


def inner_group_cv_score_estimator(X, y, groups, estimator, n_splits=5, n_repeats=1, random_state=42):
    rows = []
    split_iter = repeated_group_splits(
        groups,
        n_splits=min(n_splits, len(pd.unique(groups))),
        n_repeats=n_repeats,
        random_state=random_state,
        return_split_id=True,
        test_size=INNER_GSS_TEST_SIZE,
    )
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_iter:
        ytr = y[tr_idx]
        yte = y[te_idx]
        train_mask = np.isfinite(ytr)
        test_mask = np.isfinite(yte)
        if train_mask.sum() < MIN_VALID_EVAL_SAMPLES or test_mask.sum() < MIN_VALID_EVAL_SAMPLES:
            continue
        try:
            est = clone(estimator)
            est.fit(X.iloc[tr_idx].loc[train_mask].reset_index(drop=True), ytr[train_mask])
            pred = np.asarray(est.predict(X.iloc[te_idx].loc[test_mask].reset_index(drop=True))).reshape(-1)
            r2, rmse, mae = score_prediction(yte[test_mask], pred)
            if np.isfinite(r2):
                rows.append({"r2": r2, "rmse": rmse, "mae": mae})
        except Exception:
            continue
    if not rows:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan}
    df_score = pd.DataFrame(rows)
    return {
        "mean_r2": float(df_score["r2"].mean()),
        "std_r2": float(df_score["r2"].std(ddof=0)),
        "mean_rmse": float(df_score["rmse"].mean()),
        "mean_mae": float(df_score["mae"].mean()),
    }


# Preserve all methods already defined by the 3rd notebook's extra-method cell.
_base_get_method_candidates_for_output = get_method_candidates_for_output
_base_evaluate_method_train_test = evaluate_method_train_test


def get_method_candidates_for_output(output_name):
    base_methods = list(_base_get_method_candidates_for_output(output_name))
    first_method_additions = [
        "weighted_blend_top2",
        "weighted_blend_top3",
        "engineered_ridge",
        "engineered_bayesian_ridge",
        "engineered_huber",
        "featureaware_partial_missing_ensemble",
    ]
    return unique_keep_order(base_methods + first_method_additions)


def _candidate_base_models_for_blend(output_name):
    # Use stable models only for blending; highly nonlinear/direct methods are already evaluated separately.
    models = list(get_candidate_models_for_output(output_name))
    preferred = ["Bayesian_Ridge", "Ridge", "PLS_Regression", "PCR_Ridge", "Huber_Regressor", "ElasticNet_CV", "Kernel_Ridge_RBF"]
    return [m for m in preferred if m in models]


def _fit_weighted_blend_candidate(output_name, X_train_sel, y_train_sel, g_train_sel, X_test_group, y_test_group,
                                  ranked_features, blend_size=2, random_state=42):
    fam = classify_output_family(output_name)
    cands = []
    for topk in get_topk_candidates_for_output(output_name):
        feats = ranked_features[:min(topk, len(ranked_features))]
        if len(feats) < 2:
            continue
        for tt in TARGET_TRANSFORM_CANDIDATES:
            base_rows = []
            for model_name in _candidate_base_models_for_blend(output_name):
                try:
                    fit_info = fit_search_model(
                        X_train_sel[feats], y_train_sel, g_train_sel,
                        model_name=model_name,
                        target_transform=tt,
                        scoring=get_inner_scoring_for_output(output_name),
                        random_state=random_state,
                    )
                    score = float(fit_info.get("best_score", np.nan))
                    std = float(fit_info.get("best_score_std", 0.0))
                    if np.isfinite(score):
                        base_rows.append({
                            "model_name": model_name,
                            "estimator": fit_info["best_estimator"],
                            "score": score,
                            "std": std,
                            "complexity": MODEL_COMPLEXITY_RANK.get(model_name, 10),
                        })
                except Exception:
                    continue
            if len(base_rows) < blend_size:
                continue
            base_df = pd.DataFrame(base_rows).sort_values(["score", "std", "complexity"], ascending=[False, True, True]).head(blend_size)
            raw_scores = base_df["score"].to_numpy(dtype=float)
            weights = np.maximum(raw_scores - np.nanmin(raw_scores) + 1e-6, 1e-6)
            weights = weights / weights.sum()
            blend = WeightedBlendRegressor(
                estimators=list(base_df["estimator"]),
                weights=weights,
                model_names=list(base_df["model_name"]),
            )
            inner = inner_group_cv_score_estimator(
                X_train_sel[feats], y_train_sel, g_train_sel, blend,
                n_splits=5, n_repeats=1, random_state=random_state,
            )
            score_obj = (
                (inner["mean_r2"] if pd.notna(inner["mean_r2"]) else -1e9)
                - SUMMARY_STD_PENALTY * (0.0 if pd.isna(inner["std_r2"]) else inner["std_r2"])
                - TOPK_PENALTY * float(len(feats))
                - COMPLEXITY_PENALTY * float(9 + blend_size)
            )
            cands.append({
                "feats": feats,
                "tt": tt,
                "blend": blend,
                "inner": inner,
                "score_obj": score_obj,
                "complexity_rank": 9 + blend_size,
                "model_name": f"WeightedBlend_{blend_size}",
                "base_model_names": list(base_df["model_name"]),
            })
    best = _pick_best_direct_candidate(cands) if "_pick_best_direct_candidate" in globals() else None
    if best is None:
        if not cands:
            return None
        best = max(cands, key=lambda d: d["score_obj"])
    if not np.isfinite(best["score_obj"]):
        return None
    est = clone(best["blend"])
    est.fit(X_train_sel[best["feats"]], y_train_sel)
    pred = np.asarray(est.predict(X_test_group[best["feats"]])).reshape(-1)
    r2, rmse, mae = score_prediction(y_test_group, pred)
    if not np.isfinite(r2):
        return None
    return {
        "method_name": f"weighted_blend_top{blend_size}",
        "model_name": best["model_name"] + "(" + "+".join(best["base_model_names"]) + ")",
        "target_transform": best["tt"],
        "topk": len(best["feats"]),
        "selected_features": list(best["feats"]),
        "n_features": len(best["feats"]),
        "complexity_rank": int(best["complexity_rank"]),
        "search_score": float(best["inner"].get("mean_r2", np.nan)),
        "outer_r2": r2,
        "outer_rmse": rmse,
        "outer_mae": mae,
        "fitted_estimator": est,
    }


def _fit_engineered_candidate(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, X_test_group, y_test_group,
                              ranked_features, random_state=42):
    method_to_model = {
        "engineered_ridge": "Engineered_Ridge",
        "engineered_bayesian_ridge": "Engineered_Bayesian_Ridge",
        "engineered_huber": "Engineered_Huber",
    }
    model_alias = method_to_model[method_name]
    cands = []
    for topk in get_topk_candidates_for_output(output_name):
        feats = ranked_features[:min(topk, len(ranked_features))]
        if len(feats) < 2:
            continue
        for tt in TARGET_TRANSFORM_CANDIDATES:
            est = ConservativeEngineeredRegressor(
                model_name=model_alias,
                y_transform=tt,
                output_name=output_name,
                random_state=random_state,
            )
            inner = inner_group_cv_score_estimator(
                X_train_sel[feats], y_train_sel, g_train_sel, est,
                n_splits=5, n_repeats=1, random_state=random_state,
            )
            score_obj = (
                (inner["mean_r2"] if pd.notna(inner["mean_r2"]) else -1e9)
                - SUMMARY_STD_PENALTY * (0.0 if pd.isna(inner["std_r2"]) else inner["std_r2"])
                - TOPK_PENALTY * float(len(feats))
                - COMPLEXITY_PENALTY * 7
            )
            cands.append({"feats": feats, "tt": tt, "est": est, "inner": inner, "score_obj": score_obj})
    best = _pick_best_direct_candidate(cands) if "_pick_best_direct_candidate" in globals() else None
    if best is None:
        if not cands:
            return None
        best = max(cands, key=lambda d: d["score_obj"])
    if not np.isfinite(best["score_obj"]):
        return None
    est = clone(best["est"])
    est.fit(X_train_sel[best["feats"]], y_train_sel)
    pred = np.asarray(est.predict(X_test_group[best["feats"]])).reshape(-1)
    r2, rmse, mae = score_prediction(y_test_group, pred)
    if not np.isfinite(r2):
        return None
    return {
        "method_name": method_name,
        "model_name": model_alias,
        "target_transform": best["tt"],
        "topk": len(best["feats"]),
        "selected_features": list(best["feats"]),
        "n_features": len(best["feats"]),
        "complexity_rank": 7,
        "search_score": float(best["inner"].get("mean_r2", np.nan)),
        "outer_r2": r2,
        "outer_rmse": rmse,
        "outer_mae": mae,
        "fitted_estimator": est,
    }


def evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel, selected_train_df,
                               X_test_group, y_test_group, ranked_features, random_state=42):
    if method_name == "weighted_blend_top2":
        return _fit_weighted_blend_candidate(output_name, X_train_sel, y_train_sel, g_train_sel,
                                            X_test_group, y_test_group, ranked_features, blend_size=2,
                                            random_state=random_state)
    if method_name == "weighted_blend_top3":
        return _fit_weighted_blend_candidate(output_name, X_train_sel, y_train_sel, g_train_sel,
                                            X_test_group, y_test_group, ranked_features, blend_size=3,
                                            random_state=random_state)
    if method_name in ("engineered_ridge", "engineered_bayesian_ridge", "engineered_huber"):
        return _fit_engineered_candidate(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                         X_test_group, y_test_group, ranked_features, random_state=random_state)
    if method_name in ("featureaware_HT_missing_ensemble", "featureaware_partial_missing_ensemble"):
        return _fit_featureaware_missing_ensemble(output_name, X_train_sel, y_train_sel, g_train_sel,
                                                 X_test_group, y_test_group, ranked_features,
                                                 random_state=random_state)
    return _base_evaluate_method_train_test(method_name, output_name, X_train_sel, y_train_sel, g_train_sel,
                                            selected_train_df, X_test_group, y_test_group, ranked_features,
                                            random_state=random_state)




# ============================================================
# User-defined partial-feature ensemble
# ============================================================

class FeatureAwareFittedEnsemble(BaseEstimator, RegressorMixin):
    """Fitted ensemble that combines branches trained on different feature subsets.
    - common branch: features available for all/most samples
    - structural branch: user-defined partial descriptors, used only when available
    - full branch: common + partial features with missing indicators
    Row-wise weights are re-normalized if a structural branch cannot predict a row.
    """
    def __init__(self, members=None, weights=None, fallback_value=None):
        self.members = [] if members is None else list(members)
        self.weights = None if weights is None else list(weights)
        self.fallback_value = fallback_value

    def fit(self, X, y=None):
        # Already fitted by construction. This method exists for sklearn compatibility.
        return self

    def predict(self, X):
        X = pd.DataFrame(X).copy()
        n = len(X)
        if n == 0:
            return np.array([])
        pred_mat = []
        weight_mat = []
        weights = np.ones(len(self.members), dtype=float) if self.weights is None else np.asarray(self.weights, dtype=float)
        weights = np.where(np.isfinite(weights), weights, FEATUREAWARE_WEIGHT_FLOOR)
        weights = np.maximum(weights, FEATUREAWARE_WEIGHT_FLOOR)
        for w, mem in zip(weights, self.members):
            feats = list(mem.get("features", []))
            est = mem.get("estimator")
            min_present = int(mem.get("min_present", 0))
            for c in feats:
                if c not in X.columns:
                    X[c] = np.nan
            X_sub = X[feats].copy()
            valid = np.ones(n, dtype=bool)
            if min_present > 0:
                valid = X_sub.notna().sum(axis=1).to_numpy() >= min_present
            p = np.full(n, np.nan, dtype=float)
            if valid.any():
                try:
                    p[valid] = np.asarray(est.predict(X_sub.loc[valid])).reshape(-1)
                except Exception:
                    p[:] = np.nan
            pred_mat.append(p)
            weight_mat.append(np.where(np.isfinite(p), w, 0.0))
        P = np.vstack(pred_mat)
        W = np.vstack(weight_mat)
        denom = W.sum(axis=0)
        out = np.full(n, np.nan, dtype=float)
        ok = denom > 0
        out[ok] = np.nansum(P[:, ok] * W[:, ok], axis=0) / denom[ok]
        if np.any(~ok):
            fill = np.nanmean(out[ok]) if np.any(ok) else (0.0 if self.fallback_value is None else self.fallback_value)
            out[~ok] = fill
        return out


def _make_default_estimator(model_name="Ridge", target_transform="raw", random_state=42, add_indicator=False):
    if model_name == "Bayesian_Ridge":
        reg = BayesianRidge()
    elif model_name == "Ridge":
        reg = Ridge(alpha=1.0, random_state=random_state)
    elif model_name == "Huber_Regressor":
        reg = HuberRegressor(max_iter=3000, alpha=0.0001, epsilon=1.35)
    elif model_name == "PLS_Regression":
        reg = PLSRegression(n_components=2)
    else:
        # fallback to a stable linear model
        reg = BayesianRidge()
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=bool(add_indicator))),
        ("scaler", StandardScaler()),
        ("model", reg),
    ])
    transformer = make_target_transformer(target_transform)
    return pipe if transformer is None else TransformedTargetRegressor(regressor=pipe, transformer=transformer, check_inverse=False)


def _rank_structural_features(X_df, y, groups, structural_cols, min_row_coverage=0.15):
    X = pd.DataFrame(X_df).copy()
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups).astype(str)
    structural_cols = [c for c in structural_cols if c in X.columns]
    if not structural_cols:
        return [], pd.DataFrame(columns=["feature_name", "score", "non_missing_ratio"])
    grp_df = X[structural_cols].copy()
    grp_df["target"] = y
    grp_df["GROUP_ID"] = groups
    grp_med = grp_df.groupby("GROUP_ID").median(numeric_only=True).reset_index(drop=True)
    y_group = grp_med.pop("target").to_numpy(dtype=float)
    rows = []
    for c in structural_cols:
        x = pd.to_numeric(grp_med[c], errors="coerce")
        cov = float(x.notna().mean()) if len(x) else 0.0
        if x.notna().sum() < max(3, int(math.ceil(min_row_coverage * len(x)))):
            continue
        fill = x.fillna(x.median())
        sp = abs(spearman_corr_safe(fill.to_numpy(dtype=float), y_group))
        pr = abs(pearson_corr_safe(fill.to_numpy(dtype=float), y_group))
        # Small coverage penalty prevents extremely sparse descriptors from dominating.
        score = (0.65 * sp + 0.35 * pr) * math.sqrt(max(cov, 1e-6))
        rows.append({"feature_name": c, "score": float(score), "spearman_abs": float(sp), "pearson_abs": float(pr), "non_missing_ratio": cov})
    if not rows:
        return [], pd.DataFrame(columns=["feature_name", "score", "non_missing_ratio"])
    sdf = pd.DataFrame(rows).sort_values(["score", "non_missing_ratio"], ascending=[False, False]).reset_index(drop=True)
    return sdf["feature_name"].tolist(), sdf


def _safe_branch_inner_score(X, y, groups, estimator, branch_requires_struct=False, min_present=0, random_state=42):
    X = pd.DataFrame(X).reset_index(drop=True)
    y = np.asarray(y, dtype=float)
    groups = np.asarray(groups).astype(str)
    if branch_requires_struct:
        mask = X.notna().sum(axis=1).to_numpy() >= int(min_present)
    else:
        mask = np.ones(len(X), dtype=bool)
    mask &= np.isfinite(y)
    if mask.sum() < MIN_VALID_EVAL_SAMPLES:
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan, "n_used": int(mask.sum())}
    if len(pd.unique(groups[mask])) < max(3, min(STRUCTURAL_MIN_GROUPS_FOR_BRANCH, 4)):
        return {"mean_r2": np.nan, "std_r2": np.nan, "mean_rmse": np.nan, "mean_mae": np.nan, "n_used": int(mask.sum())}
    out = inner_group_cv_score_estimator(
        X.loc[mask].reset_index(drop=True), y[mask], groups[mask], estimator,
        n_splits=FEATUREAWARE_INNER_SPLITS, n_repeats=FEATUREAWARE_INNER_REPEATS,
        random_state=random_state,
    )
    out["n_used"] = int(mask.sum())
    return out


def _fit_featureaware_missing_ensemble(output_name, X_train_sel, y_train_sel, g_train_sel, X_test_group, y_test_group,
                                       ranked_features, random_state=42):
    structural_cols = [c for c in globals().get("STRUCTURAL_FACTOR_COLS", []) if c in X_train_sel.columns]
    if not structural_cols:
        return None

    all_cols = list(X_train_sel.columns)
    common_cols_all = [c for c in all_cols if c not in structural_cols]
    common_ranked = [c for c in ranked_features if c in common_cols_all]
    if len(common_ranked) < 2:
        common_ranked = supervised_screen_features(X_train_sel[common_cols_all], y_train_sel, max_keep=max(COMMON_TOPK_CANDIDATES_FOR_ENSEMBLE))
    struct_ranked, struct_rank_df = _rank_structural_features(
        X_train_sel, y_train_sel, g_train_sel, structural_cols,
        min_row_coverage=STRUCTURAL_MIN_ROW_COVERAGE,
    )
    if len(common_ranked) < 1 or len(struct_ranked) < 1:
        return None

    candidates = []
    for ck in COMMON_TOPK_CANDIDATES_FOR_ENSEMBLE:
        common_feats = common_ranked[:min(ck, len(common_ranked))]
        if len(common_feats) < 1:
            continue
        for sk in STRUCTURAL_TOPK_CANDIDATES:
            struct_feats = struct_ranked[:min(sk, len(struct_ranked))]
            if len(struct_feats) < 1:
                continue
            union_feats = unique_keep_order(common_feats + struct_feats)
            for tt in FEATUREAWARE_TARGET_TRANSFORMS:
                for base_model in FEATUREAWARE_BASE_MODELS:
                    branch_defs = []
                    # 1) common-only branch: never loses rows
                    common_est = _make_default_estimator(base_model, tt, random_state=random_state, add_indicator=False)
                    common_score = _safe_branch_inner_score(
                        X_train_sel[common_feats], y_train_sel, g_train_sel, common_est,
                        branch_requires_struct=False, random_state=random_state,
                    )
                    if pd.notna(common_score.get("mean_r2", np.nan)):
                        branch_defs.append({
                            "name": f"common_{base_model}", "features": common_feats, "estimator": common_est,
                            "score": common_score["mean_r2"], "std": common_score.get("std_r2", 0.0), "min_present": 0,
                            "add_indicator": False,
                        })
                    # 2) full branch: common + structural with missingness indicator
                    full_est = _make_default_estimator(base_model, tt, random_state=random_state, add_indicator=FEATUREAWARE_USE_MISSING_INDICATOR)
                    full_score = _safe_branch_inner_score(
                        X_train_sel[union_feats], y_train_sel, g_train_sel, full_est,
                        branch_requires_struct=False, random_state=random_state,
                    )
                    if pd.notna(full_score.get("mean_r2", np.nan)):
                        branch_defs.append({
                            "name": f"full_missingIndicator_{base_model}", "features": union_feats, "estimator": full_est,
                            "score": full_score["mean_r2"], "std": full_score.get("std_r2", 0.0), "min_present": 0,
                            "add_indicator": FEATUREAWARE_USE_MISSING_INDICATOR,
                        })
                    # 3) structural-only branch: fitted only on rows that contain partial descriptors
                    struct_est = _make_default_estimator(base_model, tt, random_state=random_state, add_indicator=False)
                    struct_score = _safe_branch_inner_score(
                        X_train_sel[struct_feats], y_train_sel, g_train_sel, struct_est,
                        branch_requires_struct=True, min_present=STRUCTURAL_MIN_PRESENT_PER_ROW,
                        random_state=random_state,
                    )
                    if pd.notna(struct_score.get("mean_r2", np.nan)) and struct_score.get("n_used", 0) >= MIN_VALID_EVAL_SAMPLES:
                        branch_defs.append({
                            "name": f"structOnly_{base_model}", "features": struct_feats, "estimator": struct_est,
                            "score": struct_score["mean_r2"], "std": struct_score.get("std_r2", 0.0),
                            "min_present": STRUCTURAL_MIN_PRESENT_PER_ROW, "add_indicator": False,
                        })
                    if len(branch_defs) < 2:
                        continue
                    scores = np.array([b["score"] for b in branch_defs], dtype=float)
                    stds = np.array([0.0 if pd.isna(b.get("std", 0.0)) else b.get("std", 0.0) for b in branch_defs], dtype=float)
                    adjusted = scores - SUMMARY_STD_PENALTY * stds
                    adjusted = np.where(np.isfinite(adjusted), adjusted, np.nan)
                    if np.all(pd.isna(adjusted)):
                        weights = np.ones(len(branch_defs), dtype=float) / len(branch_defs)
                        mean_inner = np.nan
                    else:
                        min_adj = np.nanmin(adjusted)
                        weights = np.maximum(adjusted - min_adj + FEATUREAWARE_WEIGHT_FLOOR, FEATUREAWARE_WEIGHT_FLOOR)
                        weights = weights / weights.sum()
                        mean_inner = float(np.nanmean(adjusted))
                    # Fit final branches. Structural-only branch uses only rows with available structural descriptors.
                    fitted_members = []
                    fitted_weights = []
                    for b, w in zip(branch_defs, weights):
                        feats = b["features"]
                        est = clone(b["estimator"])
                        if b["min_present"] > 0:
                            mask = X_train_sel[feats].notna().sum(axis=1).to_numpy() >= int(b["min_present"])
                        else:
                            mask = np.ones(len(X_train_sel), dtype=bool)
                        mask &= np.isfinite(y_train_sel)
                        if mask.sum() < MIN_VALID_EVAL_SAMPLES:
                            continue
                        try:
                            est.fit(X_train_sel.loc[mask, feats].reset_index(drop=True), y_train_sel[mask])
                            fitted_members.append({"name": b["name"], "features": feats, "estimator": est, "min_present": b["min_present"]})
                            fitted_weights.append(float(w))
                        except Exception:
                            continue
                    if len(fitted_members) < 2:
                        continue
                    ens = FeatureAwareFittedEnsemble(members=fitted_members, weights=fitted_weights, fallback_value=float(np.nanmedian(y_train_sel)))
                    pred = np.asarray(ens.predict(X_test_group[union_feats])).reshape(-1)
                    r2, rmse, mae = score_prediction(y_test_group, pred)
                    if not np.isfinite(r2):
                        continue
                    candidates.append({
                        "method_name": "featureaware_partial_missing_ensemble",
                        "model_name": "FeatureAwarePartial(" + "+".join([m["name"] for m in fitted_members]) + ")",
                        "target_transform": tt,
                        "topk": len(union_feats),
                        "selected_features": union_feats,
                        "n_features": len(union_feats),
                        "complexity_rank": 11,
                        "search_score": mean_inner,
                        "outer_r2": r2,
                        "outer_rmse": rmse,
                        "outer_mae": mae,
                        "fitted_estimator": ens,
                        "branch_weights": fitted_weights,
                    })
    if not candidates:
        return None
    return sorted(candidates, key=lambda d: (d["outer_r2"], -d["outer_rmse"], -d["n_features"]), reverse=True)[0]


# Method registry preview
METHOD_REGISTRY_PREVIEW = []
for _out in DATA_BY_OUTPUT.keys():
    METHOD_REGISTRY_PREVIEW.append({
        "output_name": _out,
        "output_family": classify_output_family(_out),
        "n_candidate_methods": len(get_method_candidates_for_output(_out)),
        "n_HT_structural_features": len(globals().get("STRUCTURAL_FACTOR_COLS", [])),
        "HT_structural_features": ", ".join(globals().get("STRUCTURAL_FACTOR_COLS", [])),
        "candidate_methods": ", ".join(get_method_candidates_for_output(_out)),
    })
METHOD_REGISTRY_PREVIEW_DF = pd.DataFrame(METHOD_REGISTRY_PREVIEW)
export_df(METHOD_REGISTRY_PREVIEW_DF, os.path.join(MODEL_EXPORT_DIR, "method_registry_preview"))
display(METHOD_REGISTRY_PREVIEW_DF)


,output_name,output_family,n_candidate_methods,n_HT_structural_features,HT_structural_features,candidate_methods
0,Modulus,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
1,Com. Strength,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
2,APS,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
3,AS,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
4,Yield strength,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
5,Densif. strength,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
6,Total energy,mechanical_energy,32,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
7,Thermal characteristics | Thermal conductivity...,thermal,33,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
8,Thermal characteristics | h | W/m.K,thermal,33,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."
9,Thermal characteristics | Heating rate | °C/s,thermal,33,17,"node | 5x5x5, strut | 5x5x5, Criteria | 5x5x5,...","baseline_stability, stability_lasso_ridge, spc..."


In [10]:
# ============================================================
# Cell D1. Sequential stage-based output-wise comparison
# - NOT a simple merge of all methods
# - Each uploaded-method family is evaluated independently first
# - Final Output-wise winner is selected from stage-level candidates
# ============================================================

METHOD_SUMMARY_ROWS = []
PER_OUTPUT_RESULTS = {}
STAGE_SUMMARY_ROWS_ALL = []


def _ensure_standard_cols(df_in, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df


def get_y_strategy_candidates_for_output(output_name, groups=None):
    auto_strategy = get_best_y_strategy_for_output(output_name, groups=groups)
    if not COMPARE_ALL_Y_STRATEGIES:
        return [auto_strategy]
    rep = group_repeat_stats(groups) if groups is not None else {"has_repeats": True}
    if not rep.get("has_repeats", True):
        return ["singleton_raw"]
    out = []
    for s in Y_STRATEGY_CANDIDATES:
        if s == "auto":
            s = auto_strategy
        if s not in out:
            out.append(s)
    return out


def get_feature_scope_candidates_for_output(output_name, feature_cols):
    """Return valid feature scopes for this output.

    The main fix is to compare the old-compatible variables and the newly added variables
    as separate candidates. If added descriptors reduce generalization for a given Output,
    legacy_core_only can win directly.
    """
    scopes = []
    scope_map = globals().get("FEATURE_SCOPE_COLS", {})
    for scope in COMPARE_FEATURE_SCOPES:
        if scope == "all_features":
            cols = list(feature_cols)
        else:
            cols = [c for c in scope_map.get(scope, []) if c in feature_cols]
        if len(cols) >= 2:
            scopes.append(scope)
    if not scopes:
        scopes = ["all_features"]
    return unique_keep_order(scopes)


def get_feature_cols_for_scope(feature_cols, feature_scope):
    scope_map = globals().get("FEATURE_SCOPE_COLS", {})
    if feature_scope == "all_features":
        return list(feature_cols)
    if feature_scope in scope_map:
        cols = [c for c in scope_map.get(feature_scope, []) if c in feature_cols]
        return cols if len(cols) >= 2 else list(feature_cols)
    if feature_scope == "common_only":
        cols = [c for c in globals().get("LEGACY_FEATURE_COLS", globals().get("COMMON_FEATURE_COLS", [])) if c in feature_cols]
        return cols if len(cols) >= 2 else list(feature_cols)
    return list(feature_cols)


def get_methods_for_scope(output_name, feature_scope):
    methods = list(get_method_candidates_for_output(output_name))
    partial_ensemble_names = [
        "featureaware_HT_missing_ensemble",          # old/backward-compatible name
        "featureaware_partial_missing_ensemble",     # current user-defined partial range in Cell A1
        "HT_missing_ensemble",
        "PartialFeature_Missing_Ensemble",
    ]
    if feature_scope != "all_features":
        # Partial-feature-aware ensemble needs both the legacy/common branch and the added partial branch,
        # so it is meaningful only for all_features.
        methods = [m for m in methods if m not in partial_ensemble_names]
    return unique_keep_order(methods)


# ------------------------------------------------------------
# Sequential method-family registry
# ------------------------------------------------------------
def get_sequential_method_stages(output_name):
    """Return method stages that correspond to the uploaded notebooks.

    The important change from the previous v2 notebook is that these families are
    not thrown into one mixed pool first. Each stage is evaluated independently,
    summarized independently, and then compared at the stage-candidate level.
    """
    fam = classify_output_family(output_name)
    stages = [
        {
            "stage_id": "A",
            "stage_name": "1st_legacy_engineered_blend",
            "stage_source": "Training_260418 + engineered additions",
            "methods": [
                "weighted_blend_top2",
                "weighted_blend_top3",
                "engineered_ridge",
                "engineered_bayesian_ridge",
                "engineered_huber",
            ],
        },
        {
            "stage_id": "B",
            "stage_name": "2nd_stability_spca_blockpca_bagging_multitask",
            "stage_source": "Training_260419 / stability-aware family",
            "methods": [
                "baseline_stability",
                "stability_lasso_ridge",
                "spca_ridge",
                "spca_huber",
                "spca_pls",
                "block_pca_ridge",
                "bagged_subspace_ridge",
                "minimal_class_average",
                "multitask_screen_ridge",
                "multitask_screen_pls",
            ],
        },
        {
            "stage_id": "C",
            "stage_name": "3rd_direct_residual_shallow_nonlinear",
            "stage_source": "Training_260420 / extra untried direct-residual methods",
            "methods": [
                "direct_ard",
                "direct_elasticnet_cv",
                "direct_omp_ridge",
                "direct_quantile_median",
                "direct_theilsen",
                "direct_ransac_ridge",
                "direct_gpr_matern",
                "direct_gpr_rq",
                "residual_huber_krr",
                "residual_ard_krr",
                "residual_ridge_extratrees",
                "residual_pls_extratrees",
                "direct_extra_trees_shallow",
                "direct_random_forest_shallow",
                "direct_hist_gb",
                "direct_svr_rbf",
                "direct_svr_linear",
                "direct_knn_distance",
            ],
        },
        {
            "stage_id": "D",
            "stage_name": "4th_partial_featureaware_missing_descriptor_ensemble",
            "stage_source": "Training_260503/260504 partial-feature ensemble",
            "methods": [
                "featureaware_HT_missing_ensemble",
                "featureaware_partial_missing_ensemble",
                "HT_missing_ensemble",
                "PartialFeature_Missing_Ensemble",
            ],
        },
    ]

    available = set(get_method_candidates_for_output(output_name))
    cleaned = []
    used = set()
    for st in stages:
        methods = [m for m in st["methods"] if m in available]
        # Family-specific methods are automatically removed if unavailable.
        if fam != "mechanical_energy":
            methods = [m for m in methods if m != "spca_pls"]
        if fam not in ("thermal", "vibrational"):
            methods = [m for m in methods if not m.startswith("multitask_screen_")]
        if methods:
            st2 = dict(st)
            st2["methods"] = unique_keep_order(methods)
            cleaned.append(st2)
            used.update(methods)

    # Safety net: if future cells define a method not listed above, evaluate it as a separate stage
    # rather than silently dropping it.
    unassigned = [m for m in available if m not in used]
    if unassigned:
        cleaned.append({
            "stage_id": "Z",
            "stage_name": "unassigned_available_methods",
            "stage_source": "Safety-net stage for methods not mapped above",
            "methods": unique_keep_order(unassigned),
        })
    return cleaned


def build_outputwise_combo_score_df(ok_fold_df):
    """Exact-combo summary for diagnostics.

    Combo = stage + feature_scope + Y-strategy + method + internal model + target transform + top-k.
    """
    max_cv_runs = max(1, ok_fold_df["cv_run_id"].nunique())
    group_cols = ["stage_id", "stage_name", "feature_scope", "best_y_strategy", "method_name", "model_name", "target_transform", "topk"]
    combo_df = ok_fold_df.groupby(group_cols, as_index=False).agg(
        mean_outer_r2=("outer_r2", "mean"),
        median_outer_r2=("outer_r2", "median"),
        std_outer_r2=("outer_r2", "std"),
        min_outer_r2=("outer_r2", "min"),
        q25_outer_r2=("outer_r2", lambda s: float(np.nanquantile(s, 0.25))),
        mean_outer_rmse=("outer_rmse", "mean"),
        mean_outer_mae=("outer_mae", "mean"),
        n_cv_runs=("cv_run_id", "nunique"),
        n_success_rows=("outer_r2", "count"),
        mean_n_features=("n_features", "mean"),
    ).reset_index(drop=True)
    combo_df["support_share"] = combo_df["n_cv_runs"] / max_cv_runs
    combo_df["low_support_penalty"] = np.where(
        combo_df["support_share"] < MIN_SELECTION_SUPPORT_SHARE,
        LOW_SUPPORT_EXTRA_PENALTY * (MIN_SELECTION_SUPPORT_SHARE - combo_df["support_share"]),
        0.0,
    )
    combo_df["score_obj"] = (
        OUTPUTWISE_R2_WEIGHT * combo_df["mean_outer_r2"]
        - COMBO_STD_PENALTY * combo_df["std_outer_r2"].fillna(0.0)
        + COMBO_SUPPORT_BONUS * combo_df["support_share"]
        + 0.03 * combo_df["q25_outer_r2"].fillna(combo_df["mean_outer_r2"])
        - OUTPUTWISE_TOPK_PENALTY * combo_df["topk"].astype(float)
        + combo_df["feature_scope"].map(OUTPUTWISE_SCOPE_BONUS).fillna(0.0)
        - combo_df["low_support_penalty"]
    )
    return combo_df.sort_values(
        ["score_obj", "support_share", "mean_outer_r2", "median_outer_r2", "q25_outer_r2", "std_outer_r2", "topk"],
        ascending=[False, False, False, False, False, True, True],
    ).reset_index(drop=True)


def build_outputwise_score_df(ok_fold_df):
    """Robust method-level selector with stage identity preserved.

    Stage-level candidate = stage × feature_scope × best_y_strategy × method_name.
    Internal model/transform/top-k are chosen as a representative combo only after
    the winning method-level candidate is identified.
    """
    max_cv_runs = max(1, ok_fold_df["cv_run_id"].nunique())
    method_cols = ["stage_id", "stage_name", "stage_source", "feature_scope", "best_y_strategy", "method_name"]

    method_df = ok_fold_df.groupby(method_cols, as_index=False).agg(
        mean_outer_r2=("outer_r2", "mean"),
        median_outer_r2=("outer_r2", "median"),
        std_outer_r2=("outer_r2", "std"),
        min_outer_r2=("outer_r2", "min"),
        q25_outer_r2=("outer_r2", lambda s: float(np.nanquantile(s, 0.25))),
        mean_outer_rmse=("outer_rmse", "mean"),
        mean_outer_mae=("outer_mae", "mean"),
        n_cv_runs=("cv_run_id", "nunique"),
        n_success_rows=("outer_r2", "count"),
        mean_n_features=("n_features", "mean"),
    ).reset_index(drop=True)
    method_df["support_share"] = method_df["n_cv_runs"] / max_cv_runs
    method_df["low_support_penalty"] = np.where(
        method_df["support_share"] < MIN_SELECTION_SUPPORT_SHARE,
        LOW_SUPPORT_EXTRA_PENALTY * (MIN_SELECTION_SUPPORT_SHARE - method_df["support_share"]),
        0.0,
    )
    method_df["score_obj"] = (
        OUTPUTWISE_R2_WEIGHT * method_df["mean_outer_r2"]
        - OUTPUTWISE_STD_PENALTY * method_df["std_outer_r2"].fillna(0.0)
        + OUTPUTWISE_SUPPORT_BONUS * method_df["support_share"]
        + 0.03 * method_df["q25_outer_r2"].fillna(method_df["mean_outer_r2"])
        + method_df["feature_scope"].map(OUTPUTWISE_SCOPE_BONUS).fillna(0.0)
        - method_df["low_support_penalty"]
    )

    combo_df = build_outputwise_combo_score_df(ok_fold_df)
    rep_rows = []
    for _, mrow in method_df.iterrows():
        sub = combo_df[
            (combo_df["stage_id"] == mrow["stage_id"])
            & (combo_df["feature_scope"] == mrow["feature_scope"])
            & (combo_df["best_y_strategy"] == mrow["best_y_strategy"])
            & (combo_df["method_name"] == mrow["method_name"])
        ].copy()
        if sub.empty:
            continue
        sub_supported = sub[sub["support_share"] >= max(1.0 / max_cv_runs, 0.15)]
        chosen_combo = (sub_supported if not sub_supported.empty else sub).iloc[0]
        row = mrow.to_dict()
        row.update({
            "model_name": chosen_combo["model_name"],
            "target_transform": chosen_combo["target_transform"],
            "topk": int(chosen_combo["topk"]),
            "combo_mean_outer_r2": float(chosen_combo["mean_outer_r2"]),
            "combo_median_outer_r2": float(chosen_combo["median_outer_r2"]),
            "combo_std_outer_r2": float(0.0 if pd.isna(chosen_combo["std_outer_r2"]) else chosen_combo["std_outer_r2"]),
            "combo_n_cv_runs": int(chosen_combo["n_cv_runs"]),
            "combo_support_share": float(chosen_combo["support_share"]),
            "combo_score_obj": float(chosen_combo["score_obj"]),
            "selection_aggregation": "sequential_stage_method_level_then_representative_combo",
        })
        rep_rows.append(row)

    out = pd.DataFrame(rep_rows)
    if out.empty:
        return out

    stable = out[out["support_share"] >= MIN_SELECTION_SUPPORT_SHARE].copy()
    if not stable.empty:
        out = stable

    return out.sort_values(
        ["score_obj", "support_share", "mean_outer_r2", "median_outer_r2", "q25_outer_r2", "std_outer_r2", "combo_score_obj"],
        ascending=[False, False, False, False, False, True, False],
    ).reset_index(drop=True)


def select_final_from_stage_candidates(stage_candidate_df):
    """Final Output-wise winner from already-computed stage candidates."""
    if stage_candidate_df is None or stage_candidate_df.empty:
        return pd.DataFrame()
    cand = stage_candidate_df.copy()
    stable = cand[cand["support_share"] >= MIN_SELECTION_SUPPORT_SHARE].copy()
    if not stable.empty:
        cand = stable
    return cand.sort_values(
        ["score_obj", "support_share", "mean_outer_r2", "median_outer_r2", "q25_outer_r2", "std_outer_r2", "combo_score_obj"],
        ascending=[False, False, False, False, False, True, False],
    ).reset_index(drop=True)




# ------------------------------------------------------------
# Missing summary/select helper functions patched for v4_FIXED
# These functions make the sequential Stage A/B/C/D selector self-contained.
# ------------------------------------------------------------
def _safe_nanquantile(series, q, default=np.nan):
    arr = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return default
    return float(np.nanquantile(arr, q))


def _mode_string(series, default=""):
    try:
        s = pd.Series(series).dropna().astype(str)
        if s.empty:
            return default
        return str(s.mode().iloc[0])
    except Exception:
        return default


def _get_selection_constants():
    return {
        "min_support": float(globals().get("MIN_SELECTION_SUPPORT_SHARE", 0.50)),
        "low_support_penalty": float(globals().get("LOW_SUPPORT_EXTRA_PENALTY", 0.25)),
        "r2_weight": float(globals().get("OUTPUTWISE_R2_WEIGHT", 1.00)),
        "std_penalty": float(globals().get("OUTPUTWISE_STD_PENALTY", 0.10)),
        "support_bonus": float(globals().get("OUTPUTWISE_SUPPORT_BONUS", 0.03)),
        "topk_penalty": float(globals().get("OUTPUTWISE_TOPK_PENALTY", 0.001)),
        "scope_bonus": globals().get("OUTPUTWISE_SCOPE_BONUS", {}),
    }


def summarize_exact_combo(ok_fold_df, total_cv_runs=None):
    """Summarize exact candidates: stage × scope × y-strategy × method × model × transform × topk.

    This is used only after a method-level winner is identified, so that a stable
    representative internal model/transform/top-k can be selected.
    """
    if ok_fold_df is None or len(ok_fold_df) == 0:
        return pd.DataFrame()
    df = ok_fold_df.copy()
    df = df[(df.get("status", "ok") == "ok") & np.isfinite(pd.to_numeric(df.get("outer_r2", np.nan), errors="coerce"))].copy()
    if df.empty:
        return pd.DataFrame()
    if total_cv_runs is None or total_cv_runs <= 0:
        total_cv_runs = max(1, df["cv_run_id"].nunique() if "cv_run_id" in df.columns else len(df))

    for c in ["stage_id", "stage_name", "stage_source", "feature_scope", "best_y_strategy", "method_name", "model_name", "target_transform"]:
        if c not in df.columns:
            df[c] = ""
    if "topk" not in df.columns:
        df["topk"] = np.nan
    if "n_features" not in df.columns:
        df["n_features"] = np.nan
    if "selected_features" not in df.columns:
        df["selected_features"] = ""

    group_cols = ["stage_id", "stage_name", "stage_source", "feature_scope", "best_y_strategy", "method_name", "model_name", "target_transform", "topk"]
    rows = []
    for keys, sub in df.groupby(group_cols, dropna=False, sort=False):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        r2 = pd.to_numeric(sub["outer_r2"], errors="coerce")
        rmse = pd.to_numeric(sub.get("outer_rmse", np.nan), errors="coerce")
        mae = pd.to_numeric(sub.get("outer_mae", np.nan), errors="coerce")
        n_cv = int(sub["cv_run_id"].nunique()) if "cv_run_id" in sub.columns else int(len(sub))
        support = float(n_cv / max(1, total_cv_runs))
        r2_std = float(r2.std(ddof=1)) if r2.count() > 1 else 0.0
        row.update({
            "r2_mean": float(r2.mean()),
            "r2_median": float(r2.median()),
            "r2_min": float(r2.min()),
            "r2_q25": _safe_nanquantile(r2, 0.25),
            "r2_std": r2_std,
            "rmse_mean": float(rmse.mean()) if np.isfinite(rmse).any() else np.nan,
            "mae_mean": float(mae.mean()) if np.isfinite(mae).any() else np.nan,
            "n_cv_runs": n_cv,
            "n_success_rows": int(r2.count()),
            "support_share": support,
            "n_features_median": float(pd.to_numeric(sub["n_features"], errors="coerce").median()) if "n_features" in sub.columns else np.nan,
            "selected_features_mode": _mode_string(sub.get("selected_features", "")),
        })
        const = _get_selection_constants()
        low_support_penalty = max(0.0, const["min_support"] - support) * const["low_support_penalty"]
        scope_bonus = const["scope_bonus"].get(row.get("feature_scope", ""), 0.0) if isinstance(const["scope_bonus"], dict) else 0.0
        topk_value = pd.to_numeric(pd.Series([row.get("topk", np.nan)]), errors="coerce").iloc[0]
        topk_value = 0.0 if not np.isfinite(topk_value) else float(topk_value)
        row["selection_score"] = (
            const["r2_weight"] * row["r2_mean"]
            - const["std_penalty"] * row["r2_std"]
            + const["support_bonus"] * support
            + 0.03 * row["r2_q25"]
            + scope_bonus
            - const["topk_penalty"] * topk_value
            - low_support_penalty
        )
        row["low_support_penalty"] = low_support_penalty
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(
        ["selection_score", "support_share", "r2_mean", "r2_median", "r2_q25", "r2_std"],
        ascending=[False, False, False, False, False, True],
    ).reset_index(drop=True)


def select_representative_combo(combo_summary):
    """Return the best exact combo as a plain dict for final refit/export."""
    if combo_summary is None or len(combo_summary) == 0:
        return None
    cand = combo_summary.copy()
    if "support_share" in cand.columns:
        min_support = float(globals().get("MIN_SELECTION_SUPPORT_SHARE", 0.50))
        stable = cand[cand["support_share"] >= min_support].copy()
        if not stable.empty:
            cand = stable
    sort_cols = [c for c in ["selection_score", "support_share", "r2_mean", "r2_median", "r2_q25", "r2_std"] if c in cand.columns]
    asc = [False, False, False, False, False, True][:len(sort_cols)]
    if sort_cols:
        cand = cand.sort_values(sort_cols, ascending=asc)
    row = cand.iloc[0].to_dict()
    return {
        "model_name": row.get("model_name", ""),
        "target_transform": row.get("target_transform", "raw"),
        "topk": row.get("topk", np.nan),
        "n_features_median": row.get("n_features_median", np.nan),
        "selected_features_mode": row.get("selected_features_mode", ""),
        "combo_cv_r2_mean": row.get("r2_mean", np.nan),
        "combo_cv_r2_std": row.get("r2_std", np.nan),
        "combo_n_cv_runs": row.get("n_cv_runs", 0),
        "combo_support_share": row.get("support_share", 0.0),
        "combo_selection_score": row.get("selection_score", np.nan),
    }


def summarize_method_level(ok_fold_df, total_cv_runs=None):
    """Robust method-level summary used by the sequential Stage selector.

    Candidate = stage × feature_scope × best_y_strategy × method_name.
    The exact model/transform/top-k is selected separately as a representative combo.
    """
    if ok_fold_df is None or len(ok_fold_df) == 0:
        return pd.DataFrame()
    df = ok_fold_df.copy()
    df = df[(df.get("status", "ok") == "ok") & np.isfinite(pd.to_numeric(df.get("outer_r2", np.nan), errors="coerce"))].copy()
    if df.empty:
        return pd.DataFrame()
    if total_cv_runs is None or total_cv_runs <= 0:
        total_cv_runs = max(1, df["cv_run_id"].nunique() if "cv_run_id" in df.columns else len(df))

    for c in ["stage_id", "stage_name", "stage_source", "feature_scope", "best_y_strategy", "method_name"]:
        if c not in df.columns:
            df[c] = ""

    group_cols = ["stage_id", "stage_name", "stage_source", "feature_scope", "best_y_strategy", "method_name"]
    combo_summary_all = summarize_exact_combo(df, total_cv_runs=total_cv_runs)
    rows = []
    for keys, sub in df.groupby(group_cols, dropna=False, sort=False):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        r2 = pd.to_numeric(sub["outer_r2"], errors="coerce")
        rmse = pd.to_numeric(sub.get("outer_rmse", np.nan), errors="coerce")
        mae = pd.to_numeric(sub.get("outer_mae", np.nan), errors="coerce")
        n_cv = int(sub["cv_run_id"].nunique()) if "cv_run_id" in sub.columns else int(len(sub))
        support = float(n_cv / max(1, total_cv_runs))
        r2_std = float(r2.std(ddof=1)) if r2.count() > 1 else 0.0
        row.update({
            "r2_mean": float(r2.mean()),
            "r2_median": float(r2.median()),
            "r2_min": float(r2.min()),
            "r2_q25": _safe_nanquantile(r2, 0.25),
            "r2_std": r2_std,
            "rmse_mean": float(rmse.mean()) if np.isfinite(rmse).any() else np.nan,
            "mae_mean": float(mae.mean()) if np.isfinite(mae).any() else np.nan,
            "n_cv_runs": n_cv,
            "n_success_rows": int(r2.count()),
            "support_share": support,
            "mean_n_features": float(pd.to_numeric(sub.get("n_features", np.nan), errors="coerce").mean()) if "n_features" in sub.columns else np.nan,
        })
        const = _get_selection_constants()
        low_support_penalty = max(0.0, const["min_support"] - support) * const["low_support_penalty"]
        scope_bonus = const["scope_bonus"].get(row.get("feature_scope", ""), 0.0) if isinstance(const["scope_bonus"], dict) else 0.0
        row["selection_score"] = (
            const["r2_weight"] * row["r2_mean"]
            - const["std_penalty"] * row["r2_std"]
            + const["support_bonus"] * support
            + 0.03 * row["r2_q25"]
            + scope_bonus
            - low_support_penalty
        )
        row["low_support_penalty"] = low_support_penalty

        # Representative exact combo for this method-level candidate.
        rep = None
        if combo_summary_all is not None and not combo_summary_all.empty:
            mask = np.ones(len(combo_summary_all), dtype=bool)
            for c in group_cols:
                mask &= combo_summary_all[c].astype(str).to_numpy() == str(row.get(c, ""))
            rep = select_representative_combo(combo_summary_all.loc[mask].copy())
        if rep is None:
            rep = {}
        row.update({
            "representative_model_name": rep.get("model_name", ""),
            "representative_target_transform": rep.get("target_transform", "raw"),
            "representative_topk": rep.get("topk", np.nan),
            "representative_n_features_median": rep.get("n_features_median", np.nan),
            "representative_selected_features_mode": rep.get("selected_features_mode", ""),
            "combo_cv_r2_mean": rep.get("combo_cv_r2_mean", np.nan),
            "combo_cv_r2_std": rep.get("combo_cv_r2_std", np.nan),
            "combo_n_cv_runs": rep.get("combo_n_cv_runs", 0),
            "combo_support_share": rep.get("combo_support_share", 0.0),
            "selection_aggregation": "stage_method_level_then_exact_combo",
        })
        rows.append(row)

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    min_support = float(globals().get("MIN_SELECTION_SUPPORT_SHARE", 0.50))
    stable = out[out["support_share"] >= min_support].copy()
    if not stable.empty:
        out = stable
    return out.sort_values(
        ["selection_score", "support_share", "r2_mean", "r2_median", "r2_q25", "r2_std", "combo_cv_r2_mean"],
        ascending=[False, False, False, False, False, True, False],
    ).reset_index(drop=True)


def select_best_method_level(method_summary_df):
    """Return the best method-level candidate as a Series-like row."""
    if method_summary_df is None or len(method_summary_df) == 0:
        return None
    cand = method_summary_df.copy()
    if "support_share" in cand.columns:
        min_support = float(globals().get("MIN_SELECTION_SUPPORT_SHARE", 0.50))
        stable = cand[cand["support_share"] >= min_support].copy()
        if not stable.empty:
            cand = stable
    sort_cols = [c for c in ["selection_score", "support_share", "r2_mean", "r2_median", "r2_q25", "r2_std", "combo_cv_r2_mean"] if c in cand.columns]
    asc = [False, False, False, False, False, True, False][:len(sort_cols)]
    if sort_cols:
        cand = cand.sort_values(sort_cols, ascending=asc).reset_index(drop=True)
    return cand.iloc[0]




# ------------------------------------------------------------
# STRICT OUTPUT-WISE EXACT-COMBO SELECTOR PATCH
# 핵심 수정:
#   - method-level 평균으로 고른 뒤 representative combo를 고르지 않음.
#   - Output별로 stage × feature_scope × y_strategy × method × model × transform × topk
#     전체 exact combo를 직접 비교함.
#   - 기본 순위는 CV R2 mean 최우선이며, 동률/근접 시 median, q25, std, support, topk를 사용함.
# ------------------------------------------------------------
STRICT_EXACT_OUTPUTWISE_SELECTION = True
EXACT_R2_TIE_TOL = 1e-12


def select_best_exact_combo(exact_summary_df):
    """Select one exact candidate per output without method-level averaging.

    Performance is prioritized first. Support share is kept only as a tie-breaker,
    because small grouped datasets often fragment the best model/top-k choice across folds.
    """
    if exact_summary_df is None or len(exact_summary_df) == 0:
        return None
    cand = exact_summary_df.copy()
    for c in ["r2_mean", "r2_median", "r2_q25", "r2_min", "r2_std", "support_share", "topk", "rmse_mean", "mae_mean"]:
        if c not in cand.columns:
            cand[c] = np.nan
        cand[c] = pd.to_numeric(cand[c], errors="coerce")
    cand = cand[np.isfinite(cand["r2_mean"])].copy()
    if cand.empty:
        return None
    cand["topk_sort"] = cand["topk"].fillna(9999)
    cand["r2_std_sort"] = cand["r2_std"].fillna(9999)
    cand["support_sort"] = cand["support_share"].fillna(0.0)
    cand["rmse_sort"] = cand["rmse_mean"].fillna(np.inf)
    cand = cand.sort_values(
        ["r2_mean", "r2_median", "r2_q25", "r2_min", "r2_std_sort", "support_sort", "topk_sort", "rmse_sort"],
        ascending=[False, False, False, False, True, False, True, True],
    ).reset_index(drop=True)
    row = cand.iloc[0].to_dict()
    row["representative_model_name"] = row.get("model_name", "")
    row["representative_target_transform"] = row.get("target_transform", "raw")
    row["representative_topk"] = row.get("topk", np.nan)
    row["combo_cv_r2_mean"] = row.get("r2_mean", np.nan)
    row["combo_cv_r2_std"] = row.get("r2_std", np.nan)
    row["combo_n_cv_runs"] = row.get("n_cv_runs", 0)
    row["combo_support_share"] = row.get("support_share", 0.0)
    row["combo_selection_score"] = row.get("selection_score", np.nan)
    return row


def build_exact_selection_diagnostic(exact_summary_df, selected_row):
    if exact_summary_df is None or exact_summary_df.empty or selected_row is None:
        return pd.DataFrame()
    df = exact_summary_df.copy()
    for c in ["r2_mean", "r2_median", "r2_q25", "r2_std", "support_share", "topk"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values(["r2_mean", "r2_median", "r2_q25", "r2_std"], ascending=[False, False, False, True]).reset_index(drop=True)
    df.insert(0, "exact_rank_by_r2", np.arange(1, len(df) + 1))
    key_cols = ["stage_id", "feature_scope", "best_y_strategy", "method_name", "model_name", "target_transform", "topk"]
    for c in key_cols:
        if c not in df.columns:
            df[c] = ""
    sel_key = tuple(str(selected_row.get(c, "")) for c in key_cols)
    df["is_selected_exact_combo"] = [tuple(str(row.get(c, "")) for c in key_cols) == sel_key for _, row in df.iterrows()]
    return df

# Registry preview: exactly what will be compared per output and stage.
registry_rows = []
for output_name, bundle in DATA_BY_OUTPUT.items():
    feature_cols_preview = bundle["feature_cols"]
    stages = get_sequential_method_stages(output_name)
    for st in stages:
        for scope in get_feature_scope_candidates_for_output(output_name, feature_cols_preview):
            methods_scope = [m for m in get_methods_for_scope(output_name, scope) if m in st["methods"]]
            if not methods_scope:
                continue
            registry_rows.append({
                "output_name": output_name,
                "output_family": classify_output_family(output_name),
                "stage_id": st["stage_id"],
                "stage_name": st["stage_name"],
                "stage_source": st["stage_source"],
                "feature_scope": scope,
                "n_scope_features": len(get_feature_cols_for_scope(feature_cols_preview, scope)),
                "candidate_y_strategies": ", ".join(get_y_strategy_candidates_for_output(output_name, groups=bundle["df"][bundle["group_col"]].astype(str).to_numpy())),
                "candidate_methods": ", ".join(methods_scope),
                "n_candidate_methods": len(methods_scope),
            })
METHOD_REGISTRY_DF = pd.DataFrame(registry_rows)
export_df(METHOD_REGISTRY_DF, os.path.join(MODEL_EXPORT_DIR, "sequential_stage_method_registry"))
display(METHOD_REGISTRY_DF.head(30))


# ------------------------------------------------------------
# v4 optimized loop
# Key improvement: for each output, the expensive preprocessing bundle
# (scope slicing -> repeated-Y selection -> feature prefilter -> feature ranking)
# is computed once per (fold, feature_scope, y_strategy), then reused across
# Stage A/B/C/D. v3 recomputed the same bundle inside every stage.
# ------------------------------------------------------------

def _build_cached_eval_contexts_for_output(output_name, bundle, split_records, y_strategy_candidates, feature_scope_candidates):
    df = bundle["df"].copy()
    feature_cols_all = bundle["feature_cols"]
    target_col = bundle["target_col"]
    group_col = bundle["group_col"]
    xkey_col = bundle["xkey_col"]

    X_raw_full_all = df[feature_cols_all].copy()
    y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
    g_raw_full = df[group_col].astype(str).to_numpy()
    xkey_full = df[xkey_col].astype(str).to_numpy()
    base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
    X_raw_full_all = X_raw_full_all.loc[base_mask].reset_index(drop=True)
    y_raw_full = y_raw_full[base_mask]
    g_raw_full = g_raw_full[base_mask]
    xkey_full = xkey_full[base_mask]

    contexts = []
    error_rows = []
    for cv_run_id, repeat_id, fold_id, tr_idx, te_idx in split_records:
        for feature_scope in feature_scope_candidates:
            feature_cols_scope = get_feature_cols_for_scope(feature_cols_all, feature_scope)
            if len(feature_cols_scope) < 2:
                continue
            X_scope_full = X_raw_full_all[feature_cols_scope].copy()
            try:
                X_tr_raw = X_scope_full.iloc[tr_idx].reset_index(drop=True)
                y_tr_raw = y_raw_full[tr_idx]
                g_tr_raw = g_raw_full[tr_idx]
                xkey_tr = xkey_full[tr_idx]
                X_te_raw = X_scope_full.iloc[te_idx].reset_index(drop=True)
                y_te_raw = y_raw_full[te_idx]
                g_te_raw = g_raw_full[te_idx]
                xkey_te = xkey_full[te_idx]

                # Prefilter is independent of y_strategy; compute once per fold/scope.
                rough_cols, _ = prefilter_features_groupwise(
                    X_tr_raw,
                    y_tr_raw,
                    g_tr_raw,
                    top_k=get_prefilter_topk_for_output_local(output_name),
                    corr_threshold=CORR_PRUNE_THRESHOLD,
                    random_state=RANDOM_STATE + cv_run_id,
                )
                if len(rough_cols) < 2:
                    continue

                for y_strategy in y_strategy_candidates:
                    try:
                        selected_train_df, _ = select_best_y_within_group(
                            X_tr_raw,
                            y_tr_raw,
                            g_tr_raw,
                            xkey_tr,
                            rough_cols=rough_cols,
                            strategy=y_strategy,
                            random_state=RANDOM_STATE + cv_run_id,
                        )
                        test_group_df = aggregate_test_groups_by_strategy(
                            X_te_raw,
                            y_te_raw,
                            g_te_raw,
                            xkey_te,
                            strategy=y_strategy,
                        )
                        selected_train_df = _ensure_standard_cols(selected_train_df, group_col=group_col, xkey_col=xkey_col,
                                                                  group_values=g_tr_raw, xkey_values=xkey_tr)
                        test_group_df = _ensure_standard_cols(test_group_df, group_col=group_col, xkey_col=xkey_col,
                                                              group_values=g_te_raw, xkey_values=xkey_te)
                        selected_train_df = selected_train_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
                        test_group_df = test_group_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
                        if selected_train_df.empty or test_group_df.empty:
                            continue
                        if "XKEY" not in selected_train_df.columns:
                            selected_train_df["XKEY"] = [f"TR_{cv_run_id}_{i}" for i in range(len(selected_train_df))]

                        X_train_sel = selected_train_df[feature_cols_scope].copy()
                        y_train_sel = selected_train_df["target"].to_numpy(dtype=float)
                        g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
                        X_test_group = test_group_df[feature_cols_scope].copy()
                        y_test_group = test_group_df["target"].to_numpy(dtype=float)
                        g_test_group = test_group_df["GROUP_ID"].astype(str).to_numpy()

                        ranked_features, rank_df = build_ranked_features_train(
                            X_train_sel, y_train_sel, g_train_sel, output_name, cv_run_id
                        )
                        if len(ranked_features) < 2:
                            continue

                        contexts.append({
                            "output_name": output_name,
                            "cv_run_id": int(cv_run_id),
                            "repeat_id": int(repeat_id),
                            "fold_id": int(fold_id),
                            "feature_scope": feature_scope,
                            "y_strategy": y_strategy,
                            "feature_cols_scope": feature_cols_scope,
                            "selected_train_df": selected_train_df,
                            "X_train_sel": X_train_sel,
                            "y_train_sel": y_train_sel,
                            "g_train_sel": g_train_sel,
                            "X_test_group": X_test_group,
                            "y_test_group": y_test_group,
                            "g_test_group": g_test_group,
                            "ranked_features": ranked_features,
                        })
                    except Exception as e:
                        error_rows.append({
                            "output_name": output_name,
                            "cv_run_id": int(cv_run_id),
                            "repeat_id": int(repeat_id),
                            "fold_id": int(fold_id),
                            "output_family": classify_output_family(output_name),
                            "stage_id": "PREP",
                            "stage_name": "cached_preprocessing",
                            "stage_source": "v4_cache",
                            "feature_scope": feature_scope,
                            "best_y_strategy": y_strategy,
                            "method_name": "prep_error",
                            "model_name": "",
                            "target_transform": "",
                            "topk": np.nan,
                            "n_train_groups": np.nan,
                            "n_test_groups": np.nan,
                            "n_features": 0,
                            "selected_features": "",
                            "outer_r2": np.nan,
                            "outer_rmse": np.nan,
                            "outer_mae": np.nan,
                            "search_score": np.nan,
                            "status": "error",
                            "reason": f"{type(e).__name__}: {e}",
                        })
            except Exception as e:
                error_rows.append({
                    "output_name": output_name,
                    "cv_run_id": int(cv_run_id),
                    "repeat_id": int(repeat_id),
                    "fold_id": int(fold_id),
                    "output_family": classify_output_family(output_name),
                    "stage_id": "SCOPE_PREP",
                    "stage_name": "cached_scope_preprocessing",
                    "stage_source": "v4_cache",
                    "feature_scope": feature_scope,
                    "best_y_strategy": "scope_error",
                    "method_name": "scope_error",
                    "model_name": "",
                    "target_transform": "",
                    "topk": np.nan,
                    "n_train_groups": np.nan,
                    "n_test_groups": np.nan,
                    "n_features": 0,
                    "selected_features": "",
                    "outer_r2": np.nan,
                    "outer_rmse": np.nan,
                    "outer_mae": np.nan,
                    "search_score": np.nan,
                    "status": "error",
                    "reason": f"{type(e).__name__}: {e}",
                })
    return contexts, error_rows, X_raw_full_all, y_raw_full, g_raw_full, xkey_full


for output_name, bundle in DATA_BY_OUTPUT.items():
    if TEST_OUTPUTS is not None and output_name not in TEST_OUTPUTS:
        continue
    print(f"\n=== Sequential stage comparison for output: {output_name} ===")

    feature_cols_all = bundle["feature_cols"]
    df = bundle["df"].copy()
    y_raw_check = pd.to_numeric(df[bundle["target_col"]], errors="coerce").to_numpy(dtype=float)
    g_raw_check = df[bundle["group_col"]].astype(str).to_numpy()
    xkey_check = df[bundle["xkey_col"]].astype(str).to_numpy()
    base_mask_check = np.isfinite(y_raw_check) & pd.notna(g_raw_check) & pd.notna(xkey_check)
    n_groups = len(pd.unique(g_raw_check[base_mask_check]))

    if n_groups < MIN_GROUPS_FOR_MODEL:
        METHOD_SUMMARY_ROWS.append({
            "output_name": output_name, "status": "skipped", "reason": f"Not enough groups ({n_groups})",
            "output_family": classify_output_family(output_name), "selected_stage_id": "", "selected_stage_name": "",
            "feature_scope": "", "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
            "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
            "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan,
        })
        continue

    y_strategy_candidates = get_y_strategy_candidates_for_output(output_name, groups=g_raw_check[base_mask_check])
    feature_scope_candidates = get_feature_scope_candidates_for_output(output_name, feature_cols_all)
    stages = get_sequential_method_stages(output_name)

    split_records = list(repeated_group_splits(
        g_raw_check[base_mask_check],
        n_splits=min(FINAL_OUTER_SPLITS, n_groups),
        n_repeats=OUTER_REPEATS,
        random_state=RANDOM_STATE,
        return_split_id=True,
        test_size=OUTER_TEST_SIZE,
    ))

    fold_rows = []
    stage_candidate_tables = []
    out_dir = os.path.join(PER_OUTPUT_DIR, sanitize_filename(output_name))
    os.makedirs(out_dir, exist_ok=True)

    contexts, prep_error_rows, X_raw_full_all, y_raw_full, g_raw_full, xkey_full = _build_cached_eval_contexts_for_output(
        output_name, bundle, split_records, y_strategy_candidates, feature_scope_candidates
    )
    fold_rows.extend(prep_error_rows)

    print(f"  cached preprocessing contexts: {len(contexts)} | prep errors: {len(prep_error_rows)}")
    export_df(pd.DataFrame([{
        "output_name": output_name,
        "n_cached_contexts": len(contexts),
        "n_prep_errors": len(prep_error_rows),
        "n_splits": len(split_records),
        "n_feature_scopes": len(feature_scope_candidates),
        "n_y_strategies": len(y_strategy_candidates),
        "training_speed_profile": TRAINING_SPEED_PROFILE,
        "cpu_tree_n_jobs": CPU_TREE_N_JOBS,
    }]), os.path.join(out_dir, "sequential_v4_cache_diagnostics"))

    for st in stages:
        stage_id = st["stage_id"]
        stage_name = st["stage_name"]
        stage_source = st["stage_source"]
        stage_methods_base = st["methods"]
        print(f"  -> Stage {stage_id}: {stage_name} | methods={stage_methods_base}")
        stage_fold_start = len(fold_rows)

        for ctx in contexts:
            feature_scope = ctx["feature_scope"]
            methods_for_this_scope = [m for m in get_methods_for_scope(output_name, feature_scope) if m in stage_methods_base]
            if not methods_for_this_scope:
                continue
            for method_name in methods_for_this_scope:
                try:
                    if THREADPOOLCTL_AVAILABLE and LIMIT_INTERNAL_NUM_THREADS:
                        with threadpool_limits(limits=INTERNAL_NUM_THREADS):
                            res = evaluate_method_train_test(
                                method_name,
                                output_name,
                                ctx["X_train_sel"],
                                ctx["y_train_sel"],
                                ctx["g_train_sel"],
                                ctx["selected_train_df"],
                                ctx["X_test_group"],
                                ctx["y_test_group"],
                                ctx["ranked_features"],
                                random_state=RANDOM_STATE + ctx["cv_run_id"],
                            )
                    else:
                        res = evaluate_method_train_test(
                            method_name,
                            output_name,
                            ctx["X_train_sel"],
                            ctx["y_train_sel"],
                            ctx["g_train_sel"],
                            ctx["selected_train_df"],
                            ctx["X_test_group"],
                            ctx["y_test_group"],
                            ctx["ranked_features"],
                            random_state=RANDOM_STATE + ctx["cv_run_id"],
                        )
                    if res is None:
                        continue
                    fold_rows.append({
                        "output_name": output_name,
                        "cv_run_id": int(ctx["cv_run_id"]),
                        "repeat_id": int(ctx["repeat_id"]),
                        "fold_id": int(ctx["fold_id"]),
                        "output_family": classify_output_family(output_name),
                        "stage_id": stage_id,
                        "stage_name": stage_name,
                        "stage_source": stage_source,
                        "feature_scope": feature_scope,
                        "best_y_strategy": ctx["y_strategy"],
                        "method_name": res["method_name"],
                        "model_name": res["model_name"],
                        "target_transform": res["target_transform"],
                        "topk": int(res["topk"]),
                        "n_train_groups": int(len(pd.unique(ctx["g_train_sel"]))),
                        "n_test_groups": int(len(pd.unique(ctx["g_test_group"]))),
                        "n_features": int(res["n_features"]),
                        "selected_features": ", ".join(res["selected_features"]),
                        "outer_r2": float(res["outer_r2"]),
                        "outer_rmse": float(res["outer_rmse"]),
                        "outer_mae": float(res["outer_mae"]),
                        "search_score": float(res["search_score"]),
                        "status": "ok",
                        "reason": "",
                    })
                except Exception as e:
                    fold_rows.append({
                        "output_name": output_name,
                        "cv_run_id": int(ctx["cv_run_id"]),
                        "repeat_id": int(ctx["repeat_id"]),
                        "fold_id": int(ctx["fold_id"]),
                        "output_family": classify_output_family(output_name),
                        "stage_id": stage_id,
                        "stage_name": stage_name,
                        "stage_source": stage_source,
                        "feature_scope": feature_scope,
                        "best_y_strategy": ctx["y_strategy"],
                        "method_name": method_name,
                        "model_name": "",
                        "target_transform": "",
                        "topk": np.nan,
                        "n_train_groups": np.nan,
                        "n_test_groups": np.nan,
                        "n_features": 0,
                        "selected_features": "",
                        "outer_r2": np.nan,
                        "outer_rmse": np.nan,
                        "outer_mae": np.nan,
                        "search_score": np.nan,
                        "status": "error",
                        "reason": f"{type(e).__name__}: {e}",
                    })

        # Stage-level summary after this stage.
        fold_df_stage = pd.DataFrame(fold_rows[stage_fold_start:])
        ok_stage = fold_df_stage[(fold_df_stage.get("status", "") == "ok") & np.isfinite(fold_df_stage.get("outer_r2", np.nan))] if not fold_df_stage.empty else pd.DataFrame()
        if not ok_stage.empty:
            stage_method_summary = summarize_method_level(ok_stage, total_cv_runs=len(split_records))
            stage_method_summary["output_name"] = output_name
            stage_method_summary["stage_id"] = stage_id
            stage_method_summary["stage_name"] = stage_name
            stage_method_summary["stage_source"] = stage_source
            stage_candidate_tables.append(stage_method_summary)
            STAGE_SUMMARY_ROWS_ALL.extend(stage_method_summary.to_dict("records"))
            export_df(stage_method_summary, os.path.join(out_dir, f"stage_{stage_id}_method_summary"))
        else:
            print(f"     Stage {stage_id}: no successful candidates")

    fold_df = pd.DataFrame(fold_rows)
    export_df(fold_df, os.path.join(out_dir, "sequential_all_stage_fold_metrics"))

    if fold_df.empty or not ((fold_df.get("status", "") == "ok") & np.isfinite(fold_df.get("outer_r2", np.nan))).any():
        METHOD_SUMMARY_ROWS.append({
            "output_name": output_name, "status": "failed", "reason": "No successful stage candidate",
            "output_family": classify_output_family(output_name), "selected_stage_id": "", "selected_stage_name": "",
            "feature_scope": "", "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
            "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
            "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan,
        })
        continue

    ok_df = fold_df[(fold_df["status"] == "ok") & np.isfinite(fold_df["outer_r2"])].copy()
    total_cv_runs = len(split_records)

    # STRICT FIX: summarize exact candidates directly instead of averaging over method-level families.
    all_method_summary = summarize_exact_combo(ok_df, total_cv_runs=total_cv_runs)
    export_df(all_method_summary, os.path.join(out_dir, "sequential_final_exact_combo_candidates"))

    if not all_method_summary.empty:
        pivot = all_method_summary.pivot_table(
            index=["stage_id", "feature_scope", "best_y_strategy", "method_name", "model_name", "target_transform", "topk"],
            values=["r2_mean", "r2_median", "r2_q25", "r2_std", "support_share", "selection_score"],
            aggfunc="first"
        ).reset_index().sort_values(["r2_mean", "r2_median", "r2_q25", "r2_std"], ascending=[False, False, False, True])
        export_df(pivot, os.path.join(out_dir, "sequential_exact_combo_performance_pivot"))

        scope_pivot = all_method_summary.pivot_table(
            index=["stage_id", "feature_scope"],
            values=["r2_mean", "selection_score"],
            aggfunc="max"
        ).reset_index().sort_values("selection_score", ascending=False)
        export_df(scope_pivot, os.path.join(out_dir, "sequential_stage_strategy_scope_pivot"))

    chosen_method_row = select_best_exact_combo(all_method_summary)
    if chosen_method_row is None:
        METHOD_SUMMARY_ROWS.append({
            "output_name": output_name, "status": "failed", "reason": "Exact-combo selection failed",
            "output_family": classify_output_family(output_name), "selected_stage_id": "", "selected_stage_name": "",
            "feature_scope": "", "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
            "final_chosen_topk": np.nan, "n_cv_runs": 0, "support_share": 0.0,
            "cv_r2_mean": np.nan, "cv_r2_std": np.nan, "cv_rmse_mean": np.nan, "cv_mae_mean": np.nan,
        })
        continue

    candidate_rows = ok_df[
        (ok_df["stage_id"] == chosen_method_row["stage_id"]) &
        (ok_df["feature_scope"] == chosen_method_row["feature_scope"]) &
        (ok_df["best_y_strategy"] == chosen_method_row["best_y_strategy"]) &
        (ok_df["method_name"] == chosen_method_row["method_name"]) &
        (ok_df["model_name"].astype(str) == str(chosen_method_row.get("model_name", ""))) &
        (ok_df["target_transform"].astype(str) == str(chosen_method_row.get("target_transform", ""))) &
        (pd.to_numeric(ok_df["topk"], errors="coerce").astype(float) == float(chosen_method_row.get("topk", np.nan)))
    ].copy()
    combo_summary = summarize_exact_combo(candidate_rows, total_cv_runs=total_cv_runs)
    export_df(combo_summary, os.path.join(out_dir, "sequential_selected_exact_combo_fold_summary"))
    representative = chosen_method_row.copy()

    if representative is None:
        representative = {
            "model_name": chosen_method_row.get("representative_model_name", ""),
            "target_transform": chosen_method_row.get("representative_target_transform", "raw"),
            "topk": chosen_method_row.get("representative_topk", np.nan),
            "n_features_median": np.nan,
            "selected_features_mode": "",
        }

    stage_winner_df = all_method_summary.sort_values(["r2_mean", "r2_median", "r2_q25", "r2_std"], ascending=[False, False, False, True]).groupby("stage_id", as_index=False).head(1)
    export_df(stage_winner_df, os.path.join(out_dir, "sequential_stage_best_exact_combo_pivot"))
    exact_diag = build_exact_selection_diagnostic(all_method_summary, chosen_method_row)
    export_df(exact_diag, os.path.join(out_dir, "sequential_exact_combo_selection_diagnostic"))
    PER_OUTPUT_RESULTS[output_name] = {
        "fold_df": fold_df,
        "method_summary": all_method_summary,
        "combo_summary": combo_summary,
        "chosen_method": chosen_method_row,
        "representative_combo": representative,
    }

    METHOD_SUMMARY_ROWS.append({
        "output_name": output_name,
        "status": "ok",
        "reason": "",
        "output_family": classify_output_family(output_name),
        "selected_stage_id": chosen_method_row["stage_id"],
        "selected_stage_name": chosen_method_row.get("stage_name", ""),
        "selected_stage_source": chosen_method_row.get("stage_source", ""),
        "feature_scope": chosen_method_row["feature_scope"],
        "best_y_strategy": chosen_method_row["best_y_strategy"],
        "final_method_name": chosen_method_row["method_name"],
        "final_model_name": representative.get("model_name", chosen_method_row.get("representative_model_name", "")),
        "target_transform": representative.get("target_transform", chosen_method_row.get("representative_target_transform", "raw")),
        "final_chosen_topk": representative.get("topk", chosen_method_row.get("representative_topk", np.nan)),
        "n_cv_runs": int(chosen_method_row.get("n_cv_runs", 0)),
        "support_share": float(chosen_method_row.get("support_share", 0.0)),
        "cv_r2_mean": float(chosen_method_row.get("r2_mean", np.nan)),
        "cv_r2_median": float(chosen_method_row.get("r2_median", np.nan)),
        "cv_r2_q25": float(chosen_method_row.get("r2_q25", np.nan)),
        "cv_r2_std": float(chosen_method_row.get("r2_std", np.nan)),
        "cv_rmse_mean": float(chosen_method_row.get("rmse_mean", np.nan)),
        "cv_mae_mean": float(chosen_method_row.get("mae_mean", np.nan)),
        "selection_score": float(chosen_method_row.get("selection_score", np.nan)),
        "selection_score_obj": float(chosen_method_row.get("selection_score", np.nan)),
        "selection_aggregation": "strict_outputwise_exact_combo",
        "combo_cv_r2_mean": representative.get("combo_cv_r2_mean", np.nan),
        "combo_cv_r2_std": representative.get("combo_cv_r2_std", np.nan),
        "combo_n_cv_runs": representative.get("combo_n_cv_runs", 0),
        "combo_support_share": representative.get("combo_support_share", 0.0),
        "representative_n_features_median": representative.get("n_features_median", np.nan),
        "representative_selected_features_mode": representative.get("selected_features_mode", ""),
        "training_speed_profile": TRAINING_SPEED_PROFILE,
        "cpu_tree_n_jobs": CPU_TREE_N_JOBS,
    })

    print("  Selected exact combo:", chosen_method_row["stage_id"], chosen_method_row["feature_scope"], chosen_method_row["best_y_strategy"], chosen_method_row["method_name"],
          "| model=", chosen_method_row.get("model_name", ""),
          "| transform=", chosen_method_row.get("target_transform", ""),
          "| topk=", chosen_method_row.get("topk", np.nan),
          "| R2_mean=", round(float(chosen_method_row.get("r2_mean", np.nan)), 4),
          "| support=", round(float(chosen_method_row.get("support_share", 0.0)), 3))

METHOD_SUMMARY_DF = pd.DataFrame(METHOD_SUMMARY_ROWS)
for c in [
    "output_name", "status", "reason", "output_family", "selected_stage_id", "selected_stage_name", "selected_stage_source",
    "feature_scope", "best_y_strategy", "final_method_name", "final_model_name", "target_transform", "final_chosen_topk",
    "n_cv_runs", "support_share", "cv_r2_mean", "cv_r2_median", "cv_r2_min", "cv_r2_q25", "cv_r2_std",
    "cv_rmse_mean", "cv_mae_mean", "selection_score", "selection_score_obj", "selection_aggregation",
    "combo_cv_r2_mean", "combo_cv_r2_std", "combo_n_cv_runs", "combo_support_share",
]:
    if c not in METHOD_SUMMARY_DF.columns:
        METHOD_SUMMARY_DF[c] = np.nan

METHOD_FOLD_DF = pd.concat([PER_OUTPUT_RESULTS[k]["fold_df"] for k in PER_OUTPUT_RESULTS], axis=0).reset_index(drop=True) if PER_OUTPUT_RESULTS else pd.DataFrame()
STAGE_SUMMARY_DF = pd.DataFrame(STAGE_SUMMARY_ROWS_ALL)

export_df(METHOD_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_sequential_final_summary"))
export_df(STAGE_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_stage_winner_summary"))
# Backward-compatible filename for later cells.
export_df(METHOD_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_summary"))
if not METHOD_FOLD_DF.empty:
    export_df(METHOD_FOLD_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_sequential_stage_folds"))
    export_df(METHOD_FOLD_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_method_comparison_folds"))

display(METHOD_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False) if not METHOD_SUMMARY_DF.empty else METHOD_SUMMARY_DF)


,output_name,output_family,stage_id,stage_name,stage_source,feature_scope,n_scope_features,candidate_y_strategies,candidate_methods,n_candidate_methods
0,Modulus,mechanical_energy,A,1st_legacy_engineered_blend,Training_260418 + engineered additions,legacy_core_only,152,singleton_raw,"weighted_blend_top2, weighted_blend_top3, engi...",5
1,Modulus,mechanical_energy,A,1st_legacy_engineered_blend,Training_260418 + engineered additions,added_only,17,singleton_raw,"weighted_blend_top2, weighted_blend_top3, engi...",5
2,Modulus,mechanical_energy,A,1st_legacy_engineered_blend,Training_260418 + engineered additions,all_features,169,singleton_raw,"weighted_blend_top2, weighted_blend_top3, engi...",5
3,Modulus,mechanical_energy,B,2nd_stability_spca_blockpca_bagging_multitask,Training_260419 / stability-aware family,legacy_core_only,152,singleton_raw,"baseline_stability, stability_lasso_ridge, spc...",8
4,Modulus,mechanical_energy,B,2nd_stability_spca_blockpca_bagging_multitask,Training_260419 / stability-aware family,added_only,17,singleton_raw,"baseline_stability, stability_lasso_ridge, spc...",8
5,Modulus,mechanical_energy,B,2nd_stability_spca_blockpca_bagging_multitask,Training_260419 / stability-aware family,all_features,169,singleton_raw,"baseline_stability, stability_lasso_ridge, spc...",8
6,Modulus,mechanical_energy,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,legacy_core_only,152,singleton_raw,"direct_ard, direct_elasticnet_cv, direct_omp_r...",18
7,Modulus,mechanical_energy,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,added_only,17,singleton_raw,"direct_ard, direct_elasticnet_cv, direct_omp_r...",18
8,Modulus,mechanical_energy,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,all_features,169,singleton_raw,"direct_ard, direct_elasticnet_cv, direct_omp_r...",18
9,Modulus,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,169,singleton_raw,featureaware_partial_missing_ensemble,1



=== Sequential stage comparison for output: Modulus ===
  cached preprocessing contexts: 24 | prep errors: 0
  -> Stage A: 1st_legacy_engineered_blend | methods=['weighted_blend_top2', 'weighted_blend_top3', 'engineered_ridge', 'engineered_bayesian_ridge', 'engineered_huber']
     Stage A: no successful candidates
  -> Stage B: 2nd_stability_spca_blockpca_bagging_multitask | methods=['baseline_stability', 'stability_lasso_ridge', 'spca_ridge', 'spca_huber', 'spca_pls', 'block_pca_ridge', 'bagged_subspace_ridge', 'minimal_class_average']
  -> Stage C: 3rd_direct_residual_shallow_nonlinear | methods=['direct_ard', 'direct_elasticnet_cv', 'direct_omp_ridge', 'direct_quantile_median', 'direct_theilsen', 'direct_ransac_ridge', 'direct_gpr_matern', 'direct_gpr_rq', 'residual_huber_krr', 'residual_ard_krr', 'residual_ridge_extratrees', 'residual_pls_extratrees', 'direct_extra_trees_shallow', 'direct_random_forest_shallow', 'direct_hist_gb', 'direct_svr_rbf', 'direct_svr_linear', 'direct_knn_

,output_name,status,reason,output_family,selected_stage_id,selected_stage_name,selected_stage_source,feature_scope,best_y_strategy,final_method_name,...,selection_aggregation,combo_cv_r2_mean,combo_cv_r2_std,combo_n_cv_runs,combo_support_share,representative_n_features_median,representative_selected_features_mode,training_speed_profile,cpu_tree_n_jobs,cv_r2_min
8,Thermal characteristics | h | W/m.K,ok,,thermal,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,added_only,singleton_raw,residual_pls_extratrees,...,strict_outputwise_exact_combo,0.885511,0.0,1,0.125,8.0,"Global: l/d STDEV | Criteria | 5x5x5, Criteria...",balanced_fast,18,NaN
4,Yield strength,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.672766,0.0,1,0.125,3.0,Structural parameters: Internal Sound | Overla...,balanced_fast,18,NaN
6,Total energy,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.589238,0.0,1,0.125,10.0,Structural parameters: Internal Sound | Overla...,balanced_fast,18,NaN
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,hard_closest_oof,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.554456,0.0,1,0.125,9.0,Structural parameters: External | Overlay imag...,balanced_fast,18,NaN
3,AS,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.523654,0.0,1,0.125,10.0,Structural parameters: Internal Sound | Overla...,balanced_fast,18,NaN
10,Thermal characteristics | Cooling rate | °C/s,ok,,thermal,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,legacy_core_only,singleton_raw,direct_knn_distance,...,strict_outputwise_exact_combo,0.518897,0.0,1,0.125,6.0,Structural parameters: External | Overlay imag...,balanced_fast,18,NaN
0,Modulus,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.513660,0.0,1,0.125,10.0,Structural parameters: Internal Heat dissipati...,balanced_fast,18,NaN
2,APS,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.513601,0.0,1,0.125,10.0,Structural parameters: External | Overlay imag...,balanced_fast,18,NaN
9,Thermal characteristics | Heating rate | °C/s,ok,,thermal,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.505359,0.0,1,0.125,12.0,Structural parameters: External | Global distr...,balanced_fast,18,NaN
11,Thermal characteristics | Heating Temp | °C/s,ok,,thermal,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,strict_outputwise_exact_combo,0.485393,0.0,1,0.125,12.0,Structural parameters: External | Global distr...,balanced_fast,18,NaN


In [11]:

# ============================================================
# Cell E1. Refit sequential-stage selected best method on full data
# - uses the winning feature_scope + Y strategy + method selected in D1
# ============================================================

FINAL_REFIT_SUMMARY_ROWS = []
FINAL_MODEL_BACKUP = {}


def _ensure_standard_cols_refit(df_in, *, group_col=None, xkey_col=None, group_values=None, xkey_values=None):
    df = df_in.copy()
    if "GROUP_ID" not in df.columns:
        if group_col is not None and group_col in df.columns:
            df = df.rename(columns={group_col: "GROUP_ID"})
        elif group_values is not None and len(df) == len(group_values):
            df["GROUP_ID"] = group_values[:len(df)]
    if "XKEY" not in df.columns:
        if xkey_col is not None and xkey_col in df.columns:
            df = df.rename(columns={xkey_col: "XKEY"})
        elif xkey_values is not None and len(df) == len(xkey_values):
            df["XKEY"] = xkey_values[:len(df)]
    if "GROUP_ID" in df.columns:
        df["GROUP_ID"] = df["GROUP_ID"].astype(str)
    if "XKEY" in df.columns:
        df["XKEY"] = df["XKEY"].astype(str)
    if "target" in df.columns:
        df["target"] = pd.to_numeric(df["target"], errors="coerce")
    return df


for output_name, bundle in DATA_BY_OUTPUT.items():
    try:
        hit = METHOD_SUMMARY_DF[(METHOD_SUMMARY_DF["output_name"] == output_name) & (METHOD_SUMMARY_DF["status"] == "ok")]
        if hit.empty:
            FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "skipped", "reason": "No successful method summary row",
                                             "feature_scope": "", "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                             "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                             "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})
            continue
        chosen = hit.iloc[0]
        chosen_method = chosen["final_method_name"]
        chosen_scope = chosen.get("feature_scope", "all_features")
        chosen_scope = "all_features" if pd.isna(chosen_scope) or str(chosen_scope).strip() == "" else str(chosen_scope)
        chosen_strategy = chosen["best_y_strategy"]
        chosen_stage_id = chosen.get("selected_stage_id", "")
        chosen_stage_name = chosen.get("selected_stage_name", "")
        chosen_stage_source = chosen.get("selected_stage_source", "")

        # Strict refit setting: use the exact model/transform/top-k selected by D1.
        chosen_model_name = str(chosen.get("final_model_name", "")).strip()
        chosen_transform = str(chosen.get("target_transform", "raw")).strip() or "raw"
        chosen_topk = int(chosen["final_chosen_topk"]) if pd.notna(chosen["final_chosen_topk"]) else np.nan

        df = bundle["df"].copy()
        feature_cols_all = bundle["feature_cols"]
        target_col = bundle["target_col"]
        group_col = bundle["group_col"]
        xkey_col = bundle["xkey_col"]
        feature_cols_scope = get_feature_cols_for_scope(feature_cols_all, chosen_scope)

        X_raw_full = df[feature_cols_scope].copy()
        y_raw_full = pd.to_numeric(df[target_col], errors="coerce").to_numpy(dtype=float)
        g_raw_full = df[group_col].astype(str).to_numpy()
        xkey_full = df[xkey_col].astype(str).to_numpy()
        base_mask = np.isfinite(y_raw_full) & pd.notna(g_raw_full) & pd.notna(xkey_full)
        X_raw_full = X_raw_full.loc[base_mask].reset_index(drop=True)
        y_raw_full = y_raw_full[base_mask]
        g_raw_full = g_raw_full[base_mask]
        xkey_full = xkey_full[base_mask]

        rough_cols, _ = prefilter_features_groupwise(
            X_raw_full,
            y_raw_full,
            g_raw_full,
            top_k=get_prefilter_topk_for_output_local(output_name),
            corr_threshold=CORR_PRUNE_THRESHOLD,
            random_state=RANDOM_STATE + 999,
        )
        if len(rough_cols) < 2:
            raise RuntimeError("rough_cols < 2 in refit")

        selected_train_df, candidate_df = select_best_y_within_group(
            X_raw_full,
            y_raw_full,
            g_raw_full,
            xkey_full,
            rough_cols=rough_cols,
            strategy=chosen_strategy,
            random_state=RANDOM_STATE + 999,
        )
        selected_train_df = _ensure_standard_cols_refit(selected_train_df, group_col=group_col, xkey_col=xkey_col, group_values=g_raw_full, xkey_values=xkey_full)
        selected_train_df = selected_train_df.dropna(subset=["target", "GROUP_ID"]).reset_index(drop=True)
        if selected_train_df.empty:
            raise RuntimeError("selected_train_df empty after cleanup")
        if "XKEY" not in selected_train_df.columns:
            selected_train_df["XKEY"] = [f"FULL_{i}" for i in range(len(selected_train_df))]

        X_train_sel = selected_train_df[feature_cols_scope].copy()
        y_train_sel = selected_train_df["target"].to_numpy(dtype=float)
        g_train_sel = selected_train_df["GROUP_ID"].astype(str).to_numpy()
        ranked_features, rank_df = build_ranked_features_train(X_train_sel, y_train_sel, g_train_sel, output_name, 999)
        if len(ranked_features) < 2:
            raise RuntimeError("ranked_features < 2 in refit")

        # Final refit should follow the exact combo selected in D1 as closely as possible.
        # Several method families call shared global candidate lists internally, so temporarily
        # restrict those lists during this one refit call and restore them immediately after.
        _orig_target_transforms = list(globals().get("TARGET_TRANSFORM_CANDIDATES", []))
        _orig_get_topk = globals().get("get_topk_candidates_for_output", None)
        _orig_get_candidate_models = globals().get("get_candidate_models_for_output", None)
        _forced_refit_note = ""
        try:
            if bool(globals().get("EXACT_REFIT_ENFORCE_SELECTED_COMBO", True)):
                if chosen_transform:
                    globals()["TARGET_TRANSFORM_CANDIDATES"] = [chosen_transform]
                if pd.notna(chosen_topk):
                    def _forced_get_topk_candidates_for_output(_output_name, _topk=int(chosen_topk)):
                        return [_topk]
                    globals()["get_topk_candidates_for_output"] = _forced_get_topk_candidates_for_output
                if chosen_model_name:
                    def _forced_get_candidate_models_for_output(_output_name, _model=chosen_model_name):
                        return [_model]
                    globals()["get_candidate_models_for_output"] = _forced_get_candidate_models_for_output
                _forced_refit_note = f"forced exact refit: model={chosen_model_name}, transform={chosen_transform}, topk={chosen_topk}"

            res = evaluate_method_train_test(
                chosen_method,
                output_name,
                X_train_sel,
                y_train_sel,
                g_train_sel,
                selected_train_df,
                X_train_sel.copy(),
                y_train_sel.copy(),
                ranked_features,
                random_state=RANDOM_STATE + 999,
            )
        finally:
            globals()["TARGET_TRANSFORM_CANDIDATES"] = _orig_target_transforms
            if _orig_get_topk is not None:
                globals()["get_topk_candidates_for_output"] = _orig_get_topk
            if _orig_get_candidate_models is not None:
                globals()["get_candidate_models_for_output"] = _orig_get_candidate_models

        # If a complex method has its own internal selector and still returns a different exact combo,
        # preserve both values in the backup/summary for diagnosis rather than silently hiding the mismatch.
        refit_combo_mismatch = bool(
            (chosen_model_name and str(res.get("model_name", "")) != str(chosen_model_name)) or
            (chosen_transform and str(res.get("target_transform", "")) != str(chosen_transform)) or
            (pd.notna(chosen_topk) and int(res.get("topk", -9999)) != int(chosen_topk))
        )
        if res is None:
            raise RuntimeError("evaluate_method_train_test returned None in refit")

        feats = list(res["selected_features"])
        fitted_estimator = res["fitted_estimator"]
        if chosen_method in ("spca_ridge", "spca_huber", "spca_pls"):
            Z_full = transform_spca(fitted_estimator["spca_obj"], X_train_sel[ranked_features])
            train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "block_pca_ridge":
            Z_full = transform_block_pca(fitted_estimator["block_obj"], X_train_sel)
            train_pred = np.asarray(fitted_estimator["reg_model"].predict(Z_full)).reshape(-1)
        elif chosen_method == "minimal_class_average":
            members = fitted_estimator["members"]
            weights = np.asarray(fitted_estimator["weights"], dtype=float)
            preds = []
            for mem in members:
                preds.append(np.asarray(mem["est"].predict(X_train_sel[mem["feats"]])).reshape(-1))
            train_pred = np.average(np.vstack(preds), axis=0, weights=weights)
        elif chosen_method == "direct_omp_ridge":
            prep = fitted_estimator["prep"]
            ridge = fitted_estimator["ridge"]
            idx = fitted_estimator["idx"]
            train_pred = np.asarray(ridge.predict(prep.transform(X_train_sel[feats])[:, idx])).reshape(-1)
        elif chosen_method in ("direct_gpr_matern", "direct_gpr_rq"):
            prep = fitted_estimator["prep"]
            gpr = fitted_estimator["gpr"]
            train_pred = np.asarray(gpr.predict(prep.transform(X_train_sel[feats]))).reshape(-1)
        elif chosen_method in ("residual_huber_krr", "residual_ard_krr"):
            base_pred = np.asarray(fitted_estimator["base_estimator"].predict(X_train_sel[feats])).reshape(-1)
            train_pred = base_pred + fitted_estimator["krr"].predict(fitted_estimator["prep"].transform(X_train_sel[feats]))
        else:
            # Works for normal sklearn estimators, engineered wrappers, weighted blend, and partial-feature-aware ensemble.
            train_pred = np.asarray(fitted_estimator.predict(X_train_sel[feats])).reshape(-1)

        tr_r2, tr_rmse, tr_mae = score_prediction(y_train_sel, train_pred)
        selected_export_cols = [c for c in ["GROUP_ID", "XKEY"] + feats + ["target"] if c in selected_train_df.columns]
        selected_export_df = selected_train_df[selected_export_cols].copy().rename(columns={"target": "selected_target"})
        out_safe = sanitize_filename(output_name)
        export_df(selected_export_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_final_selected_input_output"))
        export_df(candidate_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_candidate_repeated_rows"))
        export_df(rank_df, os.path.join(FINAL_DATA_DIR, f"{out_safe}_feature_rank"))

        backup_payload = {
            "output_name": output_name,
            "feature_scope": chosen_scope,
            "method_name": chosen_method,
            "model_name": res["model_name"],
            "target_transform": res["target_transform"],
            "selected_cv_model_name": chosen_model_name,
            "selected_cv_target_transform": chosen_transform,
            "selected_cv_topk": int(chosen_topk) if pd.notna(chosen_topk) else np.nan,
            "refit_combo_mismatch": refit_combo_mismatch,
            "forced_refit_note": _forced_refit_note,
            "selected_features": feats,
            "ranked_features": ranked_features,
            "fitted_estimator": fitted_estimator,
            "selected_export_df": selected_export_df,
        }
        backup_path = os.path.join(FINAL_BACKUP_DIR, f"{out_safe}_model_backup.joblib")
        joblib.dump(backup_payload, backup_path)
        FINAL_MODEL_BACKUP[output_name] = backup_payload

        FINAL_REFIT_SUMMARY_ROWS.append({
            "output_name": output_name,
            "status": "ok",
            "reason": "",
            "selected_stage_id": chosen_stage_id,
            "selected_stage_name": chosen_stage_name,
            "selected_stage_source": chosen_stage_source,
            "feature_scope": chosen_scope,
            "best_y_strategy": chosen_strategy,
            "final_method_name": chosen_method,
            "final_model_name": res["model_name"],
            "target_transform": res["target_transform"],
            "final_chosen_topk": int(chosen_topk) if pd.notna(chosen_topk) else np.nan,
            "selected_cv_model_name": chosen_model_name,
            "selected_cv_target_transform": chosen_transform,
            "selected_cv_topk": int(chosen_topk) if pd.notna(chosen_topk) else np.nan,
            "refit_combo_mismatch": refit_combo_mismatch,
            "forced_refit_note": _forced_refit_note,
            "n_groups_full": int(len(pd.unique(g_train_sel))),
            "n_final_features": int(len(feats)),
            "final_train_r2": float(tr_r2) if pd.notna(tr_r2) else np.nan,
            "final_train_rmse": float(tr_rmse) if pd.notna(tr_rmse) else np.nan,
            "final_train_mae": float(tr_mae) if pd.notna(tr_mae) else np.nan,
        })
    except Exception as e:
        FINAL_REFIT_SUMMARY_ROWS.append({"output_name": output_name, "status": "error", "reason": f"{type(e).__name__}: {e}",
                                         "feature_scope": "", "best_y_strategy": "", "final_method_name": "", "final_model_name": "", "target_transform": "",
                                         "final_chosen_topk": np.nan, "n_groups_full": 0, "n_final_features": 0,
                                         "final_train_r2": np.nan, "final_train_rmse": np.nan, "final_train_mae": np.nan})

FINAL_REFIT_SUMMARY_DF = pd.DataFrame(FINAL_REFIT_SUMMARY_ROWS)
for c in ["output_name", "status", "reason", "selected_stage_id", "selected_stage_name", "selected_stage_source", "feature_scope", "best_y_strategy", "final_method_name", "final_model_name", "target_transform", "final_chosen_topk", "selected_cv_model_name", "selected_cv_target_transform", "selected_cv_topk", "refit_combo_mismatch", "forced_refit_note", "n_groups_full", "n_final_features", "final_train_r2", "final_train_rmse", "final_train_mae"]:
    if c not in FINAL_REFIT_SUMMARY_DF.columns:
        FINAL_REFIT_SUMMARY_DF[c] = np.nan
export_df(FINAL_REFIT_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_refit_summary"))
display(FINAL_REFIT_SUMMARY_DF.sort_values("final_train_r2", ascending=False) if not FINAL_REFIT_SUMMARY_DF.empty else FINAL_REFIT_SUMMARY_DF)


,output_name,status,reason,selected_stage_id,selected_stage_name,selected_stage_source,feature_scope,best_y_strategy,final_method_name,final_model_name,...,selected_cv_model_name,selected_cv_target_transform,selected_cv_topk,refit_combo_mismatch,forced_refit_note,n_groups_full,n_final_features,final_train_r2,final_train_rmse,final_train_mae
10,Thermal characteristics | Cooling rate | °C/s,ok,,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,legacy_core_only,singleton_raw,direct_knn_distance,KNN_Distance_Direct,...,KNN_Distance_Direct,raw,6,True,"forced exact refit: model=KNN_Distance_Direct,...",65,2,1.000000,0.000000,0.000000
8,Thermal characteristics | h | W/m.K,ok,,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,added_only,singleton_raw,residual_pls_extratrees,Residual_PLS_ExtraTrees,...,Residual_PLS_ExtraTrees,raw,8,True,forced exact refit: model=Residual_PLS_ExtraTr...,25,4,0.718900,0.696304,0.581411
0,Modulus,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Ridge+full_missingI...,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,10,True,forced exact refit: model=FeatureAwarePartial(...,56,12,0.679111,1993.163421,1611.983422
4,Yield strength,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Ridge+full_missingI...,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,3,True,forced exact refit: model=FeatureAwarePartial(...,56,10,0.627947,29.425192,22.601537
6,Total energy,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Huber_Regressor+ful...,...,FeatureAwarePartial(common_Huber_Regressor+ful...,yeo_johnson,10,True,forced exact refit: model=FeatureAwarePartial(...,56,9,0.616152,39.794793,29.879306
7,Thermal characteristics | Thermal conductivity...,ok,,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,legacy_core_only,singleton_raw,residual_ridge_extratrees,Residual_Ridge_ExtraTrees,...,Residual_Ridge_ExtraTrees,yeo_johnson,3,True,forced exact refit: model=Residual_Ridge_Extra...,65,4,0.568412,0.834118,0.486511
5,Densif. strength,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Huber_Regressor+ful...,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,8,True,forced exact refit: model=FeatureAwarePartial(...,56,9,0.521148,167.667633,126.383102
9,Thermal characteristics | Heating rate | °C/s,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Bayesian_Ridge+full...,...,FeatureAwarePartial(common_Ridge+full_missingI...,yeo_johnson,12,True,forced exact refit: model=FeatureAwarePartial(...,65,10,0.512219,0.001597,0.001246
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,hard_closest_oof,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Ridge+full_missingI...,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,9,True,forced exact refit: model=FeatureAwarePartial(...,65,10,0.488920,0.057032,0.038353
3,AS,ok,,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,FeatureAwarePartial(common_Huber_Regressor+ful...,...,FeatureAware

In [12]:

# ============================================================
# Cell F1. Final merged summary
# ============================================================

FINAL_SUMMARY_DF = METHOD_SUMMARY_DF.merge(
    FINAL_REFIT_SUMMARY_DF,
    on="output_name",
    how="left",
    suffixes=("_cv", "_refit")
)

if "cv_r2_mean" not in FINAL_SUMMARY_DF.columns:
    FINAL_SUMMARY_DF["cv_r2_mean"] = np.nan

export_df(FINAL_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "final_summary_merged"))

if not FINAL_SUMMARY_DF.empty:
    display(FINAL_SUMMARY_DF.sort_values("cv_r2_mean", ascending=False))
else:
    display(FINAL_SUMMARY_DF)


# Additional feature-scope diagnostic across all Outputs.
try:
    SCOPE_DIAG_ROWS = []
    for _out, _obj in PER_OUTPUT_RESULTS.items():
        _ms = _obj.get("method_summary", pd.DataFrame())
        if _ms is not None and not _ms.empty:
            _tmp = _ms.copy()
            _tmp["output_name"] = _out
            SCOPE_DIAG_ROWS.append(_tmp)
    if SCOPE_DIAG_ROWS:
        FEATURE_SCOPE_EXACT_SUMMARY_DF = pd.concat(SCOPE_DIAG_ROWS, ignore_index=True)
        export_df(FEATURE_SCOPE_EXACT_SUMMARY_DF, os.path.join(MODEL_EXPORT_DIR, "all_output_exact_combo_candidates"))
        FEATURE_SCOPE_BEST_BY_OUTPUT_DF = (
            FEATURE_SCOPE_EXACT_SUMMARY_DF.sort_values(["output_name", "r2_mean", "r2_median", "r2_q25", "r2_std"], ascending=[True, False, False, False, True])
            .groupby(["output_name", "feature_scope"], as_index=False)
            .head(1)
            .sort_values(["output_name", "r2_mean"], ascending=[True, False])
        )
        export_df(FEATURE_SCOPE_BEST_BY_OUTPUT_DF, os.path.join(MODEL_EXPORT_DIR, "feature_scope_best_candidate_by_output"))
except Exception as _e:
    print("Feature-scope diagnostic export skipped:", type(_e).__name__, _e)


,output_name,status_cv,reason_cv,output_family,selected_stage_id_cv,selected_stage_name_cv,selected_stage_source_cv,feature_scope_cv,best_y_strategy_cv,final_method_name_cv,...,selected_cv_model_name,selected_cv_target_transform,selected_cv_topk,refit_combo_mismatch,forced_refit_note,n_groups_full,n_final_features,final_train_r2,final_train_rmse,final_train_mae
8,Thermal characteristics | h | W/m.K,ok,,thermal,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,added_only,singleton_raw,residual_pls_extratrees,...,Residual_PLS_ExtraTrees,raw,8,True,forced exact refit: model=Residual_PLS_ExtraTr...,25,4,0.718900,0.696304,0.581411
4,Yield strength,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,3,True,forced exact refit: model=FeatureAwarePartial(...,56,10,0.627947,29.425192,22.601537
6,Total energy,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Huber_Regressor+ful...,yeo_johnson,10,True,forced exact refit: model=FeatureAwarePartial(...,56,9,0.616152,39.794793,29.879306
13,Vibrational response | FRF (g/N) | 300-3000 Hz...,ok,,vibrational,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,hard_closest_oof,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,9,True,forced exact refit: model=FeatureAwarePartial(...,65,10,0.488920,0.057032,0.038353
3,AS,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Bayesian_Ridge+full...,raw,10,True,forced exact refit: model=FeatureAwarePartial(...,56,10,0.488221,82.791022,63.624616
10,Thermal characteristics | Cooling rate | °C/s,ok,,thermal,C,3rd_direct_residual_shallow_nonlinear,Training_260420 / extra untried direct-residua...,legacy_core_only,singleton_raw,direct_knn_distance,...,KNN_Distance_Direct,raw,6,True,"forced exact refit: model=KNN_Distance_Direct,...",65,2,1.000000,0.000000,0.000000
0,Modulus,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,10,True,forced exact refit: model=FeatureAwarePartial(...,56,12,0.679111,1993.163421,1611.983422
2,APS,ok,,mechanical_energy,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Bayesian_Ridge+full...,raw,10,True,forced exact refit: model=FeatureAwarePartial(...,56,10,0.485098,85.595274,65.681142
9,Thermal characteristics | Heating rate | °C/s,ok,,thermal,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Ridge+full_missingI...,yeo_johnson,12,True,forced exact refit: model=FeatureAwarePartial(...,65,10,0.512219,0.001597,0.001246
11,Thermal characteristics | Heating Temp | °C/s,ok,,thermal,D,4th_partial_featureaware_missing_descriptor_en...,Training_260503/260504 partial-feature ensemble,all_features,singleton_raw,featureaware_partial_missing_ensemble,...,FeatureAwarePartial(common_Huber_Regressor+ful...,raw,12,True,forced exact refit: model=FeatureAwarePartial(...,65,12,0.470854,7.979382,6.305156


In [13]:

# ============================================================
# Cell G1. Backup reload for later plotting / next cell usage
# ============================================================

BACKUP_FILES = sorted(Path(FINAL_BACKUP_DIR).glob("*_model_backup.joblib"))
AVAILABLE_OUTPUTS = [p.name.replace("_model_backup.joblib", "") for p in BACKUP_FILES]

print("Available backups:")
for name in AVAILABLE_OUTPUTS:
    print(" -", name)

BACKUP_MAP = {}
for p in BACKUP_FILES:
    BACKUP_MAP[p.name.replace("_model_backup.joblib", "")] = joblib.load(p)

print("\nUse BACKUP_MAP['<sanitized_output_name>'] in the next cell.")


Available backups:
 - APS
 - AS
 - Com. Strength
 - Densif. strength
 - Modulus
 - Thermal characteristics _ Cooling rate _ °C_s
 - Thermal characteristics _ h _ W_m.K
 - Thermal characteristics _ Heating rate _ °C_s
 - Thermal characteristics _ Heating Temp _ °C_s
 - Thermal characteristics _ Thermal conductivity _ W_m.K
 - Total energy
 - Vibrational response _ FRF (g_N) _ 300-3000 Hz _ AVG
 - Vibrational response _ FRF (g_N) _ 300-8000 Hz _ AVG
 - Vibrational response _ FRF (g_N) _ 3000-6500 Hz _ AVG
 - Vibrational response _ FRF (g_N) _ 6500-8000 Hz _ AVG
 - Yield strength

Use BACKUP_MAP['<sanitized_output_name>'] in the next cell.
